In [58]:
import torch
from pathlib import Path

PROJECT_DIR = Path.home() / "projects" / "karakalpak_stt"

print("=" * 50)
print("DGX SPARK - KARAKALPAK STT")
print("=" * 50)

print("Project:", PROJECT_DIR)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

print("\nCheckpoint exists:", (
    PROJECT_DIR / "checkpoints/xlsr_lora_step_100/training_state.pt"
).exists())

print("Audio shards:", len(list(
    (PROJECT_DIR / "audio_shards").glob("*.parquet")
)))

print("\n✅ Server notebook ready!")

DGX SPARK - KARAKALPAK STT
Project: /home/intern/projects/karakalpak_stt
PyTorch: 2.13.0+cu130
CUDA available: True
GPU: NVIDIA GB10
BF16 supported: True

Checkpoint exists: True
Audio shards: 20

✅ Server notebook ready!


In [59]:
import gc
import torch

from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from peft import LoraConfig, get_peft_model

MODEL_NAME = "facebook/wav2vec2-xls-r-300m"
DEVICE = "cuda"

CHECKPOINT_DIR = (
    PROJECT_DIR / "checkpoints" / "xlsr_lora_step_100"
)

# 1. Processor
processor = Wav2Vec2Processor.from_pretrained(
    str(PROJECT_DIR / "processor")
)

tokenizer = processor.tokenizer

assert len(tokenizer) == 36
assert tokenizer.pad_token_id == 35

# 2. Checkpointni CPU ga yuklash
checkpoint = torch.load(
    CHECKPOINT_DIR / "training_state.pt",
    map_location="cpu",
    weights_only=True
)

print("Checkpoint step:", checkpoint["global_step"])

# 3. Pretrained model + CTC head
base_model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True
)

# 4. Pretrained parametrlarni muzlatish
for param in base_model.parameters():
    param.requires_grad = False

# 5. LoRA konfiguratsiyasi
lora_config = LoraConfig.from_pretrained(
    str(CHECKPOINT_DIR)
)

model = get_peft_model(
    base_model,
    lora_config
)

# 6. Saqlangan tensorlarni tiklash
saved_weights = checkpoint["model_weights"]

current_params = {
    name: param
    for name, param in model.named_parameters()
    if param.requires_grad
}

assert set(saved_weights) == set(current_params), (
    "Checkpoint parameter names do not match!"
)

with torch.no_grad():
    for name, param in current_params.items():
        saved_tensor = saved_weights[name]

        assert param.shape == saved_tensor.shape, name

        param.copy_(
            saved_tensor.to(
                device=param.device,
                dtype=param.dtype
            )
        )

# 7. GPU ga o'tkazish
model = model.to(DEVICE)

print("\nModel device:", next(model.parameters()).device)

model.print_trainable_parameters()

print("Restored tensors:", len(saved_weights))

print("\n✅ 100-step LoRA + CTC head restored!")

Checkpoint step: 100


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Model device: cuda:0
trainable params: 823,332 || all params: 316,298,952 || trainable%: 0.2603
Restored tensors: 98

✅ 100-step LoRA + CTC head restored!


In [60]:
import torch
from transformers import get_linear_schedule_with_warmup

# 1. Trainable parametrlarni ajratamiz
lora_params = []
ctc_head_params = []

for name, param in model.named_parameters():

    if not param.requires_grad:
        continue

    if "lm_head" in name:
        ctc_head_params.append(param)

    elif "lora_" in name:
        lora_params.append(param)

    else:
        raise ValueError(
            f"Unexpected trainable parameter: {name}"
        )

print("LoRA tensors:", len(lora_params))
print("CTC head tensors:", len(ctc_head_params))

assert len(lora_params) == 96
assert len(ctc_head_params) == 2

# 2. Optimizer yaratamiz
optimizer = torch.optim.AdamW(
    [
        {
            "params": lora_params,
            "lr": 1e-4
        },
        {
            "params": ctc_head_params,
            "lr": 3e-4
        }
    ],
    weight_decay=0.01
)

# 3. Scheduler yaratamiz
TOTAL_STEPS = 13023
WARMUP_STEPS = 651
GRAD_ACCUM_STEPS = 4

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_STEPS
)

# 4. Checkpointdagi training holatini tiklaymiz
optimizer.load_state_dict(
    checkpoint["optimizer_state"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state"]
)

global_step = int(checkpoint["global_step"])

# 5. Optimizer state tensorlarini GPUga o'tkazamiz
for state in optimizer.state.values():
    for key, value in state.items():
        if isinstance(value, torch.Tensor):
            state[key] = value.to(DEVICE)

# 6. Tekshiruv
print("\n" + "=" * 55)
print("TRAINING STATE RESTORED")
print("=" * 55)

print("Global step:", global_step)
print("Scheduler last epoch:", scheduler.last_epoch)

print("\nLearning rates:")
print("LoRA LR:", optimizer.param_groups[0]["lr"])
print("CTC head LR:", optimizer.param_groups[1]["lr"])

print("\nOptimizer state entries:", len(optimizer.state))

assert global_step == 100
assert scheduler.last_epoch == 100
assert len(optimizer.state) == 98

print("\n✅ Optimizer and scheduler restored!")

LoRA tensors: 96
CTC head tensors: 2

TRAINING STATE RESTORED
Global step: 100
Scheduler last epoch: 100

Learning rates:
LoRA LR: 1.536098310291859e-05
CTC head LR: 4.608294930875576e-05

Optimizer state entries: 98

✅ Optimizer and scheduler restored!


In [61]:
import io
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf

from collections import OrderedDict
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

DATA_DIR = PROJECT_DIR / "data"
AUDIO_DIR = PROJECT_DIR / "audio_shards"

train_meta = pd.read_parquet(DATA_DIR / "train_metadata.parquet")
val_meta = pd.read_parquet(DATA_DIR / "val_metadata.parquet")
test_meta = pd.read_parquet(DATA_DIR / "test_metadata.parquet")


class KarakalpakDataset(Dataset):

    def __init__(self, metadata, audio_dir, cache_size=4):

        self.metadata = metadata.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.cache_size = cache_size

        self.cache = OrderedDict()
        self.parquet_files = {}

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):

        row = self.metadata.iloc[int(idx)]

        source_file = row["source_file"]
        source_row = int(row["source_row"])

        row_group_idx = source_row // 16
        local_idx = source_row % 16

        cache_key = (source_file, row_group_idx)

        if cache_key not in self.cache:

            if source_file not in self.parquet_files:
                self.parquet_files[source_file] = pq.ParquetFile(
                    self.audio_dir / source_file
                )

            table = self.parquet_files[source_file].read_row_group(
                row_group_idx,
                columns=["audio", "id"]
            )

            self.cache[cache_key] = table.to_pylist()

            if len(self.cache) > self.cache_size:
                self.cache.popitem(last=False)

        else:
            self.cache.move_to_end(cache_key)

        audio_row = self.cache[cache_key][local_idx]

        waveform, sr = sf.read(
            io.BytesIO(audio_row["audio"]),
            dtype="float32"
        )

        if sr != 16000:
            raise ValueError(f"Unexpected sample rate: {sr}")

        if waveform.ndim == 2:
            waveform = waveform.mean(axis=1)

        assert int(audio_row["id"]) == int(row["id"]), (
            "Audio ID mismatch!"
        )

        return {
            "audio": waveform,
            "text": str(row["cleaned_text_lowercase"]),
            "id": int(row["id"])
        }


train_dataset = KarakalpakDataset(
    train_meta,
    AUDIO_DIR
)

val_dataset = KarakalpakDataset(
    val_meta,
    AUDIO_DIR
)

test_dataset = KarakalpakDataset(
    test_meta,
    AUDIO_DIR
)

print("=" * 55)
print("DGX SPARK DATASET")
print("=" * 55)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

sample = train_dataset[0]

print("\nSample ID:", sample["id"])
print("Audio duration:", round(len(sample["audio"]) / 16000, 2))
print("Text:", sample["text"][:150])

assert len(train_dataset) == 17362
assert len(val_dataset) == 965
assert len(test_dataset) == 965

print("\n✅ Dataset ready!")

DGX SPARK DATASET
Train: 17362
Validation: 965
Test: 965

Sample ID: 5583
Audio duration: 16.52
Text: nikkei asia tashkentte oraylıq aziyanıń eń iri aeroportı qurıladı yaponiyanıń sojitz kompaniyası bıyıl tashkent qalasında jańa xalıqaralıq aeroport qu

✅ Dataset ready!


In [62]:
import torch
from torch.utils.data import DataLoader


def ctc_collate_fn(samples):

    audio_list = [
        sample["audio"]
        for sample in samples
    ]

    text_list = [
        sample["text"]
        for sample in samples
    ]

    # Audio -> input_values + attention_mask
    audio_batch = processor(
        audio_list,
        sampling_rate=16000,
        padding=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

    # Transcript -> token IDs
    label_batch = processor.tokenizer(
        text_list,
        padding=True,
        add_special_tokens=False,
        return_tensors="pt"
    )

    # CTC Loss padding pozitsiyalarini hisoblamaydi
    labels = label_batch["input_ids"].masked_fill(
        label_batch["attention_mask"] == 0,
        -100
    )

    return {
        "input_values": audio_batch["input_values"],
        "attention_mask": audio_batch["attention_mask"],
        "labels": labels
    }


BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=ctc_collate_fn,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=ctc_collate_fn,
    num_workers=0,
    pin_memory=True
)

# Tekshirish
test_batch = next(iter(train_loader))

print("=" * 55)
print("DGX SPARK DATALOADER TEST")
print("=" * 55)

print("Input values:", test_batch["input_values"].shape)
print("Attention mask:", test_batch["attention_mask"].shape)
print("Labels:", test_batch["labels"].shape)

valid_labels = test_batch["labels"][
    test_batch["labels"] != -100
]

assert valid_labels.numel() > 0
assert valid_labels.min().item() >= 0
assert valid_labels.max().item() < len(tokenizer)

# CTC blank label sifatida ishlatilmasligi kerak
assert not (
    valid_labels == tokenizer.pad_token_id
).any()

print("\nAudio duration:",
      round(test_batch["input_values"].shape[1] / 16000, 2),
      "seconds")

print("Effective batch size:", BATCH_SIZE * GRAD_ACCUM_STEPS)

print("\n✅ DGX Spark DataLoader ready!")

DGX SPARK DATALOADER TEST
Input values: torch.Size([1, 162304])
Attention mask: torch.Size([1, 162304])
Labels: torch.Size([1, 196])

Audio duration: 10.14 seconds
Effective batch size: 4

✅ DGX Spark DataLoader ready!


In [63]:
import torch
import gc

DEVICE = "cuda"

# Oldingi DataLoader batchini GPUga o'tkazamiz
gpu_batch = {
    key: value.to(DEVICE)
    for key, value in test_batch.items()
}

# Trainable parametrlar
trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

# Avvalgi optimizer va scheduler holatini saqlaymiz
assert global_step == 100
assert scheduler.last_epoch == 100

# Eski gradientlarni tozalaymiz
optimizer.zero_grad(set_to_none=True)

model.train()

torch.cuda.reset_peak_memory_stats()

# 1. Forward
with torch.autocast("cuda", dtype=torch.bfloat16):
    outputs = model(**gpu_batch)
    loss = outputs.loss

print("=" * 55)
print("DGX SPARK BF16 TRAINING TEST")
print("=" * 55)

print("Loss:", round(loss.item(), 4))
print("Loss finite:", torch.isfinite(loss).item())

assert torch.isfinite(loss).item(), "Loss NaN/Inf!"

# 2. Backward
loss.backward()

# Gradientlarni tekshirish
grad_norm = torch.nn.utils.clip_grad_norm_(
    trainable_params,
    max_norm=1.0,
    error_if_nonfinite=True
)

print("\nGradient norm:", round(grad_norm.item(), 4))
print("Gradient finite:", torch.isfinite(grad_norm).item())

# 3. GPU xotirasi
print(
    "\nPeak GPU allocated:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "Peak GPU reserved:",
    round(torch.cuda.max_memory_reserved() / 1024**3, 2),
    "GB"
)

# 4. Tozalash
optimizer.zero_grad(set_to_none=True)

del outputs, loss, gpu_batch
gc.collect()
torch.cuda.empty_cache()

print("\nGlobal step:", global_step)
print("Scheduler step:", scheduler.last_epoch)

print("\n✅ DGX Spark BF16 test completed!")

DGX SPARK BF16 TRAINING TEST
Loss: 6.2624
Loss finite: True

Gradient norm: 3.871
Gradient finite: True

Peak GPU allocated: 7.41 GB
Peak GPU reserved: 9.9 GB

Global step: 100
Scheduler step: 100

✅ DGX Spark BF16 test completed!


In [64]:
import torch
import gc
from tqdm.auto import tqdm

DEVICE = "cuda"
EXTRA_STEPS = 100

trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

assert global_step == 100
assert scheduler.last_epoch == 100

model.train()
optimizer.zero_grad(set_to_none=True)

train_iterator = iter(train_loader)

server_step_losses = []

torch.cuda.reset_peak_memory_stats()

progress = tqdm(
    range(EXTRA_STEPS),
    desc="DGX Spark XLS-R Training"
)

for _ in progress:

    accumulated_loss = 0.0

    for micro_step in range(GRAD_ACCUM_STEPS):

        try:
            batch = next(train_iterator)

        except StopIteration:
            train_iterator = iter(train_loader)
            batch = next(train_iterator)

        batch = {
            key: value.to(DEVICE, non_blocking=True)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16
        ):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM_STEPS

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Non-finite loss at step {global_step}"
            )

        loss.backward()
        accumulated_loss += loss.detach().item()

        del outputs, loss, batch

    grad_norm = torch.nn.utils.clip_grad_norm_(
        trainable_params,
        max_norm=1.0,
        error_if_nonfinite=True
    )

    optimizer.step()
    scheduler.step()
    optimizer.zero_grad(set_to_none=True)

    global_step += 1
    server_step_losses.append(accumulated_loss)

    progress.set_postfix({
        "step": global_step,
        "loss": f"{accumulated_loss:.4f}",
        "lr": f"{optimizer.param_groups[0]['lr']:.2e}"
    })

    if global_step % 25 == 0:

        print(
            f"\nStep {global_step} | "
            f"Loss: {accumulated_loss:.4f} | "
            f"Grad norm: {grad_norm.item():.4f} | "
            f"LoRA LR: {optimizer.param_groups[0]['lr']:.2e} | "
            f"CTC LR: {optimizer.param_groups[1]['lr']:.2e}"
        )

        print(
            "GPU peak allocated:",
            round(
                torch.cuda.max_memory_allocated() / 1024**3,
                2
            ),
            "GB"
        )

print("\n" + "=" * 55)
print("DGX SPARK TRAINING RESULTS")
print("=" * 55)

print("Global step:", global_step)

print(
    "First 10 steps average loss:",
    round(sum(server_step_losses[:10]) / 10, 4)
)

print(
    "Last 10 steps average loss:",
    round(sum(server_step_losses[-10:]) / 10, 4)
)

print(
    "Peak GPU allocated:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB"
)

print("\n✅ Training continued successfully!")

DGX Spark XLS-R Training:   0%|          | 0/100 [00:00<?, ?it/s]


Step 125 | Loss: 8.3056 | Grad norm: 7.3686 | LoRA LR: 1.92e-05 | CTC LR: 5.76e-05
GPU peak allocated: 8.88 GB

Step 150 | Loss: 7.5232 | Grad norm: 5.7697 | LoRA LR: 2.30e-05 | CTC LR: 6.91e-05
GPU peak allocated: 8.88 GB

Step 175 | Loss: 6.5493 | Grad norm: 5.9758 | LoRA LR: 2.69e-05 | CTC LR: 8.06e-05
GPU peak allocated: 8.88 GB

Step 200 | Loss: 6.9004 | Grad norm: 5.8899 | LoRA LR: 3.07e-05 | CTC LR: 9.22e-05
GPU peak allocated: 8.89 GB

DGX SPARK TRAINING RESULTS
Global step: 200
First 10 steps average loss: 7.8103
Last 10 steps average loss: 6.4201
Peak GPU allocated: 8.89 GB

✅ Training continued successfully!


In [65]:
import torch
from pathlib import Path

SAVE_DIR = PROJECT_DIR / "checkpoints" / "xlsr_lora_step_200"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

assert global_step == 200
assert scheduler.last_epoch == 200

# LoRA va CTC head parametrlari
trainable_state = {
    name: param.detach().cpu().clone()
    for name, param in model.named_parameters()
    if param.requires_grad
}

assert len(trainable_state) == 98

# Model konfiguratsiyasi
model.peft_config["default"].save_pretrained(str(SAVE_DIR))
processor.save_pretrained(str(SAVE_DIR))

# Checkpoint
training_state = {
    "global_step": global_step,
    "model_weights": trainable_state,
    "optimizer_state": optimizer.state_dict(),
    "scheduler_state": scheduler.state_dict(),
    "server_step_losses": server_step_losses,
    "base_model_name": MODEL_NAME,
    "vocab_size": len(tokenizer),
    "grad_accum_steps": GRAD_ACCUM_STEPS,
}

checkpoint_path = SAVE_DIR / "training_state.pt"

torch.save(training_state, checkpoint_path)

# Saqlangan faylni tekshirish
verified = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

assert verified["global_step"] == 200
assert len(verified["model_weights"]) == 98
assert len(verified["optimizer_state"]["state"]) == 98

print("=" * 55)
print("CHECKPOINT VERIFIED")
print("=" * 55)

print("Directory:", SAVE_DIR)
print("Global step:", verified["global_step"])
print("Trainable tensors:", len(verified["model_weights"]))

print(
    "Checkpoint size:",
    round(checkpoint_path.stat().st_size / 1024**2, 2),
    "MB"
)

print("\n✅ 200-step checkpoint saved successfully!")

CHECKPOINT VERIFIED
Directory: /home/intern/projects/karakalpak_stt/checkpoints/xlsr_lora_step_200
Global step: 200
Trainable tensors: 98
Checkpoint size: 9.54 MB

✅ 200-step checkpoint saved successfully!


In [66]:
import torch
import jiwer
import numpy as np

from tqdm.auto import tqdm

model.eval()

NUM_EVAL_SAMPLES = 30

references = []
predictions = []
blank_ratios = []
val_losses = []

print("=" * 60)
print(f"XLS-R-300M + LoRA VALIDATION — STEP {global_step}")
print("=" * 60)

with torch.inference_mode():

    for i, batch in enumerate(tqdm(val_loader, total=NUM_EVAL_SAMPLES)):

        if i >= NUM_EVAL_SAMPLES:
            break

        # Reference text
        labels = batch["labels"].clone()
        labels_for_decode = labels.masked_fill(labels == -100, tokenizer.pad_token_id)

        reference = processor.batch_decode(
            labels_for_decode,
            group_tokens=False
        )[0]

        # Batchni GPUga o'tkazish
        gpu_batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        # BF16 inference
        with torch.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(**gpu_batch)

        logits = outputs.logits.float()

        # CTC Greedy Decoding
        pred_ids = logits.argmax(dim=-1)

        prediction = processor.batch_decode(
            pred_ids.cpu()
        )[0]

        blank_ratio = (
            (pred_ids == tokenizer.pad_token_id)
            .float()
            .mean()
            .item()
            * 100
        )

        references.append(reference)
        predictions.append(prediction)
        blank_ratios.append(blank_ratio)
        val_losses.append(outputs.loss.item())

        del outputs, logits, pred_ids, gpu_batch

# Corpus-level WER va CER
WER = jiwer.wer(references, predictions) * 100
CER = jiwer.cer(references, predictions) * 100

average_blank = np.mean(blank_ratios)
average_val_loss = np.mean(val_losses)

print("\n" + "=" * 60)
print("VALIDATION RESULTS")
print("=" * 60)

print("Global step:", global_step)
print("Samples:", len(references))

print("Validation loss:", round(average_val_loss, 4))
print("WER:", round(WER, 2), "%")
print("CER:", round(CER, 2), "%")
print("Average blank ratio:", round(average_blank, 2), "%")

print("\n" + "=" * 60)
print("PREDICTION EXAMPLES")
print("=" * 60)

for i in range(min(3, len(references))):

    print(f"\nExample {i+1}")

    print("REFERENCE:")
    print(references[i][:250])

    print("\nPREDICTION:")
    print(repr(predictions[i][:250]))

    print("-" * 60)

# Natijalarni keyingi solishtirish uchun saqlaymiz
validation_results_200 = {
    "step": global_step,
    "samples": len(references),
    "val_loss": float(average_val_loss),
    "wer": float(WER),
    "cer": float(CER),
    "blank_ratio": float(average_blank)
}

# Modelni evaluation rejimida qoldiramiz.
# Keyingi training boshlanishida model.train() chaqiriladi.

XLS-R-300M + LoRA VALIDATION — STEP 200


  0%|          | 0/30 [00:00<?, ?it/s]


VALIDATION RESULTS
Global step: 200
Samples: 30
Validation loss: 5.3
WER: 100.0 %
CER: 99.99 %
Average blank ratio: 99.99 %

PREDICTION EXAMPLES

Example 1
REFERENCE:
sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq birge islesiwdi rawajlandırıw boyınsha pikir alıstı

PREDICTION:
''
------------------------------------------------------------

Example 2
REFERENCE:
konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perdeǵaliy dabılov instituttıń rus tili hám ádebiyatı kafedrasınıń docenti filologiya ilimleriniń kandida

PREDICTION:
''
------------------------------------------------------------

Example 3
REFERENCE:
sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy jurnalistikanıń roli málimleme shawq

In [67]:
import torch

model.eval()

batch = next(iter(val_loader))

gpu_batch = {
    key: value.to(DEVICE)
    for key, value in batch.items()
}

with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.bfloat16):
        outputs = model(
            input_values=gpu_batch["input_values"],
            attention_mask=gpu_batch["attention_mask"]
        )

logits = outputs.logits.float()
probs = torch.softmax(logits, dim=-1)

blank_id = tokenizer.pad_token_id

blank_probs = probs[0, :, blank_id]

# Eng yuqori ehtimolli nonblank token
nonblank_probs = probs.clone()
nonblank_probs[..., blank_id] = -1

best_nonblank_probs, best_nonblank_ids = (
    nonblank_probs.max(dim=-1)
)

best_nonblank_probs = best_nonblank_probs[0]

blank_ratio = (
    (logits.argmax(dim=-1) == blank_id)
    .float()
    .mean()
    .item()
    * 100
)

print("=" * 55)
print("CTC DIAGNOSTICS — STEP", global_step)
print("=" * 55)

print(
    "Mean blank probability:",
    round(blank_probs.mean().item() * 100, 2),
    "%"
)

print(
    "Mean best nonblank probability:",
    round(best_nonblank_probs.mean().item() * 100, 2),
    "%"
)

print(
    "Blank ratio:",
    round(blank_ratio, 2),
    "%"
)

print(
    "Frames where nonblank > blank:",
    (best_nonblank_probs > blank_probs).sum().item()
)

print("Total frames:", len(blank_probs))

# O'rtacha token ehtimolliklari
mean_probs = probs.mean(dim=(0, 1))

top_probs, top_ids = torch.topk(mean_probs, k=10)

print("\nTOP 10 TOKENS")

for token_id, probability in zip(top_ids, top_probs):

    token_id = token_id.item()

    token = tokenizer.convert_ids_to_tokens(token_id)

    print(
        f"{token_id:2d} "
        f"{token!r:10s} "
        f"{probability.item() * 100:.2f}%"
    )

del outputs, logits, probs, gpu_batch

CTC DIAGNOSTICS — STEP 200
Mean blank probability: 14.57 %
Mean best nonblank probability: 9.27 %
Blank ratio: 100.0 %
Frames where nonblank > blank: 0
Total frames: 738

TOP 10 TOKENS
35 '[PAD]'    14.57%
33 '|'        9.27%
 0 'a'        7.69%
 8 'i'        4.70%
 4 'e'        4.14%
30 'ı'        3.76%
11 'l'        3.62%
13 'n'        2.74%
17 'r'        2.48%
14 'o'        2.04%


In [68]:
import torch
import jiwer
import numpy as np

def evaluate_model(model, val_loader, processor, device="cuda", max_samples=30):

    model.eval()

    references = []
    predictions = []
    blank_ratios = []
    losses = []

    iterator = iter(val_loader)

    with torch.inference_mode():

        for _ in range(max_samples):

            try:
                batch = next(iterator)
            except StopIteration:
                break

            # Reference
            labels = batch["labels"].clone()
            labels[labels == -100] = processor.tokenizer.pad_token_id

            reference = processor.batch_decode(
                labels,
                group_tokens=False
            )[0]

            # GPU
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = model(**batch)

            pred_ids = outputs.logits.argmax(dim=-1)

            prediction = processor.batch_decode(
                pred_ids.cpu()
            )[0]

            blank_ratio = (
                (pred_ids == processor.tokenizer.pad_token_id)
                .float()
                .mean()
                .item()
                * 100
            )

            references.append(reference)
            predictions.append(prediction)
            blank_ratios.append(blank_ratio)
            losses.append(outputs.loss.item())

            del outputs, batch

    result = {
        "step": global_step,
        "samples": len(references),
        "val_loss": float(np.mean(losses)),
        "WER": jiwer.wer(references, predictions) * 100,
        "CER": jiwer.cer(references, predictions) * 100,
        "blank_ratio": float(np.mean(blank_ratios))
    }

    print("=" * 55)
    print("VALIDATION RESULTS")
    print("=" * 55)

    for key, value in result.items():
        print(f"{key}: {value:.4f}" if isinstance(value, float)
              else f"{key}: {value}")

    print("\nPrediction examples:")

    for i in range(min(3, len(predictions))):
        print(f"\nExample {i+1}")
        print("Reference:", references[i][:150])
        print("Prediction:", repr(predictions[i][:150]))

    return result

In [69]:
validation_200 = evaluate_model(
    model=model,
    val_loader=val_loader,
    processor=processor,
    device="cuda",
    max_samples=30
)

VALIDATION RESULTS
step: 200
samples: 30
val_loss: 5.3000
WER: 100.0000
CER: 99.9882
blank_ratio: 99.9946

Prediction examples:

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: ''

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: ''

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: ''


In [70]:
import torch
import numpy as np
from tqdm.auto import tqdm

TARGET_STEP = 500
GRAD_ACCUM_STEPS = 4

assert global_step == 200
assert scheduler.last_epoch == 200

trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

train_iterator = iter(train_loader)

training_history = []
validation_history = [validation_200]

optimizer.zero_grad(set_to_none=True)

progress = tqdm(
    total=TARGET_STEP - global_step,
    desc="DGX Spark XLS-R + LoRA"
)

while global_step < TARGET_STEP:

    model.train()

    accumulated_loss = 0.0

    for micro_step in range(GRAD_ACCUM_STEPS):

        try:
            batch = next(train_iterator)

        except StopIteration:
            train_iterator = iter(train_loader)
            batch = next(train_iterator)

        batch = {
            key: value.to("cuda", non_blocking=True)
            for key, value in batch.items()
        }

        with torch.autocast("cuda", dtype=torch.bfloat16):

            outputs = model(**batch)

            loss = outputs.loss / GRAD_ACCUM_STEPS

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Non-finite loss at step {global_step}"
            )

        loss.backward()
        accumulated_loss += loss.detach().item()

        del outputs, loss, batch

    grad_norm = torch.nn.utils.clip_grad_norm_(
        trainable_params,
        max_norm=1.0,
        error_if_nonfinite=True
    )

    optimizer.step()
    scheduler.step()
    optimizer.zero_grad(set_to_none=True)

    global_step += 1

    training_history.append({
        "step": global_step,
        "loss": accumulated_loss,
        "grad_norm": float(grad_norm.item()),
        "lora_lr": optimizer.param_groups[0]["lr"],
        "ctc_lr": optimizer.param_groups[1]["lr"]
    })

    progress.update(1)

    progress.set_postfix({
        "step": global_step,
        "loss": f"{accumulated_loss:.4f}"
    })

    if global_step % 25 == 0:
        print(
            f"\nStep {global_step} | "
            f"Loss: {accumulated_loss:.4f} | "
            f"Grad norm: {grad_norm.item():.4f}"
        )

    # Har 100 stepda validation
    if global_step % 100 == 0:

        result = evaluate_model(
            model=model,
            val_loader=val_loader,
            processor=processor,
            device="cuda",
            max_samples=30
        )

        validation_history.append(result)

progress.close()

print("\n" + "=" * 55)
print("TRAINING COMPLETED")
print("=" * 55)

print("Global step:", global_step)
print("Validation checks:", len(validation_history))
print("Last training loss:", training_history[-1]["loss"])

DGX Spark XLS-R + LoRA:   0%|          | 0/300 [00:00<?, ?it/s]


Step 225 | Loss: 5.6519 | Grad norm: 6.2861

Step 250 | Loss: 3.9120 | Grad norm: 4.5266

Step 275 | Loss: 3.5343 | Grad norm: 3.6834

Step 300 | Loss: 3.6959 | Grad norm: 1.8387
VALIDATION RESULTS
step: 300
samples: 30
val_loss: 3.2463
WER: 100.0000
CER: 100.0000
blank_ratio: 100.0000

Prediction examples:

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: ''

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: ''

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: ''

Step 325 | Loss: 3.1706 | Grad norm: 0.3424

Step 350 | Loss: 3.1388 | Grad norm: 0.3914

Step 375 | Loss: 3.2960 | Grad norm

In [71]:
import pandas as pd

validation_df = pd.DataFrame(validation_history)

display(
    validation_df[
        ["step", "val_loss", "WER", "CER", "blank_ratio"]
    ]
)

,step,val_loss,WER,CER,blank_ratio
0,200,5.300003,100.0,99.988202,99.994597
1,300,3.246309,100.0,100.000000,99.999999
2,400,3.166302,100.0,100.000000,99.999999
3,500,3.151878,100.0,100.000000,99.999999


In [72]:
import torch
from pathlib import Path

assert global_step == 500
assert scheduler.last_epoch == 500

SAVE_DIR = PROJECT_DIR / "checkpoints" / "xlsr_lora_step_500"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

trainable_state = {
    name: param.detach().cpu().clone()
    for name, param in model.named_parameters()
    if param.requires_grad
}

assert len(trainable_state) == 98

# LoRA konfiguratsiyasi
model.peft_config["default"].save_pretrained(str(SAVE_DIR))

# Tokenizer va feature extractor
processor.save_pretrained(str(SAVE_DIR))

# Checkpoint
training_state = {
    "global_step": global_step,
    "model_weights": trainable_state,
    "optimizer_state": optimizer.state_dict(),
    "scheduler_state": scheduler.state_dict(),
    "training_history": training_history,
    "validation_history": validation_history,
    "base_model_name": MODEL_NAME,
    "vocab_size": len(tokenizer),
    "grad_accum_steps": GRAD_ACCUM_STEPS,
}

checkpoint_path = SAVE_DIR / "training_state.pt"

torch.save(training_state, checkpoint_path)

# Qayta yuklab tekshirish
verified = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

assert verified["global_step"] == 500
assert len(verified["model_weights"]) == 98
assert len(verified["optimizer_state"]["state"]) == 98

print("=" * 55)
print("CHECKPOINT VERIFIED")
print("=" * 55)

print("Directory:", SAVE_DIR)
print("Global step:", verified["global_step"])
print("Trainable tensors:", len(verified["model_weights"]))

print(
    "Checkpoint size:",
    round(checkpoint_path.stat().st_size / 1024**2, 2),
    "MB"
)

print("\n✅ 500-step checkpoint saved successfully!")

CHECKPOINT VERIFIED
Directory: /home/intern/projects/karakalpak_stt/checkpoints/xlsr_lora_step_500
Global step: 500
Trainable tensors: 98
Checkpoint size: 9.56 MB

✅ 500-step checkpoint saved successfully!


In [73]:
import torch
from pathlib import Path


def save_training_checkpoint(step):

    save_dir = (
        PROJECT_DIR
        / "checkpoints"
        / f"xlsr_lora_step_{step}"
    )

    save_dir.mkdir(parents=True, exist_ok=True)

    # Faqat LoRA + CTC head
    trainable_state = {
        name: param.detach().cpu().clone()
        for name, param in model.named_parameters()
        if param.requires_grad
    }

    assert len(trainable_state) == 98

    # Konfiguratsiyalar
    model.peft_config["default"].save_pretrained(str(save_dir))
    processor.save_pretrained(str(save_dir))

    # Training holati
    state = {
        "global_step": step,
        "model_weights": trainable_state,
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "training_history": training_history,
        "validation_history": validation_history,
        "base_model_name": MODEL_NAME,
        "vocab_size": len(tokenizer),
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": torch.cuda.get_rng_state_all(),
    }

    path = save_dir / "training_state.pt"

    torch.save(state, path)

    print(f"\n✅ Checkpoint saved: step {step}")
    print("Path:", path)

    return path


print("✅ Automatic checkpoint function ready!")

✅ Automatic checkpoint function ready!


In [74]:
import torch
from tqdm.auto import tqdm

TARGET_STEP = 1000

assert global_step == 500
assert scheduler.last_epoch == 500

trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

train_iterator = iter(train_loader)

optimizer.zero_grad(set_to_none=True)

progress = tqdm(
    total=TARGET_STEP - global_step,
    desc="DGX Spark XLS-R Fine-tuning"
)

while global_step < TARGET_STEP:

    model.train()

    accumulated_loss = 0.0

    for micro_step in range(GRAD_ACCUM_STEPS):

        try:
            batch = next(train_iterator)

        except StopIteration:
            train_iterator = iter(train_loader)
            batch = next(train_iterator)

        batch = {
            key: value.to("cuda", non_blocking=True)
            for key, value in batch.items()
        }

        with torch.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM_STEPS

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Non-finite loss at step {global_step}"
            )

        loss.backward()
        accumulated_loss += loss.detach().item()

        del outputs, loss, batch

    grad_norm = torch.nn.utils.clip_grad_norm_(
        trainable_params,
        max_norm=1.0,
        error_if_nonfinite=True
    )

    optimizer.step()
    scheduler.step()
    optimizer.zero_grad(set_to_none=True)

    global_step += 1

    training_history.append({
        "step": global_step,
        "loss": accumulated_loss,
        "grad_norm": float(grad_norm.item()),
        "lora_lr": optimizer.param_groups[0]["lr"],
        "ctc_lr": optimizer.param_groups[1]["lr"]
    })

    progress.update(1)

    progress.set_postfix({
        "step": global_step,
        "loss": f"{accumulated_loss:.4f}"
    })

    # Har 100 stepda validation
    if global_step % 100 == 0:

        result = evaluate_model(
            model=model,
            val_loader=val_loader,
            processor=processor,
            device="cuda",
            max_samples=30
        )

        validation_history.append(result)

    # Har 250 stepda checkpoint
    if global_step % 250 == 0:
        save_training_checkpoint(global_step)

progress.close()

print("\n" + "=" * 55)
print("TRAINING COMPLETED")
print("=" * 55)
print("Global step:", global_step)
print("✅ Checkpoints saved at steps 750 and 1000!")

DGX Spark XLS-R Fine-tuning:   0%|          | 0/500 [00:00<?, ?it/s]

VALIDATION RESULTS
step: 600
samples: 30
val_loss: 3.1747
WER: 100.0000
CER: 100.0000
blank_ratio: 100.0000

Prediction examples:

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: ''

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: ''

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: ''
VALIDATION RESULTS
step: 700
samples: 30
val_loss: 3.1358
WER: 100.0000
CER: 100.0000
blank_ratio: 100.0000

Prediction examples:

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Pre

In [75]:
import torch

model.eval()

batch = next(iter(val_loader))

gpu_batch = {
    k: v.to("cuda")
    for k, v in batch.items()
}

with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.bfloat16):

        outputs = model(
            input_values=gpu_batch["input_values"],
            attention_mask=gpu_batch["attention_mask"]
        )

logits = outputs.logits.float()
probs = torch.softmax(logits, dim=-1)

blank_id = tokenizer.pad_token_id

blank_probs = probs[..., blank_id]

nonblank_probs = probs.clone()
nonblank_probs[..., blank_id] = 0

best_nonblank = nonblank_probs.max(dim=-1).values

print("=" * 55)
print("CTC DIAGNOSTICS")
print("=" * 55)

print("Global step:", global_step)

print(
    "Mean blank probability:",
    round(blank_probs.mean().item() * 100, 2),
    "%"
)

print(
    "Mean best nonblank probability:",
    round(best_nonblank.mean().item() * 100, 2),
    "%"
)

print(
    "Frames where nonblank > blank:",
    (best_nonblank > blank_probs).sum().item()
)

print(
    "Total frames:",
    blank_probs.numel()
)

print(
    "Blank argmax ratio:",
    round(
        (logits.argmax(dim=-1) == blank_id)
        .float().mean().item() * 100,
        2
    ),
    "%"
)


CTC DIAGNOSTICS
Global step: 1000
Mean blank probability: 71.12 %
Mean best nonblank probability: 3.44 %
Frames where nonblank > blank: 0
Total frames: 738
Blank argmax ratio: 100.0 %


In [76]:
import numpy as np
from tqdm.auto import tqdm

def check_ctc_alignment(dataset, processor, limit=200):

    invalid_samples = []
    ratios = []

    for i in tqdm(range(min(limit, len(dataset)))):

        sample = dataset[i]

        waveform = sample["audio"]
        text = sample["text"]

        # Audio sample uzunligi
        audio_samples = len(waveform)

        # XLS-R encoder frame uzunligi
        # Wav2Vec2 convolution strides: 5, 2, 2, 2, 2, 2, 2
        input_length = (audio_samples - 10) // 5 + 1

        for kernel, stride in [
            (3, 2),
            (3, 2),
            (3, 2),
            (3, 2),
            (2, 2),
            (2, 2),
        ]:
            input_length = (
                (input_length - kernel) // stride + 1
            )

        token_ids = processor.tokenizer(
            text,
            add_special_tokens=False
        )["input_ids"]

        target_length = len(token_ids)

        repeats = sum(
            token_ids[j] == token_ids[j - 1]
            for j in range(1, len(token_ids))
        )

        minimum_frames = target_length + repeats

        ratio = input_length / max(minimum_frames, 1)
        ratios.append(ratio)

        if input_length < minimum_frames:

            invalid_samples.append({
                "id": sample["id"],
                "frames": input_length,
                "tokens": target_length,
                "repeats": repeats,
                "required_frames": minimum_frames
            })

    print("=" * 55)
    print("CTC ALIGNMENT CHECK")
    print("=" * 55)

    print("Checked samples:", len(ratios))
    print("Invalid samples:", len(invalid_samples))

    print(
        "Minimum frame/token ratio:",
        round(min(ratios), 3)
    )

    print(
        "Average frame/token ratio:",
        round(np.mean(ratios), 3)
    )

    print("\nInvalid examples:")
    for item in invalid_samples[:10]:
        print(item)

    return invalid_samples, ratios


invalid_samples, ratios = check_ctc_alignment(
    train_dataset,
    processor,
    limit=200
)

  0%|          | 0/200 [00:00<?, ?it/s]

CTC ALIGNMENT CHECK
Checked samples: 200
Invalid samples: 0
Minimum frame/token ratio: 2.053
Average frame/token ratio: 3.136

Invalid examples:


In [77]:
import numpy as np

# Dastlabki 100 ta namunadan 5–10 sekundlik audio qidiramiz
selected_sample = None

for i in range(min(100, len(train_dataset))):

    sample = train_dataset[i]

    duration = len(sample["audio"]) / 16000

    if 5 <= duration <= 10:
        selected_sample = sample
        break

assert selected_sample is not None, "5–10 sekundlik audio topilmadi"

sample_id = selected_sample["id"]
sample_audio = selected_sample["audio"]
sample_text = selected_sample["text"]

# Tokenizatsiya
token_ids = tokenizer(
    sample_text,
    add_special_tokens=False
)["input_ids"]

print("=" * 55)
print("SINGLE SAMPLE OVERFIT TEST")
print("=" * 55)

print("Sample ID:", sample_id)
print("Audio duration:", round(len(sample_audio) / 16000, 2), "seconds")
print("Text:", sample_text)
print("Token count:", len(token_ids))

print("\n✅ Single sample selected!")

SINGLE SAMPLE OVERFIT TEST
Sample ID: 13987
Audio duration: 9.01 seconds
Text: tórtkúl akademiyalıq liceyinde jumıs baslaǵanımda oqıwshılardıń fizika pánine qızıǵıwshılıǵın kórip repetitor bolıw pikiri tuwıldı sonnan keyin sırt elge ketiw pikirinen qayttım
Token count: 177

✅ Single sample selected!


In [78]:
import torch
from transformers import Wav2Vec2ForCTC
from peft import LoraConfig, get_peft_model

EXPERIMENT_DIR = (
    PROJECT_DIR / "checkpoints" / "xlsr_lora_step_500"
)

experiment_checkpoint = torch.load(
    EXPERIMENT_DIR / "training_state.pt",
    map_location="cpu",
    weights_only=True
)

# 1. Pretrained XLS-R model
experiment_base = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True
)

# 2. Pretrained encoderni muzlatamiz
for param in experiment_base.parameters():
    param.requires_grad = False

# 3. LoRA konfiguratsiyasini yuklaymiz
experiment_lora_config = LoraConfig.from_pretrained(
    str(EXPERIMENT_DIR)
)

experiment_model = get_peft_model(
    experiment_base,
    experiment_lora_config
)

# 4. Checkpointdagi 98 ta trainable tensorni tiklaymiz
experiment_params = {
    name: param
    for name, param in experiment_model.named_parameters()
    if param.requires_grad
}

saved_weights = experiment_checkpoint["model_weights"]

assert set(experiment_params) == set(saved_weights)

with torch.no_grad():
    for name, param in experiment_params.items():
        param.copy_(
            saved_weights[name].to(
                device=param.device,
                dtype=param.dtype
            )
        )

experiment_model = experiment_model.to("cuda")

print("=" * 55)
print("SINGLE-SAMPLE EXPERIMENT MODEL")
print("=" * 55)

print("Checkpoint step:", experiment_checkpoint["global_step"])
print("Trainable tensors:", len(experiment_params))
print("Device:", next(experiment_model.parameters()).device)

print("\n✅ Experiment model ready!")

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


SINGLE-SAMPLE EXPERIMENT MODEL
Checkpoint step: 500
Trainable tensors: 98
Device: cuda:0

✅ Experiment model ready!


In [79]:
import torch
import jiwer
from tqdm.auto import tqdm

# 1. Bitta audio uchun batch
single_batch = ctc_collate_fn([selected_sample])

single_batch = {
    key: value.to("cuda")
    for key, value in single_batch.items()
}

# 2. Alohida optimizer
experiment_lora_params = []
experiment_head_params = []

for name, param in experiment_model.named_parameters():

    if not param.requires_grad:
        continue

    if "lm_head" in name:
        experiment_head_params.append(param)

    elif "lora_" in name:
        experiment_lora_params.append(param)

experiment_optimizer = torch.optim.AdamW(
    [
        {"params": experiment_lora_params, "lr": 1e-4},
        {"params": experiment_head_params, "lr": 3e-4},
    ],
    weight_decay=0.01
)

# 3. Prediction tekshirish funksiyasi
def evaluate_single_sample():

    experiment_model.eval()

    with torch.inference_mode():

        with torch.autocast("cuda", dtype=torch.bfloat16):
            outputs = experiment_model(**single_batch)

        pred_ids = outputs.logits.argmax(dim=-1)

        prediction = processor.batch_decode(
            pred_ids.cpu()
        )[0]

        blank_ratio = (
            (pred_ids == tokenizer.pad_token_id)
            .float()
            .mean()
            .item() * 100
        )

        cer = jiwer.cer(sample_text, prediction) * 100

        loss = outputs.loss.item()

    return loss, cer, blank_ratio, prediction


# 4. Boshlang'ich natija
loss, cer, blank, prediction = evaluate_single_sample()

print("=" * 55)
print("BEFORE SINGLE-SAMPLE TRAINING")
print("=" * 55)

print("Loss:", round(loss, 4))
print("CER:", round(cer, 2), "%")
print("Blank ratio:", round(blank, 2), "%")
print("Prediction:", repr(prediction[:200]))

# 5. Overfitting
EXPERIMENT_STEPS = 200

history = []

progress = tqdm(
    range(1, EXPERIMENT_STEPS + 1),
    desc="Single Sample Overfit"
)

for step in progress:

    experiment_model.train()
    experiment_optimizer.zero_grad(set_to_none=True)

    with torch.autocast("cuda", dtype=torch.bfloat16):
        outputs = experiment_model(**single_batch)
        loss = outputs.loss

    if not torch.isfinite(loss).item():
        raise RuntimeError(f"Non-finite loss at step {step}")

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        experiment_model.parameters(),
        max_norm=1.0
    )

    experiment_optimizer.step()

    if step % 50 == 0:

        val_loss, cer, blank, prediction = evaluate_single_sample()

        history.append({
            "step": step,
            "loss": val_loss,
            "cer": cer,
            "blank_ratio": blank,
            "prediction": prediction
        })

        print(f"\nStep {step}")
        print("Loss:", round(val_loss, 4))
        print("CER:", round(cer, 2), "%")
        print("Blank ratio:", round(blank, 2), "%")
        print("Prediction:", repr(prediction[:200]))

    progress.set_postfix(
        train_loss=f"{loss.item():.4f}"
    )

print("\n✅ Single-sample experiment completed!")

BEFORE SINGLE-SAMPLE TRAINING
Loss: 3.228
CER: 100.0 %
Blank ratio: 100.0 %
Prediction: ''


Single Sample Overfit:   0%|          | 0/200 [00:00<?, ?it/s]


Step 50
Loss: 3.08
CER: 100.0 %
Blank ratio: 100.0 %
Prediction: ''

Step 100
Loss: 3.0693
CER: 100.0 %
Blank ratio: 100.0 %
Prediction: ''

Step 150
Loss: 3.0773
CER: 100.0 %
Blank ratio: 100.0 %
Prediction: ''

Step 200
Loss: 3.0668
CER: 100.0 %
Blank ratio: 100.0 %
Prediction: ''

✅ Single-sample experiment completed!


In [80]:
import torch

print("=" * 60)
print("CTC MODEL DIAGNOSTICS")
print("=" * 60)

# 1. Tokenizer
print("\n1. TOKENIZER")

print("Tokenizer vocab size:", len(tokenizer))
print("Tokenizer PAD ID:", tokenizer.pad_token_id)
print("Tokenizer UNK ID:", tokenizer.unk_token_id)

token_ids = tokenizer(
    sample_text,
    add_special_tokens=False
)["input_ids"]

decoded_text = tokenizer.decode(
    token_ids,
    group_tokens=False,
    skip_special_tokens=False
)

print("Original text:", repr(sample_text[:150]))
print("Decoded text:", repr(decoded_text[:150]))

print("Unknown tokens:", token_ids.count(tokenizer.unk_token_id))
print("Blank tokens in labels:", token_ids.count(tokenizer.pad_token_id))

# 2. Model configuration
print("\n2. MODEL CONFIGURATION")

print("Model vocab size:", experiment_model.config.vocab_size)
print("Model PAD ID:", experiment_model.config.pad_token_id)
print("Model CTC reduction:", experiment_model.config.ctc_loss_reduction)
print("Model CTC zero infinity:", experiment_model.config.ctc_zero_infinity)

# 3. Trainable parameters
print("\n3. TRAINABLE PARAMETERS")

lora_count = 0
head_count = 0

for name, param in experiment_model.named_parameters():

    if not param.requires_grad:
        continue

    if "lora_" in name:
        lora_count += 1

    if "lm_head" in name:
        head_count += 1

print("LoRA trainable tensors:", lora_count)
print("CTC head trainable tensors:", head_count)

# 4. Output size
print("\n4. AUDIO/LABEL SHAPES")

print("Input shape:", tuple(single_batch["input_values"].shape))
print("Label shape:", tuple(single_batch["labels"].shape))

with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.bfloat16):
        test_outputs = experiment_model(**single_batch)

print("Logits shape:", tuple(test_outputs.logits.shape))
print("Loss:", round(test_outputs.loss.item(), 4))

print("\n✅ Diagnostics completed!")

CTC MODEL DIAGNOSTICS

1. TOKENIZER
Tokenizer vocab size: 36
Tokenizer PAD ID: 35
Tokenizer UNK ID: 34
Original text: 'tórtkúl akademiyalıq liceyinde jumıs baslaǵanımda oqıwshılardıń fizika pánine qızıǵıwshılıǵın kórip repetitor bolıw pikiri tuwıldı sonnan keyin sırt e'
Decoded text: 'tórtkúl akademiyalıq liceyinde jumıs baslaǵanımda oqıwshılardıń fizika pánine qızıǵıwshılıǵın kórip repetitor bolıw pikiri tuwıldı sonnan keyin sırt e'
Unknown tokens: 0
Blank tokens in labels: 0

2. MODEL CONFIGURATION
Model vocab size: 36
Model PAD ID: 35
Model CTC reduction: mean
Model CTC zero infinity: True

3. TRAINABLE PARAMETERS
LoRA trainable tensors: 96
CTC head trainable tensors: 2

4. AUDIO/LABEL SHAPES
Input shape: (1, 144214)
Label shape: (1, 177)
Logits shape: (1, 450, 36)
Loss: 3.0668

✅ Diagnostics completed!


In [81]:
import torch
import numpy as np

experiment_model.train()
experiment_optimizer.zero_grad(set_to_none=True)

# Forward
with torch.autocast("cuda", dtype=torch.bfloat16):
    outputs = experiment_model(**single_batch)
    loss = outputs.loss

# Backward
loss.backward()

print("=" * 60)
print("GRADIENT DIAGNOSTICS")
print("=" * 60)

print("Loss:", round(loss.item(), 4))

lora_grad_norms = []
head_grad_norms = []

missing_gradients = []

for name, param in experiment_model.named_parameters():

    if not param.requires_grad:
        continue

    if param.grad is None:
        missing_gradients.append(name)
        continue

    grad_norm = param.grad.detach().float().norm().item()

    if "lora_" in name:
        lora_grad_norms.append(grad_norm)

    elif "lm_head" in name:
        head_grad_norms.append(grad_norm)

print("\nLoRA:")
print("Gradient tensors:", len(lora_grad_norms))
print("Mean grad norm:", np.mean(lora_grad_norms))
print("Max grad norm:", np.max(lora_grad_norms))
print("Nonzero gradients:", sum(x > 0 for x in lora_grad_norms))

print("\nCTC Head:")
print("Gradient tensors:", len(head_grad_norms))
print("Mean grad norm:", np.mean(head_grad_norms))
print("Max grad norm:", np.max(head_grad_norms))
print("Nonzero gradients:", sum(x > 0 for x in head_grad_norms))

print("\nMissing gradients:", len(missing_gradients))

if missing_gradients:
    print("Examples:", missing_gradients[:10])

# Gradientlarni tozalaymiz
experiment_optimizer.zero_grad(set_to_none=True)

del outputs, loss

print("\n✅ Gradient diagnostics completed!")

GRADIENT DIAGNOSTICS
Loss: 3.0593

LoRA:
Gradient tensors: 92
Mean grad norm: 0.003865751490415594
Max grad norm: 0.03832046687602997
Nonzero gradients: 92

CTC Head:
Gradient tensors: 2
Mean grad norm: 0.2518407888710499
Max grad norm: 0.4289301633834839
Nonzero gradients: 2

Missing gradients: 4
Examples: ['base_model.model.wav2vec2.encoder.layers.5.attention.v_proj.lora_A.default.weight', 'base_model.model.wav2vec2.encoder.layers.5.attention.v_proj.lora_B.default.weight', 'base_model.model.wav2vec2.encoder.layers.5.attention.q_proj.lora_A.default.weight', 'base_model.model.wav2vec2.encoder.layers.5.attention.q_proj.lora_B.default.weight']

✅ Gradient diagnostics completed!


In [82]:
import torch

print("=" * 60)
print("XLS-R LAYERDROP DIAGNOSTICS")
print("=" * 60)

# LayerDrop konfiguratsiyasi
print(
    "LayerDrop:",
    experiment_model.config.layerdrop
)

print(
    "Number of encoder layers:",
    experiment_model.config.num_hidden_layers
)

# LayerDrop ta'sirini vaqtincha o'chirish
old_layerdrop = experiment_model.config.layerdrop

experiment_model.config.layerdrop = 0.0

experiment_model.train()
experiment_optimizer.zero_grad(set_to_none=True)

with torch.autocast("cuda", dtype=torch.bfloat16):
    outputs = experiment_model(**single_batch)

loss = outputs.loss
loss.backward()

missing = []

for name, param in experiment_model.named_parameters():

    if param.requires_grad and param.grad is None:
        missing.append(name)

print("\nLoss:", round(loss.item(), 4))
print("Missing gradients:", len(missing))

if missing:
    for name in missing[:10]:
        print(name)

# Avvalgi sozlamani tiklash
experiment_model.config.layerdrop = old_layerdrop

experiment_optimizer.zero_grad(set_to_none=True)

print("\n✅ LayerDrop diagnostics completed!")

XLS-R LAYERDROP DIAGNOSTICS
LayerDrop: 0.1
Number of encoder layers: 24

Loss: 3.0603
Missing gradients: 0

✅ LayerDrop diagnostics completed!


In [83]:
# XLS-R encoder qatlamlari
encoder_layers = (
    experiment_model
    .base_model
    .model
    .wav2vec2
    .encoder
    .layers
)

print("=" * 55)
print("XLS-R ENCODER LAYERS")
print("=" * 55)

print("Total layers:", len(encoder_layers))

for i in range(20, 24):

    layer = encoder_layers[i]

    total_params = sum(
        p.numel()
        for p in layer.parameters()
    )

    trainable_params = sum(
        p.numel()
        for p in layer.parameters()
        if p.requires_grad
    )

    print(
        f"Layer {i}: "
        f"Total={total_params:,} | "
        f"Trainable={trainable_params:,}"
    )

print("\n✅ Encoder layers inspected!")

XLS-R ENCODER LAYERS
Total layers: 24
Layer 20: Total=12,628,992 | Trainable=32,768
Layer 21: Total=12,628,992 | Trainable=32,768
Layer 22: Total=12,628,992 | Trainable=32,768
Layer 23: Total=12,628,992 | Trainable=32,768

✅ Encoder layers inspected!


In [84]:
import torch

# Faqat experiment_model ustida ishlaymiz
experiment_model.config.layerdrop = 0.0

encoder_layers = (
    experiment_model
    .base_model
    .model
    .wav2vec2
    .encoder
    .layers
)

# Oxirgi ikkita Transformer qatlam
for layer_idx in [22, 23]:

    layer = encoder_layers[layer_idx]

    for name, param in layer.named_parameters():

        # LoRA allaqachon trainable
        # Endi qolgan asosiy parametrlarni ham ochamiz
        param.requires_grad = True

# Parametrlarni sanaymiz
lora_count = 0
head_count = 0
encoder_count = 0

lora_params = []
head_params = []
encoder_params = []

for name, param in experiment_model.named_parameters():

    if not param.requires_grad:
        continue

    if "lora_" in name:
        lora_params.append(param)
        lora_count += param.numel()

    elif "lm_head" in name:
        head_params.append(param)
        head_count += param.numel()

    else:
        encoder_params.append(param)
        encoder_count += param.numel()

print("=" * 60)
print("PARTIAL ENCODER UNFREEZE")
print("=" * 60)

print("LoRA parameters:", f"{lora_count:,}")
print("CTC head parameters:", f"{head_count:,}")
print("Unfrozen encoder parameters:", f"{encoder_count:,}")

print(
    "Total trainable:",
    f"{lora_count + head_count + encoder_count:,}"
)

print("LayerDrop:", experiment_model.config.layerdrop)

print("\n✅ Last 2 Transformer layers unfrozen!")

PARTIAL ENCODER UNFREEZE
LoRA parameters: 786,432
CTC head parameters: 36,900
Unfrozen encoder parameters: 25,192,448
Total trainable: 26,015,780
LayerDrop: 0.0

✅ Last 2 Transformer layers unfrozen!


In [85]:
import torch
import jiwer
from tqdm.auto import tqdm

# 1. Yangi optimizer
experiment_optimizer = torch.optim.AdamW(
    [
        {
            "params": lora_params,
            "lr": 1e-4
        },
        {
            "params": head_params,
            "lr": 3e-4
        },
        {
            "params": encoder_params,
            "lr": 1e-5
        }
    ],
    weight_decay=0.01
)

# 2. Training
EXTRA_STEPS = 100

experiment_model.train()
experiment_optimizer.zero_grad(set_to_none=True)

print("=" * 60)
print("PARTIAL UNFREEZE OVERFIT TEST")
print("=" * 60)

for step in tqdm(range(1, EXTRA_STEPS + 1)):

    experiment_model.train()
    experiment_optimizer.zero_grad(set_to_none=True)

    with torch.autocast("cuda", dtype=torch.bfloat16):
        outputs = experiment_model(**single_batch)
        loss = outputs.loss

    if not torch.isfinite(loss).item():
        raise RuntimeError(f"Non-finite loss at step {step}")

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        experiment_model.parameters(),
        max_norm=1.0,
        error_if_nonfinite=True
    )

    experiment_optimizer.step()

    if step % 25 == 0:

        val_loss, cer, blank, prediction = evaluate_single_sample()

        print(f"\nStep {step}")
        print("Loss:", round(val_loss, 4))
        print("CER:", round(cer, 2), "%")
        print("Blank ratio:", round(blank, 2), "%")
        print("Gradient norm:", round(grad_norm.item(), 4))
        print("Prediction:", repr(prediction[:200]))

    del outputs, loss

print("\n✅ Partial unfreeze experiment completed!")

PARTIAL UNFREEZE OVERFIT TEST


  0%|          | 0/100 [00:00<?, ?it/s]


Step 25
Loss: 3.0453
CER: 100.0 %
Blank ratio: 100.0 %
Gradient norm: 0.0731
Prediction: ''

Step 50
Loss: 3.0163
CER: 100.0 %
Blank ratio: 100.0 %
Gradient norm: 0.2466
Prediction: ''

Step 75
Loss: 3.0292
CER: 100.0 %
Blank ratio: 100.0 %
Gradient norm: 0.5773
Prediction: ''

Step 100
Loss: 2.9036
CER: 100.0 %
Blank ratio: 100.0 %
Gradient norm: 0.7173
Prediction: ''

✅ Partial unfreeze experiment completed!


In [86]:
import torch

experiment_model.eval()

with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.bfloat16):
        outputs = experiment_model(
            input_values=single_batch["input_values"],
            attention_mask=single_batch["attention_mask"]
        )

logits = outputs.logits.float()
probs = torch.softmax(logits, dim=-1)

blank_id = tokenizer.pad_token_id

blank_probs = probs[0, :, blank_id]

nonblank_probs = probs[0].clone()
nonblank_probs[:, blank_id] = 0

best_nonblank_probs, best_nonblank_ids = nonblank_probs.max(dim=-1)

print("=" * 55)
print("CTC OUTPUT DIAGNOSTICS")
print("=" * 55)

print("Logits shape:", tuple(logits.shape))
print("Blank ID:", blank_id)

print("\nMean blank probability:",
      round(blank_probs.mean().item() * 100, 2), "%")

print("Mean best nonblank probability:",
      round(best_nonblank_probs.mean().item() * 100, 2), "%")

print("Maximum nonblank probability:",
      round(best_nonblank_probs.max().item() * 100, 2), "%")

print("Frames where nonblank > blank:",
      int((best_nonblank_probs > blank_probs).sum().item()))

print("Total frames:", len(blank_probs))

print("\nTop 10 tokens by mean probability:")

mean_probs = probs[0].mean(dim=0)

top_values, top_indices = torch.topk(mean_probs, k=10)

for token_id, probability in zip(top_indices, top_values):
    token_id = token_id.item()
    token = tokenizer.convert_ids_to_tokens(token_id)

    print(f"{token!r:10s}: {probability.item() * 100:.2f}%")

CTC OUTPUT DIAGNOSTICS
Logits shape: (1, 450, 36)
Blank ID: 35

Mean blank probability: 70.54 %
Mean best nonblank probability: 5.08 %
Maximum nonblank probability: 17.43 %
Frames where nonblank > blank: 0
Total frames: 450

Top 10 tokens by mean probability:
'[PAD]'   : 70.54%
'|'       : 4.19%
'ı'       : 3.62%
'n'       : 2.25%
'a'       : 2.13%
'e'       : 1.99%
'i'       : 1.99%
'l'       : 1.63%
's'       : 1.20%
't'       : 1.18%


In [87]:
from IPython.display import Audio, display

print("Sample ID:", selected_sample["id"])
print("Duration:", len(selected_sample["audio"]) / 16000)

print("\nREFERENCE TEXT:")
print(selected_sample["text"])

print("\nAUDIO:")
display(
    Audio(
        selected_sample["audio"],
        rate=16000
    )
)

Sample ID: 13987
Duration: 9.013375

REFERENCE TEXT:
tórtkúl akademiyalıq liceyinde jumıs baslaǵanımda oqıwshılardıń fizika pánine qızıǵıwshılıǵın kórip repetitor bolıw pikiri tuwıldı sonnan keyin sırt elge ketiw pikirinen qayttım

AUDIO:


In [88]:
import torch
import torch.nn.functional as F

experiment_model.eval()

with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.bfloat16):
        outputs = experiment_model(**single_batch)

logits = outputs.logits.float()

# Encoder chiqishidagi haqiqiy uzunlik
input_lengths = experiment_model.base_model.model._get_feat_extract_output_lengths(
    single_batch["attention_mask"].sum(dim=-1)
).long()

# Padding qilinmagan labels
labels = single_batch["labels"]

target_lengths = (labels >= 0).sum(dim=-1).long()

targets = labels[labels >= 0].long()

# CTCLoss uchun [T, B, V]
log_probs = F.log_softmax(logits, dim=-1).transpose(0, 1)

manual_loss = F.ctc_loss(
    log_probs,
    targets,
    input_lengths,
    target_lengths,
    blank=tokenizer.pad_token_id,
    reduction="mean",
    zero_infinity=True
)

print("=" * 60)
print("CTC LOSS VERIFICATION")
print("=" * 60)

print("Model loss:", outputs.loss.item())
print("Manual CTC loss:", manual_loss.item())

print("Input frames:", input_lengths.tolist())
print("Target tokens:", target_lengths.tolist())

print(
    "Loss difference:",
    abs(outputs.loss.item() - manual_loss.item())
)

assert torch.allclose(
    outputs.loss.float(),
    manual_loss.float(),
    atol=0.05,
    rtol=0.01
), "CTC Loss mismatch!"

print("\n✅ CTC Loss calculation verified!")

CTC LOSS VERIFICATION
Model loss: 2.903646230697632
Manual CTC loss: 2.903646230697632
Input frames: [450]
Target tokens: [177]
Loss difference: 0.0

✅ CTC Loss calculation verified!


In [89]:
import torch
import torch.nn.functional as F

experiment_model.eval()

with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.bfloat16):
        outputs = experiment_model(
            input_values=single_batch["input_values"],
            attention_mask=single_batch["attention_mask"],
            output_hidden_states=True
        )

# Oxirgi encoder hidden states: [batch, frames, hidden_size]
hidden = outputs.hidden_states[-1].float()[0]

print("=" * 60)
print("XLS-R ENCODER FEATURES DIAGNOSTICS")
print("=" * 60)

print("Hidden shape:", tuple(hidden.shape))
print("Hidden mean:", round(hidden.mean().item(), 6))
print("Hidden std:", round(hidden.std().item(), 6))

# Har bir frame bo'yicha o'zgaruvchanlik
temporal_std = hidden.std(dim=0).mean()

print(
    "Mean temporal std:",
    round(temporal_std.item(), 6)
)

# Qo'shni framelar o'xshashligi
adjacent_similarity = F.cosine_similarity(
    hidden[:-1],
    hidden[1:],
    dim=-1
).mean()

print(
    "Adjacent frame cosine similarity:",
    round(adjacent_similarity.item(), 6)
)

# Uzoqroq framelar o'xshashligi
distant_similarity = F.cosine_similarity(
    hidden[:-50],
    hidden[50:],
    dim=-1
).mean()

print(
    "Distant frame cosine similarity:",
    round(distant_similarity.item(), 6)
)

# Encoder output NaN/Inf tekshiruvi
print("All hidden values finite:",
      torch.isfinite(hidden).all().item())

print("\n✅ Encoder diagnostics completed!")

XLS-R ENCODER FEATURES DIAGNOSTICS
Hidden shape: (450, 1024)
Hidden mean: 0.687701
Hidden std: 13.455336
Mean temporal std: 5.704636
Adjacent frame cosine similarity: 0.923647
Distant frame cosine similarity: 0.766765
All hidden values finite: True

✅ Encoder diagnostics completed!


In [90]:
import torch
import torch.nn.functional as F

# PEFT modules_to_save orqali o'ralgan CTC headni topamiz
ctc_head = (
    experiment_model
    .base_model
    .model
    .lm_head
)

print("=" * 60)
print("CTC HEAD DIAGNOSTICS")
print("=" * 60)

print("Head type:", type(ctc_head).__name__)

# Haqiqatan forwardda ishlatilayotgan trainable head
if hasattr(ctc_head, "modules_to_save"):
    head = ctc_head.modules_to_save["default"]
else:
    head = ctc_head

W = head.weight.detach().float()
b = head.bias.detach().float()

blank_id = tokenizer.pad_token_id

print("\nWeight shape:", tuple(W.shape))
print("Bias shape:", tuple(b.shape))

print("\nWEIGHT STATISTICS")
print("Weight mean:", round(W.mean().item(), 6))
print("Weight std:", round(W.std().item(), 6))
print("Weight norm:", round(W.norm().item(), 6))

print("\nBLANK TOKEN")
print("Blank bias:", round(b[blank_id].item(), 6))
print("Mean nonblank bias:",
      round(torch.cat([b[:blank_id], b[blank_id+1:]]).mean().item(), 6))

print("Blank weight norm:", round(W[blank_id].norm().item(), 6))

# Oldingi cell'dagi hidden statesdan foydalanamiz
hidden = outputs.hidden_states[-1].float()[0]

# Logitlarni qismlarga ajratamiz
linear_term = F.linear(hidden, W, None)
bias_term = b

mean_linear = linear_term.mean(dim=0)
mean_logits = mean_linear + bias_term

print("\nMEAN LOGIT COMPONENTS")

for token_id in [blank_id, tokenizer.convert_tokens_to_ids("|"),
                 tokenizer.convert_tokens_to_ids("a")]:

    token = tokenizer.convert_ids_to_tokens(token_id)

    print(
        f"{token!r:8s} | "
        f"Linear: {mean_linear[token_id].item():.4f} | "
        f"Bias: {bias_term[token_id].item():.4f} | "
        f"Total: {mean_logits[token_id].item():.4f}"
    )

print("\n✅ CTC head diagnostics completed!")

CTC HEAD DIAGNOSTICS
Head type: ModulesToSaveWrapper

Weight shape: (36, 1024)
Bias shape: (36,)

WEIGHT STATISTICS
Weight mean: -0.000142
Weight std: 0.042348
Weight norm: 8.130701

BLANK TOKEN
Blank bias: 0.032231
Mean nonblank bias: -0.030161
Blank weight norm: 1.099156

MEAN LOGIT COMPONENTS
'[PAD]'  | Linear: 270.2463 | Bias: 0.0322 | Total: 270.2786
'|'      | Linear: 68.5770 | Bias: 0.0061 | Total: 68.5831
'a'      | Linear: 7.9826 | Bias: 0.0016 | Total: 7.9842

✅ CTC head diagnostics completed!


In [91]:
import torch
import torch.nn.functional as F

experiment_model.eval()

ctc_head = experiment_model.base_model.model.lm_head

if hasattr(ctc_head, "modules_to_save"):
    head = ctc_head.modules_to_save["default"]
else:
    head = ctc_head

with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.bfloat16):

        result = experiment_model(
            input_values=single_batch["input_values"],
            attention_mask=single_batch["attention_mask"],
            output_hidden_states=True
        )

actual_logits = result.logits.float()

hidden = result.hidden_states[-1]

# CTC head orqali qayta hisoblaymiz
with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.bfloat16):
        reconstructed_logits = head(hidden).float()

difference = (
    actual_logits - reconstructed_logits
).abs()

print("=" * 60)
print("CTC HEAD INPUT VERIFICATION")
print("=" * 60)

print("Hidden shape:", tuple(hidden.shape))
print("Actual logits:", tuple(actual_logits.shape))
print("Reconstructed logits:", tuple(reconstructed_logits.shape))

print("\nMaximum difference:", difference.max().item())
print("Mean difference:", difference.mean().item())

print("\nActual logit statistics:")
print("Mean:", actual_logits.mean().item())
print("Std:", actual_logits.std().item())
print("Min:", actual_logits.min().item())
print("Max:", actual_logits.max().item())

print("\nHidden statistics:")
print("Mean:", hidden.float().mean().item())
print("Std:", hidden.float().std().item())

print("\n✅ Verification completed!")

CTC HEAD INPUT VERIFICATION
Hidden shape: (1, 450, 1024)
Actual logits: (1, 450, 36)
Reconstructed logits: (1, 450, 36)

Maximum difference: 977.9375
Mean difference: 191.81959533691406

Actual logit statistics:
Mean: -2.2399556636810303
Std: 2.9210000038146973
Min: -10.5625
Max: 4.0

Hidden statistics:
Mean: 0.6877008676528931
Std: 13.45533561706543

✅ Verification completed!


In [92]:
import torch

experiment_model.eval()

captured = {}

# Haqiqiy forward vaqtida ishlatiladigan CTC head
ctc_wrapper = experiment_model.base_model.model.lm_head

def capture_head_input(module, inputs, output):
    captured["input"] = inputs[0].detach().float().cpu()
    captured["output"] = output.detach().float().cpu()

# Wrapperga hook qo'yamiz
hook_handle = ctc_wrapper.register_forward_hook(
    capture_head_input
)

try:
    with torch.inference_mode():
        with torch.autocast("cuda", dtype=torch.bfloat16):

            result = experiment_model(
                input_values=single_batch["input_values"],
                attention_mask=single_batch["attention_mask"],
                output_hidden_states=True
            )
finally:
    hook_handle.remove()

actual_logits = result.logits.detach().float().cpu()
head_input = captured["input"]
head_output = captured["output"]
last_hidden = result.hidden_states[-1].detach().float().cpu()

print("=" * 60)
print("REAL CTC HEAD INPUT DIAGNOSTICS")
print("=" * 60)

print("Last hidden shape:", tuple(last_hidden.shape))
print("Head input shape:", tuple(head_input.shape))
print("Head output shape:", tuple(head_output.shape))

print("\nLast hidden std:", last_hidden.std().item())
print("Head input std:", head_input.std().item())

print(
    "\nHidden vs head input max difference:",
    (last_hidden - head_input).abs().max().item()
)

print(
    "Head output vs actual logits max difference:",
    (head_output - actual_logits).abs().max().item()
)

print("\nHead input mean:", head_input.mean().item())
print("Head input min:", head_input.min().item())
print("Head input max:", head_input.max().item())

print("\n✅ Real CTC head input captured!")

REAL CTC HEAD INPUT DIAGNOSTICS
Last hidden shape: (1, 450, 1024)
Head input shape: (1, 450, 1024)
Head output shape: (1, 450, 36)

Last hidden std: 13.45533561706543
Head input std: 0.18063046038150787

Hidden vs head input max difference: 162.00001525878906
Head output vs actual logits max difference: 0.0

Head input mean: -0.0004257021064404398
Head input min: -1.4435820579528809
Head input max: 1.1182230710983276

✅ Real CTC head input captured!


In [93]:
import inspect
from transformers import Wav2Vec2ForCTC

print("=" * 65)
print("WAV2VEC2 FOR CTC FORWARD")
print("=" * 65)

source = inspect.getsource(Wav2Vec2ForCTC.forward)

for i, line in enumerate(source.splitlines(), start=1):
    if any(key in line for key in [
        "wav2vec2(",
        "hidden_states",
        "last_hidden_state",
        "dropout",
        "lm_head",
        "logits",
    ]):
        print(f"{i:3d}: {line}")

print("\n" + "=" * 65)
print("MODEL DROPOUT CONFIGURATION")
print("=" * 65)

print("Final dropout:", experiment_model.config.final_dropout)
print("Hidden dropout:", experiment_model.config.hidden_dropout)
print("LayerDrop:", experiment_model.config.layerdrop)

print("\n✅ Forward inspection completed!")

WAV2VEC2 FOR CTC FORWARD
 13:             the sequence length of the output logits. Indices are selected in `[-100, 0, ..., config.vocab_size - 1]`.
 20:         outputs = self.wav2vec2(
 26:         hidden_states = outputs.last_hidden_state
 27:         hidden_states = self.dropout(hidden_states)
 29:         logits = self.lm_head(hidden_states)
 46:             log_probs = nn.functional.log_softmax(logits, dim=-1, dtype=torch.float32).transpose(0, 1)
 60:             loss=loss, logits=logits, hidden_states=outputs.hidden_states, attentions=outputs.attentions

MODEL DROPOUT CONFIGURATION
Final dropout: 0.0
Hidden dropout: 0.1
LayerDrop: 0.0

✅ Forward inspection completed!


In [94]:
import torch

experiment_model.eval()

with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.bfloat16):

        # Asosiy Wav2Vec2 encoder
        encoder_outputs = (
            experiment_model
            .base_model
            .model
            .wav2vec2(
                input_values=single_batch["input_values"],
                attention_mask=single_batch["attention_mask"],
                output_hidden_states=True
            )
        )

last_hidden = encoder_outputs.last_hidden_state.float()
intermediate_hidden = encoder_outputs.hidden_states[-1].float()

print("=" * 60)
print("ENCODER OUTPUT COMPARISON")
print("=" * 60)

print("Last hidden shape:", tuple(last_hidden.shape))

print("\nLast hidden STD:", last_hidden.std().item())
print("Intermediate hidden STD:", intermediate_hidden.std().item())

print(
    "\nMaximum difference:",
    (last_hidden - intermediate_hidden).abs().max().item()
)

# CTC headga kiradigan haqiqiy tensor
ctc_wrapper = experiment_model.base_model.model.lm_head

with torch.autocast("cuda", dtype=torch.bfloat16):
    reconstructed_logits = ctc_wrapper(
        encoder_outputs.last_hidden_state
    )

print(
    "\nReconstructed logits shape:",
    tuple(reconstructed_logits.shape)
)

print("\n✅ Encoder output verified!")

ENCODER OUTPUT COMPARISON
Last hidden shape: (1, 450, 1024)

Last hidden STD: 0.18063044548034668
Intermediate hidden STD: 13.45533561706543

Maximum difference: 162.00001525878906

Reconstructed logits shape: (1, 450, 36)

✅ Encoder output verified!


In [95]:
import torch
import torch.nn.functional as F

experiment_model.eval()

with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.bfloat16):

        encoder_outputs = (
            experiment_model
            .base_model
            .model
            .wav2vec2(
                input_values=single_batch["input_values"],
                attention_mask=single_batch["attention_mask"]
            )
        )

hidden = encoder_outputs.last_hidden_state.float()[0]

print("=" * 60)
print("FINAL ENCODER REPRESENTATION DIAGNOSTICS")
print("=" * 60)

print("Shape:", tuple(hidden.shape))
print("Mean:", hidden.mean().item())
print("STD:", hidden.std().item())

# Har bir feature vaqt bo'yicha qanchalik o'zgaradi?
temporal_std = hidden.std(dim=0)

print("\nMean temporal STD:", temporal_std.mean().item())
print("Max temporal STD:", temporal_std.max().item())
print("Min temporal STD:", temporal_std.min().item())

# Qo'shni framelar
adjacent_cosine = F.cosine_similarity(
    hidden[:-1],
    hidden[1:],
    dim=-1
)

# Uzoq framelar
distant_cosine = F.cosine_similarity(
    hidden[:-50],
    hidden[50:],
    dim=-1
)

print("\nAdjacent cosine:", adjacent_cosine.mean().item())
print("Distant cosine:", distant_cosine.mean().item())

# Har bir frame vektori o'rtacha vektordan
# qanchalik uzoqda?
centered = hidden - hidden.mean(dim=0, keepdim=True)

print(
    "Mean centered vector norm:",
    centered.norm(dim=-1).mean().item()
)

print(
    "Mean original vector norm:",
    hidden.norm(dim=-1).mean().item()
)

print("\n✅ Final encoder representation checked!")

FINAL ENCODER REPRESENTATION DIAGNOSTICS
Shape: (450, 1024)
Mean: -0.0004257021355442703
STD: 0.18063044548034668

Mean temporal STD: 0.06611859798431396
Max temporal STD: 0.4185565710067749
Min temporal STD: 1.3159517038729973e-05

Adjacent cosine: 0.9412485361099243
Distant cosine: 0.8180908560752869
Mean centered vector norm: 2.313267707824707
Mean original vector norm: 5.771000385284424

✅ Final encoder representation checked!


In [96]:
import torch

experiment_model.eval()

with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.bfloat16):
        outputs = experiment_model(
            input_values=single_batch["input_values"],
            attention_mask=single_batch["attention_mask"]
        )

logits = outputs.logits.float()[0]
probs = torch.softmax(logits, dim=-1)

blank_id = tokenizer.pad_token_id

blank_probs = probs[:, blank_id]

# Eng yaxshi nonblank token
nonblank_logits = logits.clone()
nonblank_logits[:, blank_id] = -float("inf")

best_nonblank_logits, best_nonblank_ids = nonblank_logits.max(dim=-1)

blank_logits = logits[:, blank_id]

# Blank va eng yaxshi harf orasidagi farq
margin = blank_logits - best_nonblank_logits

print("=" * 60)
print("CTC TOKEN MARGIN DIAGNOSTICS")
print("=" * 60)

print("Frames:", len(margin))

print("\nBlank vs best nonblank logit margin:")
print("Minimum:", round(margin.min().item(), 4))
print("Mean:", round(margin.mean().item(), 4))
print("Maximum:", round(margin.max().item(), 4))

print("\nFrames with small margin:")
print("Margin < 0.5:", int((margin < 0.5).sum().item()))
print("Margin < 1.0:", int((margin < 1.0).sum().item()))
print("Margin < 2.0:", int((margin < 2.0).sum().item()))

# Blankga eng yaqin kelgan 10 ta frame
values, indices = torch.topk(margin, k=10, largest=False)

print("\n10 FRAMES CLOSEST TO NONBLANK")

for frame_idx, value in zip(indices.tolist(), values.tolist()):

    token_id = best_nonblank_ids[frame_idx].item()
    token = tokenizer.convert_ids_to_tokens(token_id)

    print(
        f"Frame {frame_idx:3d} | "
        f"Best token: {token!r:5s} | "
        f"Margin: {value:.4f} | "
        f"Blank prob: {blank_probs[frame_idx].item() * 100:.2f}%"
    )

print("\n✅ Margin diagnostics completed!")

CTC TOKEN MARGIN DIAGNOSTICS
Frames: 450

Blank vs best nonblank logit margin:
Minimum: 0.2266
Mean: 2.6675
Maximum: 3.3965

Frames with small margin:
Margin < 0.5: 2
Margin < 1.0: 6
Margin < 2.0: 33

10 FRAMES CLOSEST TO NONBLANK
Frame 310 | Best token: 'i'   | Margin: 0.2266 | Blank prob: 21.86%
Frame 268 | Best token: 't'   | Margin: 0.2969 | Blank prob: 22.81%
Frame 309 | Best token: 'i'   | Margin: 0.7422 | Blank prob: 30.59%
Frame 439 | Best token: 't'   | Margin: 0.7422 | Blank prob: 34.65%
Frame 414 | Best token: 't'   | Margin: 0.8203 | Blank prob: 34.94%
Frame 409 | Best token: 'i'   | Margin: 0.8516 | Blank prob: 34.35%
Frame 413 | Best token: 'i'   | Margin: 1.0703 | Blank prob: 40.11%
Frame 311 | Best token: 'i'   | Margin: 1.1797 | Blank prob: 43.16%
Frame 378 | Best token: 'i'   | Margin: 1.1953 | Blank prob: 34.92%
Frame 253 | Best token: 'i'   | Margin: 1.2031 | Blank prob: 37.47%

✅ Margin diagnostics completed!


In [97]:
import torch
import jiwer

experiment_model.eval()

with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.bfloat16):
        outputs = experiment_model(
            input_values=single_batch["input_values"],
            attention_mask=single_batch["attention_mask"]
        )

original_logits = outputs.logits.float()

blank_id = tokenizer.pad_token_id

print("=" * 65)
print("CTC BLANK PENALTY EXPERIMENT")
print("=" * 65)

print("REFERENCE:")
print(sample_text[:200])

print("\nRESULTS")

for penalty in [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]:

    logits = original_logits.clone()

    # Faqat decoding vaqtida blank logitini kamaytiramiz
    logits[..., blank_id] -= penalty

    pred_ids = logits.argmax(dim=-1)

    prediction = processor.batch_decode(
        pred_ids.cpu()
    )[0]

    cer = jiwer.cer(
        sample_text,
        prediction
    ) * 100

    blank_ratio = (
        (pred_ids == blank_id)
        .float()
        .mean()
        .item() * 100
    )

    print(f"\nPenalty: {penalty}")
    print("CER:", round(cer, 2), "%")
    print("Blank ratio:", round(blank_ratio, 2), "%")
    print("Prediction:", repr(prediction[:200]))

print("\n✅ Blank penalty diagnostics completed!")

CTC BLANK PENALTY EXPERIMENT
REFERENCE:
tórtkúl akademiyalıq liceyinde jumıs baslaǵanımda oqıwshılardıń fizika pánine qızıǵıwshılıǵın kórip repetitor bolıw pikiri tuwıldı sonnan keyin sırt elge ketiw pikirinen qayttım

RESULTS

Penalty: 0.0
CER: 100.0 %
Blank ratio: 100.0 %
Prediction: ''

Penalty: 0.5
CER: 98.87 %
Blank ratio: 99.56 %
Prediction: 'ti'

Penalty: 1.0
CER: 97.18 %
Blank ratio: 98.67 %
Prediction: 'tiitt'

Penalty: 1.5
CER: 92.09 %
Blank ratio: 96.0 %
Prediction: 'tiiittitiiiitt'

Penalty: 2.0
CER: 88.14 %
Blank ratio: 92.67 %
Prediction: 'tiiiiiititiiiiitiiiiittt'

Penalty: 2.5
CER: 80.79 %
Blank ratio: 85.56 %
Prediction: 'titiiııaiıaııııtiiıiıiititiiiiit i  i iiititt'

Penalty: 3.0
CER: 70.62 %
Blank ratio: 12.44 %
Prediction: 'titi  i ıai ı ıaiıaı ıitı ıı i i iıiı ı ı ı ı i ı it iti i ii it    ı i i i iiti ı tt'

✅ Blank penalty diagnostics completed!


In [98]:
import torch
from transformers import Wav2Vec2ForCTC

head_only_model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True
)

# Encoderni muzlatamiz
for param in head_only_model.parameters():
    param.requires_grad = False

# Faqat CTC Headni o'qitamiz
for param in head_only_model.lm_head.parameters():
    param.requires_grad = True

# Tajriba uchun LayerDrop o'chiriladi
head_only_model.config.layerdrop = 0.0

head_only_model = head_only_model.to("cuda")

trainable = sum(
    p.numel()
    for p in head_only_model.parameters()
    if p.requires_grad
)

print("=" * 55)
print("HEAD-ONLY EXPERIMENT")
print("=" * 55)

print("Trainable parameters:", trainable)
print("Expected:", 36900)

assert trainable == 36900

print("\n✅ Head-only model ready!")

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


HEAD-ONLY EXPERIMENT
Trainable parameters: 36900
Expected: 36900

✅ Head-only model ready!


In [99]:
import torch
import jiwer
from tqdm.auto import tqdm

# Faqat CTC Head uchun optimizer
head_optimizer = torch.optim.AdamW(
    head_only_model.lm_head.parameters(),
    lr=1e-3,
    weight_decay=0.0
)

def evaluate_head_only():
    head_only_model.eval()

    with torch.inference_mode():
        with torch.autocast("cuda", dtype=torch.bfloat16):
            outputs = head_only_model(**single_batch)

    pred_ids = outputs.logits.argmax(dim=-1)

    prediction = processor.batch_decode(
        pred_ids.cpu()
    )[0]

    blank_ratio = (
        (pred_ids == tokenizer.pad_token_id)
        .float()
        .mean()
        .item() * 100
    )

    cer = jiwer.cer(sample_text, prediction) * 100

    return outputs.loss.item(), cer, blank_ratio, prediction


print("=" * 60)
print("HEAD-ONLY OVERFIT EXPERIMENT")
print("=" * 60)

loss, cer, blank, prediction = evaluate_head_only()

print("\nBEFORE TRAINING")
print("Loss:", round(loss, 4))
print("CER:", round(cer, 2), "%")
print("Blank ratio:", round(blank, 2), "%")
print("Prediction:", repr(prediction[:150]))

head_history = []

for step in tqdm(range(1, 201)):

    head_only_model.train()

    # Frozen encoderda dropoutni o'chiramiz
    head_only_model.wav2vec2.eval()

    head_optimizer.zero_grad(set_to_none=True)

    with torch.autocast("cuda", dtype=torch.bfloat16):
        outputs = head_only_model(**single_batch)
        loss = outputs.loss

    if not torch.isfinite(loss):
        raise RuntimeError(f"Invalid loss at step {step}")

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        head_only_model.lm_head.parameters(),
        max_norm=1.0
    )

    head_optimizer.step()

    if step % 25 == 0:

        val_loss, cer, blank, prediction = evaluate_head_only()

        head_history.append({
            "step": step,
            "loss": val_loss,
            "cer": cer,
            "blank_ratio": blank,
            "prediction": prediction
        })

        print(f"\nStep {step}")
        print("Loss:", round(val_loss, 4))
        print("CER:", round(cer, 2), "%")
        print("Blank ratio:", round(blank, 2), "%")
        print("Prediction:", repr(prediction[:150]))

    del outputs, loss

print("\n✅ Head-only experiment completed!")

HEAD-ONLY OVERFIT EXPERIMENT

BEFORE TRAINING
Loss: 6.4967
CER: 96.05 %
Blank ratio: 96.44 %
Prediction: 'c  c[UNK]  [UNK]ffcfct'


  0%|          | 0/200 [00:00<?, ?it/s]


Step 25
Loss: 3.1193
CER: 100.0 %
Blank ratio: 100.0 %
Prediction: ''

Step 50
Loss: 2.9999
CER: 98.87 %
Blank ratio: 99.56 %
Prediction: 'tm'

Step 75
Loss: 2.9178
CER: 96.61 %
Blank ratio: 98.67 %
Prediction: 'tmtttm'

Step 100
Loss: 2.827
CER: 94.92 %
Blank ratio: 97.78 %
Prediction: 'tabatttbtm'

Step 125
Loss: 2.7328
CER: 93.79 %
Blank ratio: 97.33 %
Prediction: 'tabazátǵtbtm'

Step 150
Loss: 2.639
CER: 92.09 %
Blank ratio: 96.89 %
Prediction: 'tamazáǵǵtbattm'

Step 175
Loss: 2.5477
CER: 90.96 %
Blank ratio: 96.0 %
Prediction: 'tamaziáǵǵttbattm'

Step 200
Loss: 2.4589
CER: 88.7 %
Blank ratio: 94.89 %
Prediction: 'úaimaaziázǵittbtattm'

✅ Head-only experiment completed!


In [100]:
from tqdm.auto import tqdm
import torch

# Oldingi tajriba 200 stepda tugagan
EXTRA_STEPS = 300

for step in tqdm(range(201, 501)):

    head_only_model.train()
    head_only_model.wav2vec2.eval()

    head_optimizer.zero_grad(set_to_none=True)

    with torch.autocast("cuda", dtype=torch.bfloat16):
        outputs = head_only_model(**single_batch)
        loss = outputs.loss

    if not torch.isfinite(loss).item():
        raise RuntimeError(f"Invalid loss at step {step}")

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        head_only_model.lm_head.parameters(),
        max_norm=1.0
    )

    head_optimizer.step()

    if step % 50 == 0:

        val_loss, cer, blank, prediction = evaluate_head_only()

        head_history.append({
            "step": step,
            "loss": val_loss,
            "cer": cer,
            "blank_ratio": blank,
            "prediction": prediction
        })

        print(f"\nStep {step}")
        print("Loss:", round(val_loss, 4))
        print("CER:", round(cer, 2), "%")
        print("Blank ratio:", round(blank, 2), "%")
        print("Prediction:", repr(prediction[:200]))

    del outputs, loss

print("\n✅ Head-only training reached 500 steps!")

  0%|          | 0/300 [00:00<?, ?it/s]


Step 250
Loss: 2.2894
CER: 87.01 %
Blank ratio: 94.22 %
Prediction: 'úaimbaaziáizǵirtbttattm'

Step 300
Loss: 2.1295
CER: 79.66 %
Blank ratio: 91.11 %
Prediction: 'úakimbaanzikaáizǵǵrrtbiritnntiriattm'

Step 350
Loss: 1.9792
CER: 77.4 %
Blank ratio: 90.44 %
Prediction: 'úakiimbaanzikaáinzǵǵrirtbiritunntiriattm'

Step 400
Loss: 1.838
CER: 73.45 %
Blank ratio: 88.67 %
Prediction: 'úakiiinmbaanazikaáinızǵǵrirtbiritunntkiiriqyttm'

Step 450
Loss: 1.7074
CER: 71.75 %
Blank ratio: 87.78 %
Prediction: 'rúaakiiinumbaanazikapáinızǵǵrirtbiritunntkiiriqyttm'

Step 500
Loss: 1.5867
CER: 70.06 %
Blank ratio: 87.11 %
Prediction: 'rúaakiiinumbaaaazikapáinqızıǵǵrirtbiritunntkpiiriqyttm'

✅ Head-only training reached 500 steps!


In [101]:
import torch
from pathlib import Path

SAVE_DIR = PROJECT_DIR / "experiments" / "head_only_500"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "step": 500,
        "lm_head_state": {
            name: tensor.detach().cpu().clone()
            for name, tensor in head_only_model.lm_head.state_dict().items()
        },
        "optimizer_state": head_optimizer.state_dict(),
        "history": head_history,
        "sample_id": sample_id,
        "base_model_name": MODEL_NAME,
        "vocab_size": len(tokenizer),
    },
    SAVE_DIR / "training_state.pt"
)

print("✅ Head-only experiment saved!")
print("Path:", SAVE_DIR)

✅ Head-only experiment saved!
Path: /home/intern/projects/karakalpak_stt/experiments/head_only_500


In [102]:
import torch
from transformers import Wav2Vec2ForCTC
from peft import LoraConfig, get_peft_model

# 1. Pretrained XLS-R-300M
lora_test_base = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True
)

# 2. Encoderning asosiy parametrlarini muzlatamiz
for param in lora_test_base.parameters():
    param.requires_grad = False

# 3. LayerDrop o'chiriladi
lora_test_base.config.layerdrop = 0.0

# 4. LoRA konfiguratsiyasi
lora_test_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    modules_to_save=["lm_head"]
)

# 5. LoRA adapterlarini qo'shamiz
lora_test_model = get_peft_model(
    lora_test_base,
    lora_test_config
).to("cuda")

# 6. Trainable parametrlarni tekshiramiz
lora_trainable = []
head_trainable = []

for name, param in lora_test_model.named_parameters():

    if not param.requires_grad:
        continue

    if "lora_" in name:
        lora_trainable.append(param)

    elif "lm_head" in name:
        head_trainable.append(param)

    else:
        raise ValueError(f"Unexpected trainable parameter: {name}")

print("=" * 60)
print("NEW XLS-R + LoRA EXPERIMENT")
print("=" * 60)

print("LoRA parameters:", sum(p.numel() for p in lora_trainable))
print("CTC Head parameters:", sum(p.numel() for p in head_trainable))

print(
    "Total trainable:",
    sum(p.numel() for p in lora_test_model.parameters() if p.requires_grad)
)

print("LayerDrop:", lora_test_model.config.layerdrop)
print("LoRA dropout:", lora_test_config.lora_dropout)

assert sum(p.numel() for p in lora_trainable) == 786432
assert sum(p.numel() for p in head_trainable) == 36900

print("\n✅ New LoRA experiment ready!")

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


NEW XLS-R + LoRA EXPERIMENT
LoRA parameters: 786432
CTC Head parameters: 36900
Total trainable: 823332
LayerDrop: 0.0
LoRA dropout: 0.0

✅ New LoRA experiment ready!


In [103]:
import torch
import jiwer
from tqdm.auto import tqdm

# Faqat yangi LoRA tajribasi uchun optimizer
lora_test_optimizer = torch.optim.AdamW(
    [
        {"params": lora_trainable, "lr": 1e-4},
        {"params": head_trainable, "lr": 1e-3}
    ],
    weight_decay=0.0
)

# LoRA modelini baholash
def evaluate_lora_test():

    lora_test_model.eval()

    with torch.inference_mode():
        with torch.autocast("cuda", dtype=torch.bfloat16):
            outputs = lora_test_model(**single_batch)

    pred_ids = outputs.logits.argmax(dim=-1)

    prediction = processor.batch_decode(
        pred_ids.cpu()
    )[0]

    blank_ratio = (
        (pred_ids == tokenizer.pad_token_id)
        .float()
        .mean()
        .item() * 100
    )

    cer = jiwer.cer(sample_text, prediction) * 100

    return outputs.loss.item(), cer, blank_ratio, prediction


print("=" * 60)
print("LoRA + CTC HEAD OVERFIT EXPERIMENT")
print("=" * 60)

loss, cer, blank, prediction = evaluate_lora_test()

print("\nBEFORE TRAINING")
print("Loss:", round(loss, 4))
print("CER:", round(cer, 2), "%")
print("Blank ratio:", round(blank, 2), "%")
print("Prediction:", repr(prediction[:150]))

lora_test_history = []

for step in tqdm(range(1, 201)):

    lora_test_model.train()

    # Encoder dropoutlarini o'chiramiz.
    # LoRA parametrlari trainable bo'lib qoladi.
    lora_test_model.base_model.model.wav2vec2.eval()

    lora_test_optimizer.zero_grad(set_to_none=True)

    with torch.autocast("cuda", dtype=torch.bfloat16):
        outputs = lora_test_model(**single_batch)
        loss = outputs.loss

    if not torch.isfinite(loss).item():
        raise RuntimeError(f"Non-finite loss at step {step}")

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        lora_test_model.parameters(),
        max_norm=1.0,
        error_if_nonfinite=True
    )

    lora_test_optimizer.step()

    if step % 25 == 0:

        val_loss, cer, blank, prediction = evaluate_lora_test()

        lora_test_history.append({
            "step": step,
            "loss": val_loss,
            "cer": cer,
            "blank_ratio": blank,
            "prediction": prediction
        })

        print(f"\nStep {step}")
        print("Loss:", round(val_loss, 4))
        print("CER:", round(cer, 2), "%")
        print("Blank ratio:", round(blank, 2), "%")
        print("Prediction:", repr(prediction[:150]))

    del outputs, loss

print("\n✅ LoRA experiment reached 200 steps!")

LoRA + CTC HEAD OVERFIT EXPERIMENT

BEFORE TRAINING
Loss: 6.6045
CER: 97.74 %
Blank ratio: 0.0 %
Prediction: 'momım'


  0%|          | 0/200 [00:00<?, ?it/s]


Step 25
Loss: 3.1308
CER: 100.0 %
Blank ratio: 100.0 %
Prediction: ''

Step 50
Loss: 3.0025
CER: 98.87 %
Blank ratio: 99.56 %
Prediction: 'mm'

Step 75
Loss: 2.8488
CER: 93.79 %
Blank ratio: 96.0 %
Prediction: 'mttmmmmmmmttbtttm'

Step 100
Loss: 2.4618
CER: 82.49 %
Blank ratio: 90.44 %
Prediction: 'ttktakmmbaatamazkaáttǵkttbkttktktttm'

Step 125
Loss: 2.0576
CER: 70.06 %
Blank ratio: 86.44 %
Prediction: 'ttkúakamiyaiiumbaaǵamdaakizikaáizǵǵktitbiitutktiiiyttm'

Step 150
Loss: 1.6921
CER: 59.32 %
Blank ratio: 81.56 %
Prediction: 'trtkúakamiyaiijum baaǵamdaarfizikapáizıǵǵkrprtitrbpikirituytktipiriayttm'

Step 175
Loss: 1.2647
CER: 48.59 %
Blank ratio: 77.56 %
Prediction: 'trtkú akamiyaqicinjum baaǵaımdaardńfizikapáinızıǵǵkórprtitrbpikiritunnantyit kti pikiriayttm'

Step 200
Loss: 0.8601
CER: 32.2 %
Blank ratio: 71.33 %
Prediction: 'tórtkú akadmiyaqicyind jumı baaǵaımda qardń fizikanpáninızıǵhǵn kórip rptitr bo pikiri tu nnan kyinrt  kti pikirinn ayttm'

✅ LoRA experiment reached 200 step

In [104]:
import torch
from pathlib import Path

SAVE_DIR = PROJECT_DIR / "experiments" / "lora_single_sample_200"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

trainable_state = {
    name: param.detach().cpu().clone()
    for name, param in lora_test_model.named_parameters()
    if param.requires_grad
}

lora_test_model.peft_config["default"].save_pretrained(str(SAVE_DIR))
processor.save_pretrained(str(SAVE_DIR))

state = {
    "step": 200,
    "model_weights": trainable_state,
    "optimizer_state": lora_test_optimizer.state_dict(),
    "history": lora_test_history,
    "sample_id": sample_id,
    "base_model_name": MODEL_NAME,
    "vocab_size": len(tokenizer),
}

path = SAVE_DIR / "training_state.pt"
torch.save(state, path)

verified = torch.load(path, map_location="cpu", weights_only=True)

assert verified["step"] == 200
assert len(verified["model_weights"]) == 98

print("=" * 55)
print("LoRA EXPERIMENT SAVED")
print("=" * 55)

print("Step:", verified["step"])
print("Trainable tensors:", len(verified["model_weights"]))
print("Path:", path)

print("\n✅ LoRA single-sample checkpoint saved!")

LoRA EXPERIMENT SAVED
Step: 200
Trainable tensors: 98
Path: /home/intern/projects/karakalpak_stt/experiments/lora_single_sample_200/training_state.pt

✅ LoRA single-sample checkpoint saved!


In [105]:
import numpy as np

from torch.utils.data import Subset, DataLoader

# Takrorlanadigan tajriba uchun seed
rng = np.random.default_rng(42)

# 100 ta turli train namunasi
train_indices = rng.choice(
    len(train_dataset),
    size=100,
    replace=False
).tolist()

# Validation uchun alohida splitdan 30 ta
val_indices = list(range(30))

mini_train_dataset = Subset(
    train_dataset,
    train_indices
)

mini_val_dataset = Subset(
    val_dataset,
    val_indices
)

mini_train_loader = DataLoader(
    mini_train_dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=ctc_collate_fn,
    num_workers=0,
    pin_memory=True
)

mini_val_loader = DataLoader(
    mini_val_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=ctc_collate_fn,
    num_workers=0,
    pin_memory=True
)

# ID lar takrorlanmasligini tekshirish
selected_ids = [
    int(train_meta.iloc[i]["id"])
    for i in train_indices
]

assert len(selected_ids) == len(set(selected_ids))

# Train va validation bir-biridan mustaqil
train_ids = set(selected_ids)

validation_ids = {
    int(val_meta.iloc[i]["id"])
    for i in val_indices
}

assert train_ids.isdisjoint(validation_ids)

print("=" * 55)
print("MULTI-SAMPLE EXPERIMENT DATASET")
print("=" * 55)

print("Train samples:", len(mini_train_dataset))
print("Validation samples:", len(mini_val_dataset))
print("Duplicate train IDs:", len(selected_ids) - len(set(selected_ids)))
print("Train/Validation overlap:", len(train_ids & validation_ids))

print("\n✅ Multi-sample dataset ready!")

MULTI-SAMPLE EXPERIMENT DATASET
Train samples: 100
Validation samples: 30
Duplicate train IDs: 0
Train/Validation overlap: 0

✅ Multi-sample dataset ready!


In [106]:
import torch
from transformers import Wav2Vec2ForCTC
from peft import LoraConfig, get_peft_model

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# 1. Pretrained XLS-R-300M
multi_base = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True
)

# 2. Pretrained weightlarni muzlatamiz
for param in multi_base.parameters():
    param.requires_grad = False

# 3. LayerDrop o'chiriladi
multi_base.config.layerdrop = 0.0

# 4. LoRA konfiguratsiyasi
multi_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    modules_to_save=["lm_head"]
)

# 5. LoRA adapterlarini qo'shamiz
multi_model = get_peft_model(
    multi_base,
    multi_lora_config
).to("cuda")

# 6. Trainable parametrlarni ajratamiz
multi_lora_params = []
multi_head_params = []

for name, param in multi_model.named_parameters():

    if not param.requires_grad:
        continue

    if "lora_" in name:
        multi_lora_params.append(param)

    elif "lm_head" in name:
        multi_head_params.append(param)

    else:
        raise ValueError(
            f"Unexpected trainable parameter: {name}"
        )

lora_count = sum(p.numel() for p in multi_lora_params)
head_count = sum(p.numel() for p in multi_head_params)

assert lora_count == 786432
assert head_count == 36900

# 7. Dataset mavjudligini tekshiramiz
assert len(mini_train_dataset) == 100
assert len(mini_val_dataset) == 30

print("=" * 60)
print("MULTI-SAMPLE XLS-R + LoRA MODEL")
print("=" * 60)

print("Base model:", MODEL_NAME)
print("LoRA parameters:", f"{lora_count:,}")
print("CTC Head parameters:", f"{head_count:,}")
print("Total trainable:", f"{lora_count + head_count:,}")
print("LayerDrop:", multi_model.config.layerdrop)
print("Device:", next(multi_model.parameters()).device)

print("\nTrain samples:", len(mini_train_dataset))
print("Validation samples:", len(mini_val_dataset))

print("\n✅ Multi-sample LoRA model ready!")

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


MULTI-SAMPLE XLS-R + LoRA MODEL
Base model: facebook/wav2vec2-xls-r-300m
LoRA parameters: 786,432
CTC Head parameters: 36,900
Total trainable: 823,332
LayerDrop: 0.0
Device: cuda:0

Train samples: 100
Validation samples: 30

✅ Multi-sample LoRA model ready!


In [107]:
import torch
from transformers import get_linear_schedule_with_warmup

# Training konfiguratsiyasi
MULTI_TOTAL_STEPS = 300
MULTI_WARMUP_STEPS = 20
MULTI_GRAD_ACCUM = 4

# Optimizer
multi_optimizer = torch.optim.AdamW(
    [
        {
            "params": multi_lora_params,
            "lr": 1e-4,
            "weight_decay": 0.0
        },
        {
            "params": multi_head_params,
            "lr": 1e-3,
            "weight_decay": 0.0
        }
    ]
)

# Learning rate scheduler
multi_scheduler = get_linear_schedule_with_warmup(
    multi_optimizer,
    num_warmup_steps=MULTI_WARMUP_STEPS,
    num_training_steps=MULTI_TOTAL_STEPS
)

multi_global_step = 0

# Training tarixini saqlash uchun
multi_train_history = []
multi_validation_history = []

print("=" * 60)
print("MULTI-SAMPLE TRAINING CONFIGURATION")
print("=" * 60)

print("Optimizer:", type(multi_optimizer).__name__)
print("LoRA initial LR:", multi_optimizer.param_groups[0]["lr"])
print("CTC Head initial LR:", multi_optimizer.param_groups[1]["lr"])

print("\nTotal optimizer steps:", MULTI_TOTAL_STEPS)
print("Warmup steps:", MULTI_WARMUP_STEPS)
print("Gradient accumulation:", MULTI_GRAD_ACCUM)

print(
    "Effective batch size:",
    mini_train_loader.batch_size * MULTI_GRAD_ACCUM
)

print("Trainable parameters:", sum(
    p.numel() for p in multi_model.parameters()
    if p.requires_grad
))

assert len(multi_lora_params) == 96
assert len(multi_head_params) == 2
assert multi_global_step == 0

print("\n✅ Multi-sample optimizer and scheduler ready!")

MULTI-SAMPLE TRAINING CONFIGURATION
Optimizer: AdamW
LoRA initial LR: 0.0
CTC Head initial LR: 0.0

Total optimizer steps: 300
Warmup steps: 20
Gradient accumulation: 4
Effective batch size: 4
Trainable parameters: 823332

✅ Multi-sample optimizer and scheduler ready!


In [108]:
import torch
import jiwer
import numpy as np

from pathlib import Path
from tqdm.auto import tqdm


def evaluate_multi_model(max_samples=30):

    multi_model.eval()

    references = []
    predictions = []
    losses = []
    blank_ratios = []

    with torch.inference_mode():

        for i, batch in enumerate(mini_val_loader):

            if i >= max_samples:
                break

            labels = batch["labels"].clone()

            labels[labels == -100] = tokenizer.pad_token_id

            reference = processor.batch_decode(
                labels,
                group_tokens=False
            )[0]

            gpu_batch = {
                k: v.to("cuda")
                for k, v in batch.items()
            }

            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = multi_model(**gpu_batch)

            pred_ids = outputs.logits.argmax(dim=-1)

            prediction = processor.batch_decode(
                pred_ids.cpu()
            )[0]

            blank_ratio = (
                (pred_ids == tokenizer.pad_token_id)
                .float()
                .mean()
                .item() * 100
            )

            references.append(reference)
            predictions.append(prediction)
            losses.append(outputs.loss.item())
            blank_ratios.append(blank_ratio)

    result = {
        "step": multi_global_step,
        "val_loss": float(np.mean(losses)),
        "WER": jiwer.wer(references, predictions) * 100,
        "CER": jiwer.cer(references, predictions) * 100,
        "blank_ratio": float(np.mean(blank_ratios)),
    }

    print(
        f"\nStep {multi_global_step} | "
        f"Val Loss: {result['val_loss']:.4f} | "
        f"WER: {result['WER']:.2f}% | "
        f"CER: {result['CER']:.2f}% | "
        f"Blank: {result['blank_ratio']:.2f}%"
    )

    print("Prediction:", repr(predictions[0][:150]))

    return result


def save_multi_checkpoint():

    save_dir = (
        PROJECT_DIR
        / "experiments"
        / f"multi_lora_step_{multi_global_step}"
    )

    save_dir.mkdir(parents=True, exist_ok=True)

    trainable_state = {
        name: param.detach().cpu().clone()
        for name, param in multi_model.named_parameters()
        if param.requires_grad
    }

    multi_model.peft_config["default"].save_pretrained(
        str(save_dir)
    )

    processor.save_pretrained(str(save_dir))

    state = {
        "global_step": multi_global_step,
        "model_weights": trainable_state,
        "optimizer_state": multi_optimizer.state_dict(),
        "scheduler_state": multi_scheduler.state_dict(),
        "training_history": multi_train_history,
        "validation_history": multi_validation_history,
        "base_model_name": MODEL_NAME,
        "vocab_size": len(tokenizer),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": torch.cuda.get_rng_state_all(),
    }

    # Avval vaqtinchalik faylga yozamiz
    final_path = save_dir / "training_state.pt"
    temp_path = save_dir / "training_state.tmp"

    torch.save(state, temp_path)
    temp_path.replace(final_path)

    print(f"✅ Checkpoint saved: step {multi_global_step}")

In [109]:
assert multi_global_step == 0
assert multi_scheduler.last_epoch == 0

trainable_params = [
    p for p in multi_model.parameters()
    if p.requires_grad
]

train_iterator = iter(mini_train_loader)

multi_optimizer.zero_grad(set_to_none=True)

# Trainingdan oldingi validation
initial_result = evaluate_multi_model()
multi_validation_history.append(initial_result)

progress = tqdm(
    total=MULTI_TOTAL_STEPS,
    desc="Multi-Sample XLS-R + LoRA"
)

try:

    while multi_global_step < MULTI_TOTAL_STEPS:

        multi_model.train()

        # Pretrained encoder dropoutlarini o'chiramiz.
        # LoRA weightlari trainable bo'lib qoladi.
        multi_model.base_model.model.wav2vec2.eval()

        accumulated_loss = 0.0

        for _ in range(MULTI_GRAD_ACCUM):

            try:
                batch = next(train_iterator)

            except StopIteration:
                train_iterator = iter(mini_train_loader)
                batch = next(train_iterator)

            batch = {
                k: v.to("cuda", non_blocking=True)
                for k, v in batch.items()
            }

            with torch.autocast("cuda", dtype=torch.bfloat16):

                outputs = multi_model(**batch)

                loss = outputs.loss / MULTI_GRAD_ACCUM

            if not torch.isfinite(loss).item():
                raise RuntimeError(
                    f"Invalid loss at step {multi_global_step}"
                )

            loss.backward()
            accumulated_loss += loss.detach().item()

            del outputs, loss, batch

        grad_norm = torch.nn.utils.clip_grad_norm_(
            trainable_params,
            max_norm=1.0,
            error_if_nonfinite=True
        )

        multi_optimizer.step()
        multi_scheduler.step()
        multi_optimizer.zero_grad(set_to_none=True)

        multi_global_step += 1

        multi_train_history.append({
            "step": multi_global_step,
            "loss": accumulated_loss,
            "grad_norm": float(grad_norm.item())
        })

        progress.update(1)

        progress.set_postfix({
            "step": multi_global_step,
            "loss": f"{accumulated_loss:.4f}"
        })

        # Har 50 stepda validation
        if multi_global_step % 50 == 0:

            result = evaluate_multi_model()

            multi_validation_history.append(result)

            # Validationdan keyin checkpoint
            save_multi_checkpoint()

finally:
    progress.close()

print("\n✅ Multi-sample training completed!")
print("Global step:", multi_global_step)


Step 0 | Val Loss: 8.4869 | WER: 100.00% | CER: 96.76% | Blank: 0.00%
Prediction: 'wfwfwfwxwfwfwfwfwfwfwfxwfwáfwáwfwfwífwfxwfwfádwxwfwxwfwfw[UNK]wfwáwfwdwfwfxfwxwfwxwáwxwfwá'


Multi-Sample XLS-R + LoRA:   0%|          | 0/300 [00:00<?, ?it/s]


Step 50 | Val Loss: 3.1689 | WER: 100.00% | CER: 99.87% | Blank: 99.95%
Prediction: ''
✅ Checkpoint saved: step 50

Step 100 | Val Loss: 3.1253 | WER: 100.00% | CER: 99.43% | Blank: 99.80%
Prediction: 'ıı'
✅ Checkpoint saved: step 100

Step 150 | Val Loss: 3.1142 | WER: 100.00% | CER: 98.94% | Blank: 99.61%
Prediction: 'ııı'
✅ Checkpoint saved: step 150

Step 200 | Val Loss: 3.0846 | WER: 100.00% | CER: 96.19% | Blank: 98.39%
Prediction: 'stssstss'
✅ Checkpoint saved: step 200

Step 250 | Val Loss: 3.0592 | WER: 100.00% | CER: 95.17% | Blank: 97.18%
Prediction: 'ssqsssssstsss'
✅ Checkpoint saved: step 250

Step 300 | Val Loss: 3.0483 | WER: 100.00% | CER: 95.00% | Blank: 96.72%
Prediction: 'sssqsssssstssss'
✅ Checkpoint saved: step 300

✅ Multi-sample training completed!
Global step: 300


In [110]:
import pandas as pd

multi_results = pd.DataFrame(multi_validation_history)

display(
    multi_results[
        ["step", "val_loss", "WER", "CER", "blank_ratio"]
    ]
)

,step,val_loss,WER,CER,blank_ratio
0,0,8.486918,100.0,96.755545,0.000000
1,50,3.168944,100.0,99.870222,99.948325
2,100,3.125303,100.0,99.433695,99.803178
3,150,3.114231,100.0,98.938178,99.606043
4,200,3.084556,100.0,96.189240,98.393765
5,250,3.059195,100.0,95.174611,97.180630
6,300,3.048277,100.0,94.997640,96.720434


In [111]:
import torch
import jiwer
import numpy as np
from torch.utils.data import DataLoader


def evaluate_dataset(model, loader, max_samples=30):

    model.eval()

    references = []
    predictions = []
    blank_ratios = []

    with torch.inference_mode():

        for i, batch in enumerate(loader):

            if i >= max_samples:
                break

            labels = batch["labels"].clone()
            labels[labels == -100] = tokenizer.pad_token_id

            reference = processor.batch_decode(
                labels,
                group_tokens=False
            )[0]

            gpu_batch = {
                k: v.to("cuda")
                for k, v in batch.items()
            }

            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = model(**gpu_batch)

            pred_ids = outputs.logits.argmax(dim=-1)

            prediction = processor.batch_decode(
                pred_ids.cpu()
            )[0]

            references.append(reference)
            predictions.append(prediction)

            blank_ratios.append(
                (pred_ids == tokenizer.pad_token_id)
                .float()
                .mean()
                .item() * 100
            )

    return {
        "WER": jiwer.wer(references, predictions) * 100,
        "CER": jiwer.cer(references, predictions) * 100,
        "blank_ratio": float(np.mean(blank_ratios)),
        "prediction": predictions[0]
    }


# Train dataset uchun shuffle=False
train_eval_loader = DataLoader(
    mini_train_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=ctc_collate_fn,
    num_workers=0
)

train_result = evaluate_dataset(
    multi_model,
    train_eval_loader,
    max_samples=30
)

val_result = evaluate_dataset(
    multi_model,
    mini_val_loader,
    max_samples=30
)

print("=" * 60)
print("TRAIN VS VALIDATION — STEP 300")
print("=" * 60)

print("\nTRAIN")
print("WER:", round(train_result["WER"], 2), "%")
print("CER:", round(train_result["CER"], 2), "%")
print("Blank:", round(train_result["blank_ratio"], 2), "%")
print("Prediction:", repr(train_result["prediction"][:150]))

print("\nVALIDATION")
print("WER:", round(val_result["WER"], 2), "%")
print("CER:", round(val_result["CER"], 2), "%")
print("Blank:", round(val_result["blank_ratio"], 2), "%")
print("Prediction:", repr(val_result["prediction"][:150]))

TRAIN VS VALIDATION — STEP 300

TRAIN
WER: 100.0 %
CER: 94.71 %
Blank: 96.65 %
Prediction: 'qsqssssssssqssss'

VALIDATION
WER: 100.0 %
CER: 95.0 %
Blank: 96.72 %
Prediction: 'sssqsssssstssss'


In [112]:
print("=" * 60)
print("MULTI-SAMPLE TRAINING STATE")
print("=" * 60)

print("Global step:", multi_global_step)
print("Scheduler step:", multi_scheduler.last_epoch)

print("\nOptimizer learning rates:")

for i, group in enumerate(multi_optimizer.param_groups):
    print(
        f"Group {i}:",
        group["lr"]
    )

print("\nTrainable parameters:")

trainable = sum(
    p.numel()
    for p in multi_model.parameters()
    if p.requires_grad
)

print(f"{trainable:,}")

print("\nOptimizer state entries:", len(multi_optimizer.state))

print("\n✅ Training state verified!")

MULTI-SAMPLE TRAINING STATE
Global step: 300
Scheduler step: 300

Optimizer learning rates:
Group 0: 0.0
Group 1: 0.0

Trainable parameters:
823,332

Optimizer state entries: 98

✅ Training state verified!


In [113]:
import torch
from transformers import get_linear_schedule_with_warmup

# Hozirgi training holatini tekshiramiz
assert multi_global_step == 300
assert multi_scheduler.last_epoch == 300
assert len(multi_optimizer.state) == 98

# Yangi training rejasi
NEXT_TARGET_STEP = 1000
NEW_WARMUP_STEPS = 20

remaining_steps = NEXT_TARGET_STEP - multi_global_step

# Learning rate'larni qayta o'rnatamiz
multi_optimizer.param_groups[0]["lr"] = 1e-4
multi_optimizer.param_groups[1]["lr"] = 1e-3

# Oldingi scheduler o'rniga yangisini yaratamiz
multi_scheduler = get_linear_schedule_with_warmup(
    multi_optimizer,
    num_warmup_steps=NEW_WARMUP_STEPS,
    num_training_steps=remaining_steps
)

print("=" * 60)
print("TRAINING RESUME CONFIGURATION")
print("=" * 60)

print("Current step:", multi_global_step)
print("Target step:", NEXT_TARGET_STEP)
print("Remaining steps:", remaining_steps)

print("\nOptimizer state entries:", len(multi_optimizer.state))
print("New scheduler step:", multi_scheduler.last_epoch)

print("\nLearning rates after scheduler initialization:")
print("LoRA LR:", multi_optimizer.param_groups[0]["lr"])
print("CTC Head LR:", multi_optimizer.param_groups[1]["lr"])

print("\n✅ Optimizer preserved, new scheduler ready!")

TRAINING RESUME CONFIGURATION
Current step: 300
Target step: 1000
Remaining steps: 700

Optimizer state entries: 98
New scheduler step: 0

Learning rates after scheduler initialization:
LoRA LR: 0.0
CTC Head LR: 0.0

✅ Optimizer preserved, new scheduler ready!


In [114]:
import torch
from tqdm.auto import tqdm

assert multi_global_step == 300
assert multi_scheduler.last_epoch == 0
assert len(multi_optimizer.state) == 98

TARGET_STEP = 1000

trainable_params = [
    p for p in multi_model.parameters()
    if p.requires_grad
]

train_iterator = iter(mini_train_loader)

multi_optimizer.zero_grad(set_to_none=True)

progress = tqdm(
    total=TARGET_STEP - multi_global_step,
    desc="XLS-R + LoRA | Step 301–1000"
)

try:
    while multi_global_step < TARGET_STEP:

        multi_model.train()

        # Encoderning dropout qatlamlarini o'chiramiz.
        # LoRA adapterlari trainable bo'lib qoladi.
        multi_model.base_model.model.wav2vec2.eval()

        accumulated_loss = 0.0

        for _ in range(MULTI_GRAD_ACCUM):

            try:
                batch = next(train_iterator)

            except StopIteration:
                train_iterator = iter(mini_train_loader)
                batch = next(train_iterator)

            batch = {
                key: value.to("cuda", non_blocking=True)
                for key, value in batch.items()
            }

            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = multi_model(**batch)
                loss = outputs.loss / MULTI_GRAD_ACCUM

            if not torch.isfinite(loss).item():
                raise RuntimeError(
                    f"Non-finite loss at step {multi_global_step}"
                )

            loss.backward()
            accumulated_loss += loss.detach().item()

            del outputs, loss, batch

        grad_norm = torch.nn.utils.clip_grad_norm_(
            trainable_params,
            max_norm=1.0,
            error_if_nonfinite=True
        )

        multi_optimizer.step()
        multi_scheduler.step()
        multi_optimizer.zero_grad(set_to_none=True)

        multi_global_step += 1

        multi_train_history.append({
            "step": multi_global_step,
            "loss": accumulated_loss,
            "grad_norm": float(grad_norm.item()),
            "lora_lr": multi_optimizer.param_groups[0]["lr"],
            "ctc_lr": multi_optimizer.param_groups[1]["lr"]
        })

        progress.update(1)

        progress.set_postfix({
            "step": multi_global_step,
            "loss": f"{accumulated_loss:.4f}"
        })

        # Har 100 stepda validation
        if multi_global_step % 100 == 0:

            print("\n" + "=" * 60)
            print(f"VALIDATION — STEP {multi_global_step}")
            print("=" * 60)

            # Train datasetdan 30 ta namuna
            train_result = evaluate_dataset(
                multi_model,
                train_eval_loader,
                max_samples=30
            )

            # Validation datasetdan 30 ta namuna
            val_result = evaluate_dataset(
                multi_model,
                mini_val_loader,
                max_samples=30
            )

            print(
                f"TRAIN | "
                f"WER: {train_result['WER']:.2f}% | "
                f"CER: {train_result['CER']:.2f}% | "
                f"Blank: {train_result['blank_ratio']:.2f}%"
            )

            print(
                f"VAL   | "
                f"WER: {val_result['WER']:.2f}% | "
                f"CER: {val_result['CER']:.2f}% | "
                f"Blank: {val_result['blank_ratio']:.2f}%"
            )

            print(
                "Val prediction:",
                repr(val_result["prediction"][:150])
            )

            multi_validation_history.append({
                "step": multi_global_step,
                "train_WER": train_result["WER"],
                "train_CER": train_result["CER"],
                "train_blank": train_result["blank_ratio"],
                "val_WER": val_result["WER"],
                "val_CER": val_result["CER"],
                "val_blank": val_result["blank_ratio"]
            })

            # Har 100 stepda checkpoint
            save_multi_checkpoint()

finally:
    progress.close()

print("\n✅ Training completed!")
print("Global step:", multi_global_step)

XLS-R + LoRA | Step 301–1000:   0%|          | 0/700 [00:00<?, ?it/s]


VALIDATION — STEP 400
TRAIN | WER: 100.00% | CER: 76.37% | Blank: 85.89%
VAL   | WER: 100.00% | CER: 78.49% | Blank: 86.85%
Val prediction: 'snatssarjttsn qatsshtsmketirtkkqbrmtkemrasnnǵeemqbressdrjndrbshkrs'
✅ Checkpoint saved: step 400

VALIDATION — STEP 500
TRAIN | WER: 99.20% | CER: 35.51% | Blank: 59.89%
VAL   | WER: 99.41% | CER: 41.53% | Blank: 61.27%
Val prediction: 'ndaqqdelegcseaǵzaları lajlarda belsneqatnasıshıpshmamleketlergrtekalıq bilemlendrmakemellerarasındaǵılmhámaedmlıq brgeslezidiraojlandırbonshapkralıstı'
✅ Checkpoint saved: step 500

VALIDATION — STEP 600
TRAIN | WER: 85.49% | CER: 21.61% | Blank: 52.93%
VAL   | WER: 90.50% | CER: 29.28% | Blank: 54.02%
Val prediction: 'sondayqqdelegcya aǵzaları lajlarda belsene qatnası shıpısha mámleketleri retikalıq bilemlendirıw mákemeleri arasındaǵıilm hám admiylıq birge slezsider'
✅ Checkpoint saved: step 600

VALIDATION — STEP 700
TRAIN | WER: 72.77% | CER: 16.78% | Blank: 50.03%
VAL   | WER: 85.90% | CER: 25.96% | Blank: 51.

In [115]:
import pandas as pd

results_df = pd.DataFrame(multi_validation_history)

display(
    results_df[
        [
            "step",
            "train_WER",
            "train_CER",
            "train_blank",
            "val_WER",
            "val_CER",
            "val_blank"
        ]
    ].dropna(subset=["train_CER"])
)

,step,train_WER,train_CER,train_blank,val_WER,val_CER,val_blank
7,400,100.000000,76.366802,85.892664,100.000000,78.492213,86.846289
8,500,99.199199,35.507835,59.885938,99.412341,41.529023,61.269554
9,600,85.485485,21.613465,52.934830,90.499510,29.282681,54.020000
10,700,72.772773,16.784678,50.032043,85.896180,25.955639,51.135566
11,800,68.368368,14.822983,48.874506,84.818805,24.929212,50.034992
12,900,64.664665,13.569356,47.419909,82.174339,24.256725,48.616903
13,1000,64.464464,13.163088,47.267166,82.761998,24.150543,48.288543


In [116]:
from pathlib import Path
import torch

checkpoint_dir = (
    PROJECT_DIR
    / "experiments"
    / "multi_lora_step_1000"
)

checkpoint_path = checkpoint_dir / "training_state.pt"

assert checkpoint_path.exists(), "Checkpoint topilmadi!"

state = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("=" * 55)
print("MULTI-SAMPLE CHECKPOINT VERIFICATION")
print("=" * 55)

print("Checkpoint step:", state["global_step"])
print("Trainable tensors:", len(state["model_weights"]))

print(
    "Checkpoint size:",
    round(checkpoint_path.stat().st_size / 1024**2, 2),
    "MB"
)

print("Validation history:", len(state["validation_history"]))

assert state["global_step"] == 1000
assert len(state["model_weights"]) == 98

print("\n✅ 1000-step checkpoint verified!")

MULTI-SAMPLE CHECKPOINT VERIFICATION
Checkpoint step: 1000
Trainable tensors: 98
Checkpoint size: 9.61 MB
Validation history: 14

✅ 1000-step checkpoint verified!


In [117]:
import torch
from transformers import Wav2Vec2ForCTC
from peft import LoraConfig, get_peft_model

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# 1. Pretrained XLS-R-300M
full_base = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True
)

# 2. Pretrained weightlarni muzlatamiz
for param in full_base.parameters():
    param.requires_grad = False

# 3. LayerDrop o'chiriladi
full_base.config.layerdrop = 0.0

# 4. LoRA konfiguratsiyasi
full_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    modules_to_save=["lm_head"]
)

# 5. LoRA adapterlarini qo'shamiz
full_model = get_peft_model(
    full_base,
    full_lora_config
).to("cuda")

# 6. Trainable parametrlarni ajratamiz
full_lora_params = []
full_head_params = []

for name, param in full_model.named_parameters():

    if not param.requires_grad:
        continue

    if "lora_" in name:
        full_lora_params.append(param)

    elif "lm_head" in name:
        full_head_params.append(param)

    else:
        raise ValueError(
            f"Unexpected trainable parameter: {name}"
        )

lora_count = sum(p.numel() for p in full_lora_params)
head_count = sum(p.numel() for p in full_head_params)

assert lora_count == 786432
assert head_count == 36900

print("=" * 60)
print("FULL DATASET XLS-R + LoRA MODEL")
print("=" * 60)

print("LoRA parameters:", f"{lora_count:,}")
print("CTC Head parameters:", f"{head_count:,}")
print("Total trainable:", f"{lora_count + head_count:,}")
print("LayerDrop:", full_model.config.layerdrop)
print("Device:", next(full_model.parameters()).device)

print("\n✅ Full dataset LoRA model ready!")

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


FULL DATASET XLS-R + LoRA MODEL
LoRA parameters: 786,432
CTC Head parameters: 36,900
Total trainable: 823,332
LayerDrop: 0.0
Device: cuda:0

✅ Full dataset LoRA model ready!


In [118]:
import math
import torch

from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup

# =====================================
# 1. TRAINING CONFIG
# =====================================

FULL_BATCH_SIZE = 1
FULL_GRAD_ACCUM = 4
FULL_EPOCHS = 3

LORA_LR = 1e-4
CTC_LR = 1e-3

# =====================================
# 2. DATALOADERS
# =====================================

full_train_loader = DataLoader(
    train_dataset,
    batch_size=FULL_BATCH_SIZE,
    shuffle=True,
    collate_fn=ctc_collate_fn,
    num_workers=0,
    pin_memory=True
)

full_val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=ctc_collate_fn,
    num_workers=0,
    pin_memory=True
)

# =====================================
# 3. TRAINING STEPS
# =====================================

steps_per_epoch = math.ceil(
    len(full_train_loader) / FULL_GRAD_ACCUM
)

FULL_TOTAL_STEPS = steps_per_epoch * FULL_EPOCHS

FULL_WARMUP_STEPS = int(FULL_TOTAL_STEPS * 0.05)

# =====================================
# 4. OPTIMIZER
# =====================================

full_optimizer = torch.optim.AdamW(
    [
        {
            "params": full_lora_params,
            "lr": LORA_LR,
            "weight_decay": 0.0
        },
        {
            "params": full_head_params,
            "lr": CTC_LR,
            "weight_decay": 0.0
        }
    ]
)

# =====================================
# 5. SCHEDULER
# =====================================

full_scheduler = get_linear_schedule_with_warmup(
    full_optimizer,
    num_warmup_steps=FULL_WARMUP_STEPS,
    num_training_steps=FULL_TOTAL_STEPS
)

# =====================================
# 6. TRAINING STATE
# =====================================

full_global_step = 0

full_train_history = []
full_validation_history = []

# =====================================
# 7. VERIFICATION
# =====================================

assert len(full_train_loader.dataset) == 17362
assert len(full_val_loader.dataset) == 965

assert len(full_lora_params) == 96
assert len(full_head_params) == 2

print("=" * 60)
print("FULL DATASET TRAINING CONFIGURATION")
print("=" * 60)

print("Train samples:", len(full_train_loader.dataset))
print("Validation samples:", len(full_val_loader.dataset))

print("\nBatch size:", FULL_BATCH_SIZE)
print("Gradient accumulation:", FULL_GRAD_ACCUM)
print("Effective batch size:", FULL_BATCH_SIZE * FULL_GRAD_ACCUM)

print("\nEpochs:", FULL_EPOCHS)
print("Steps per epoch:", steps_per_epoch)
print("Total optimizer steps:", FULL_TOTAL_STEPS)
print("Warmup steps:", FULL_WARMUP_STEPS)

print("\nLoRA target LR:", LORA_LR)
print("CTC Head target LR:", CTC_LR)

print("\nOptimizer state entries:", len(full_optimizer.state))
print("Scheduler step:", full_scheduler.last_epoch)

print("\n✅ Full dataset training configuration ready!")

FULL DATASET TRAINING CONFIGURATION
Train samples: 17362
Validation samples: 965

Batch size: 1
Gradient accumulation: 4
Effective batch size: 4

Epochs: 3
Steps per epoch: 4341
Total optimizer steps: 13023
Warmup steps: 651

LoRA target LR: 0.0001
CTC Head target LR: 0.001

Optimizer state entries: 0
Scheduler step: 0

✅ Full dataset training configuration ready!


In [119]:
import torch
from pathlib import Path
from tqdm.auto import tqdm

# =====================================
# 1. CHECKPOINT FUNCTION
# =====================================

def save_full_checkpoint():

    save_dir = (
        PROJECT_DIR
        / "checkpoints"
        / f"full_lora_step_{full_global_step}"
    )

    save_dir.mkdir(parents=True, exist_ok=True)

    trainable_state = {
        name: param.detach().cpu().clone()
        for name, param in full_model.named_parameters()
        if param.requires_grad
    }

    assert len(trainable_state) == 98

    full_model.peft_config["default"].save_pretrained(
        str(save_dir)
    )

    processor.save_pretrained(str(save_dir))

    state = {
        "global_step": full_global_step,
        "model_weights": trainable_state,
        "optimizer_state": full_optimizer.state_dict(),
        "scheduler_state": full_scheduler.state_dict(),
        "training_history": full_train_history,
        "validation_history": full_validation_history,
        "base_model_name": MODEL_NAME,
        "vocab_size": len(tokenizer),
        "grad_accum_steps": FULL_GRAD_ACCUM,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": torch.cuda.get_rng_state_all()
    }

    checkpoint_path = save_dir / "training_state.pt"
    temp_path = save_dir / "training_state.tmp"

    torch.save(state, temp_path)
    temp_path.replace(checkpoint_path)

    print(f"\n✅ Checkpoint saved: step {full_global_step}")

    return checkpoint_path


# =====================================
# 2. TRAINING TEST
# =====================================

assert full_global_step == 0
assert full_scheduler.last_epoch == 0

TEST_STEPS = 100

trainable_params = [
    p for p in full_model.parameters()
    if p.requires_grad
]

full_train_iterator = iter(full_train_loader)

full_optimizer.zero_grad(set_to_none=True)

torch.cuda.reset_peak_memory_stats()

progress = tqdm(
    range(TEST_STEPS),
    desc="Full Dataset XLS-R + LoRA"
)

for _ in progress:

    full_model.train()

    # Pretrained encoder dropoutlarini o'chiramiz
    full_model.base_model.model.wav2vec2.eval()

    accumulated_loss = 0.0

    for micro_step in range(FULL_GRAD_ACCUM):

        try:
            batch = next(full_train_iterator)

        except StopIteration:
            full_train_iterator = iter(full_train_loader)
            batch = next(full_train_iterator)

        batch = {
            key: value.to("cuda", non_blocking=True)
            for key, value in batch.items()
        }

        with torch.autocast("cuda", dtype=torch.bfloat16):

            outputs = full_model(**batch)

            loss = outputs.loss / FULL_GRAD_ACCUM

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Non-finite loss at step {full_global_step}"
            )

        loss.backward()

        accumulated_loss += loss.detach().item()

        del outputs, loss, batch

    grad_norm = torch.nn.utils.clip_grad_norm_(
        trainable_params,
        max_norm=1.0,
        error_if_nonfinite=True
    )

    full_optimizer.step()
    full_scheduler.step()
    full_optimizer.zero_grad(set_to_none=True)

    full_global_step += 1

    full_train_history.append({
        "step": full_global_step,
        "loss": accumulated_loss,
        "grad_norm": float(grad_norm.item()),
        "lora_lr": full_optimizer.param_groups[0]["lr"],
        "ctc_lr": full_optimizer.param_groups[1]["lr"]
    })

    progress.set_postfix({
        "step": full_global_step,
        "loss": f"{accumulated_loss:.4f}"
    })

    if full_global_step % 25 == 0:

        print(
            f"\nStep {full_global_step} | "
            f"Loss: {accumulated_loss:.4f} | "
            f"Grad norm: {grad_norm.item():.4f} | "
            f"LoRA LR: {full_optimizer.param_groups[0]['lr']:.2e} | "
            f"CTC LR: {full_optimizer.param_groups[1]['lr']:.2e}"
        )

    if full_global_step % 50 == 0:
        save_full_checkpoint()


# =====================================
# 3. RESULTS
# =====================================

print("\n" + "=" * 60)
print("FULL DATASET — FIRST 100 STEPS")
print("=" * 60)

print("Global step:", full_global_step)

print(
    "First 10 steps average loss:",
    round(
        sum(x["loss"] for x in full_train_history[:10]) / 10,
        4
    )
)

print(
    "Last 10 steps average loss:",
    round(
        sum(x["loss"] for x in full_train_history[-10:]) / 10,
        4
    )
)

print(
    "Peak GPU allocated:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB"
)

print("Scheduler step:", full_scheduler.last_epoch)

print("\n✅ First 100 training steps completed!")

Full Dataset XLS-R + LoRA:   0%|          | 0/100 [00:00<?, ?it/s]


Step 25 | Loss: 9.0753 | Grad norm: 5.9158 | LoRA LR: 3.84e-06 | CTC LR: 3.84e-05

Step 50 | Loss: 8.3322 | Grad norm: 6.1808 | LoRA LR: 7.68e-06 | CTC LR: 7.68e-05

✅ Checkpoint saved: step 50

Step 75 | Loss: 6.3644 | Grad norm: 5.2061 | LoRA LR: 1.15e-05 | CTC LR: 1.15e-04

Step 100 | Loss: 6.2053 | Grad norm: 7.4489 | LoRA LR: 1.54e-05 | CTC LR: 1.54e-04

✅ Checkpoint saved: step 100

FULL DATASET — FIRST 100 STEPS
Global step: 100
First 10 steps average loss: 8.7077
Last 10 steps average loss: 6.5379
Peak GPU allocated: 9.14 GB
Scheduler step: 100

✅ First 100 training steps completed!


In [120]:
import torch
import jiwer
import numpy as np

from tqdm.auto import tqdm


def evaluate_full_model(max_samples=30):

    full_model.eval()

    references = []
    predictions = []
    losses = []
    blank_ratios = []

    with torch.inference_mode():

        for i, batch in enumerate(
            tqdm(full_val_loader, total=max_samples)
        ):

            if i >= max_samples:
                break

            labels = batch["labels"].clone()

            labels[labels == -100] = tokenizer.pad_token_id

            reference = processor.batch_decode(
                labels,
                group_tokens=False
            )[0]

            gpu_batch = {
                k: v.to("cuda", non_blocking=True)
                for k, v in batch.items()
            }

            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = full_model(**gpu_batch)

            pred_ids = outputs.logits.argmax(dim=-1)

            prediction = processor.batch_decode(
                pred_ids.cpu()
            )[0]

            blank_ratio = (
                (pred_ids == tokenizer.pad_token_id)
                .float()
                .mean()
                .item() * 100
            )

            references.append(reference)
            predictions.append(prediction)

            losses.append(outputs.loss.item())
            blank_ratios.append(blank_ratio)

            del outputs, gpu_batch

    result = {
        "step": full_global_step,
        "samples": len(references),
        "val_loss": float(np.mean(losses)),
        "WER": jiwer.wer(references, predictions) * 100,
        "CER": jiwer.cer(references, predictions) * 100,
        "blank_ratio": float(np.mean(blank_ratios)),
    }

    print("\n" + "=" * 60)
    print("FULL DATASET VALIDATION")
    print("=" * 60)

    for key, value in result.items():
        print(f"{key}: {value:.4f}" if isinstance(value, float)
              else f"{key}: {value}")

    print("\nPREDICTION EXAMPLES")

    for i in range(min(3, len(predictions))):
        print(f"\nExample {i+1}")
        print("Reference:", references[i][:150])
        print("Prediction:", repr(predictions[i][:150]))

    return result


# 100-step validation
result_100 = evaluate_full_model(max_samples=30)

full_validation_history.append(result_100)

  0%|          | 0/30 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 100
samples: 30
val_loss: 6.1885
WER: 100.0000
CER: 99.6461
blank_ratio: 0.0000

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: ''

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'ı'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: ''


In [121]:
import torch
from collections import Counter

full_model.eval()

batch = next(iter(full_val_loader))

gpu_batch = {
    key: value.to("cuda")
    for key, value in batch.items()
}

with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.bfloat16):
        outputs = full_model(
            input_values=gpu_batch["input_values"],
            attention_mask=gpu_batch["attention_mask"]
        )

logits = outputs.logits.float()
pred_ids = logits.argmax(dim=-1)[0]

token_counts = Counter(pred_ids.cpu().tolist())

print("=" * 60)
print("CTC TOKEN DISTRIBUTION — STEP", full_global_step)
print("=" * 60)

print("Total frames:", len(pred_ids))

print("\nTOP 10 PREDICTED TOKENS")

for token_id, count in token_counts.most_common(10):

    token = tokenizer.convert_ids_to_tokens(token_id)

    print(
        f"{token!r:10s} | "
        f"Count: {count:4d} | "
        f"Ratio: {count / len(pred_ids) * 100:.2f}%"
    )

print("\nBlank ID:", tokenizer.pad_token_id)

print(
    "Blank frames:",
    token_counts.get(tokenizer.pad_token_id, 0)
)

print(
    "\nDecoded prediction:",
    repr(processor.batch_decode(pred_ids.unsqueeze(0).cpu())[0][:200])
)

print("\n✅ Token distribution checked!")

CTC TOKEN DISTRIBUTION — STEP 100
Total frames: 738

TOP 10 PREDICTED TOKENS
'|'        | Count:  738 | Ratio: 100.00%

Blank ID: 35
Blank frames: 0

Decoded prediction: ''

✅ Token distribution checked!


In [122]:
import torch
from tqdm.auto import tqdm

TARGET_STEP = 500

assert full_global_step == 100
assert full_scheduler.last_epoch == 100

trainable_params = [
    p for p in full_model.parameters()
    if p.requires_grad
]

full_optimizer.zero_grad(set_to_none=True)

progress = tqdm(
    total=TARGET_STEP - full_global_step,
    desc="Full XLS-R + LoRA Training"
)

try:
    while full_global_step < TARGET_STEP:

        full_model.train()

        # Oldingi tajribada ishlagan konfiguratsiya
        full_model.base_model.model.wav2vec2.eval()

        accumulated_loss = 0.0

        for _ in range(FULL_GRAD_ACCUM):

            try:
                batch = next(full_train_iterator)

            except StopIteration:
                full_train_iterator = iter(full_train_loader)
                batch = next(full_train_iterator)

            batch = {
                k: v.to("cuda", non_blocking=True)
                for k, v in batch.items()
            }

            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = full_model(**batch)
                loss = outputs.loss / FULL_GRAD_ACCUM

            if not torch.isfinite(loss).item():
                raise RuntimeError(
                    f"Non-finite loss at step {full_global_step}"
                )

            loss.backward()
            accumulated_loss += loss.detach().item()

            del outputs, loss, batch

        grad_norm = torch.nn.utils.clip_grad_norm_(
            trainable_params,
            max_norm=1.0,
            error_if_nonfinite=True
        )

        full_optimizer.step()
        full_scheduler.step()
        full_optimizer.zero_grad(set_to_none=True)

        full_global_step += 1

        full_train_history.append({
            "step": full_global_step,
            "loss": accumulated_loss,
            "grad_norm": float(grad_norm.item()),
            "lora_lr": full_optimizer.param_groups[0]["lr"],
            "ctc_lr": full_optimizer.param_groups[1]["lr"]
        })

        progress.update(1)

        progress.set_postfix({
            "step": full_global_step,
            "loss": f"{accumulated_loss:.4f}"
        })

        # Har 100-stepda validation
        if full_global_step % 100 == 0:

            result = evaluate_full_model(max_samples=30)

            full_validation_history.append(result)

            # Har 100-stepda checkpoint
            save_full_checkpoint()

finally:
    progress.close()

print("\n✅ Training reached step:", full_global_step)

Full XLS-R + LoRA Training:   0%|          | 0/400 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 200
samples: 30
val_loss: 3.1422
WER: 100.0000
CER: 100.0000
blank_ratio: 100.0000

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: ''

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: ''

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: ''

✅ Checkpoint saved: step 200


  0%|          | 0/30 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 300
samples: 30
val_loss: 3.1271
WER: 100.0000
CER: 99.9882
blank_ratio: 99.9946

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: ''

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: ''

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: ''

✅ Checkpoint saved: step 300


  0%|          | 0/30 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 400
samples: 30
val_loss: 3.1079
WER: 100.0000
CER: 99.5281
blank_ratio: 99.8361

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: ''

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'i'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: ''

✅ Checkpoint saved: step 400


  0%|          | 0/30 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 500
samples: 30
val_loss: 2.9081
WER: 100.0000
CER: 92.5437
blank_ratio: 94.7099

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'ssqssstsstqtsqsstss'

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'qsssssqsstistssttsssstqttssssqssstsssss'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'ittsssqsssqssssqssqts'

✅ Checkpoint saved: step 500

✅ Training reached step: 500


In [123]:
import torch
from tqdm.auto import tqdm

TARGET_STEP = 1500

assert full_global_step == 500
assert full_scheduler.last_epoch == 500

trainable_params = [
    p for p in full_model.parameters()
    if p.requires_grad
]

full_optimizer.zero_grad(set_to_none=True)

progress = tqdm(
    total=TARGET_STEP - full_global_step,
    desc="XLS-R + LoRA | Full Dataset"
)

try:
    while full_global_step < TARGET_STEP:

        full_model.train()

        # Frozen encoder dropoutlarini o'chiramiz
        full_model.base_model.model.wav2vec2.eval()

        accumulated_loss = 0.0

        for _ in range(FULL_GRAD_ACCUM):

            try:
                batch = next(full_train_iterator)

            except StopIteration:
                full_train_iterator = iter(full_train_loader)
                batch = next(full_train_iterator)

            batch = {
                k: v.to("cuda", non_blocking=True)
                for k, v in batch.items()
            }

            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = full_model(**batch)
                loss = outputs.loss / FULL_GRAD_ACCUM

            if not torch.isfinite(loss).item():
                raise RuntimeError(
                    f"Non-finite loss at step {full_global_step}"
                )

            loss.backward()
            accumulated_loss += loss.detach().item()

            del outputs, loss, batch

        grad_norm = torch.nn.utils.clip_grad_norm_(
            trainable_params,
            max_norm=1.0,
            error_if_nonfinite=True
        )

        full_optimizer.step()
        full_scheduler.step()
        full_optimizer.zero_grad(set_to_none=True)

        full_global_step += 1

        full_train_history.append({
            "step": full_global_step,
            "loss": accumulated_loss,
            "grad_norm": float(grad_norm.item()),
            "lora_lr": full_optimizer.param_groups[0]["lr"],
            "ctc_lr": full_optimizer.param_groups[1]["lr"]
        })

        progress.update(1)

        progress.set_postfix({
            "step": full_global_step,
            "loss": f"{accumulated_loss:.4f}"
        })

        # Har 250 stepda tekshiramiz
        if full_global_step % 250 == 0:

            result = evaluate_full_model(max_samples=30)

            full_validation_history.append(result)

            save_full_checkpoint()

finally:
    progress.close()

print("\n✅ Training reached step:", full_global_step)

XLS-R + LoRA | Full Dataset:   0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 750
samples: 30
val_loss: 1.0185
WER: 88.0509
CER: 25.8141
blank_ratio: 50.8653

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'sondayqqdelegacya aǵzaları ilajlarda belsene qatnası shııshı mámleketleri rietikalıq bilemlendiriw mákemelleri  arasındaǵıilmihámaedemiylıq birge isle'

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'konferecy qarl baqsrspublikasi ministerkeńisi baslıǵınamaslaqathsi ilalgy ilimleri boynha filasoiya doktorrosent perdaǵaly dablip institutıńrustrli há'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'solseb

  0%|          | 0/30 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 1000
samples: 30
val_loss: 0.8488
WER: 75.1224
CER: 20.7645
blank_ratio: 49.9247

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'sondayaq delegaciya aǵzaları ilajlardı belsene qatnası shııshı mámleketleri ritikalıq bilemlendiriw mákemeleri arasındaǵı iimi hám ademiyalıq birge is'

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'konfereciya qararbaqsırespublikası minister keńisi baslıǵın maslaqathsi filalogiy ilimleri boynsha filosofiya doktorırosent perdaǵaliya daplip institt'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'sol s

  0%|          | 0/30 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 1250
samples: 30
val_loss: 0.7885
WER: 71.7924
CER: 19.0892
blank_ratio: 48.5132

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shııshı mámleketleri úritikalıq bilemlendiriw mákemelleri arasındaǵı ilimiy hám akdemiyalıq bi'

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'konfereciya qararbaqsta respublikası minister keńisi baslıǵını maslaqat hsi filologiy ilimleri boyınsha filosofiya doktorırosent perdaǵaliya dáblip in'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'sol s

  0%|          | 0/30 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 1500
samples: 30
val_loss: 0.7569
WER: 71.2047
CER: 19.3487
blank_ratio: 53.0698

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'sondayaq delegaciya aǵzaları ilajlardı belsene qatnasıp shııshı mámleketleri uritikalıq bilemlendiriw mákemeleri arasındaǵı iimiy hám akdemiyalıq birg'

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'konfereciya qararbaqstı respublikası minister keńisi baslıǵını maslaqat hsi filologiy ilimleri boyınsha filosofiya doktorırosent perdaǵaliya dáblip in'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'sol s

In [124]:
import torch
from tqdm.auto import tqdm

TARGET_STEP = 4341

assert full_global_step == 1500
assert full_scheduler.last_epoch == 1500

trainable_params = [
    p for p in full_model.parameters()
    if p.requires_grad
]

full_optimizer.zero_grad(set_to_none=True)

progress = tqdm(
    total=TARGET_STEP - full_global_step,
    desc="XLS-R + LoRA | Epoch 1"
)

try:
    while full_global_step < TARGET_STEP:

        full_model.train()

        # Frozen encoder dropoutlarini o'chiramiz
        full_model.base_model.model.wav2vec2.eval()

        accumulated_loss = 0.0

        for _ in range(FULL_GRAD_ACCUM):

            try:
                batch = next(full_train_iterator)

            except StopIteration:
                full_train_iterator = iter(full_train_loader)
                batch = next(full_train_iterator)

            batch = {
                k: v.to("cuda", non_blocking=True)
                for k, v in batch.items()
            }

            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = full_model(**batch)
                loss = outputs.loss / FULL_GRAD_ACCUM

            if not torch.isfinite(loss).item():
                raise RuntimeError(
                    f"Non-finite loss at step {full_global_step}"
                )

            loss.backward()
            accumulated_loss += loss.detach().item()

            del outputs, loss, batch

        grad_norm = torch.nn.utils.clip_grad_norm_(
            trainable_params,
            max_norm=1.0,
            error_if_nonfinite=True
        )

        full_optimizer.step()
        full_scheduler.step()
        full_optimizer.zero_grad(set_to_none=True)

        full_global_step += 1

        full_train_history.append({
            "step": full_global_step,
            "loss": accumulated_loss,
            "grad_norm": float(grad_norm.item()),
            "lora_lr": full_optimizer.param_groups[0]["lr"],
            "ctc_lr": full_optimizer.param_groups[1]["lr"]
        })

        progress.update(1)
        progress.set_postfix({
            "step": full_global_step,
            "loss": f"{accumulated_loss:.4f}"
        })

        # Har 250 stepda validation va checkpoint
        if full_global_step % 250 == 0:

            result = evaluate_full_model(max_samples=30)
            full_validation_history.append(result)

            save_full_checkpoint()

finally:
    progress.close()

# Birinchi epoch yakunidagi checkpoint
assert full_global_step == 4341
save_full_checkpoint()

print("\n✅ EPOCH 1 TRAINING COMPLETED!")
print("Global step:", full_global_step)

XLS-R + LoRA | Epoch 1:   0%|          | 0/2841 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 1750
samples: 30
val_loss: 0.7271
WER: 69.6376
CER: 18.3459
blank_ratio: 51.3596

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'sondayaq delegaciya aǵzaları ilajlardı belsene qatnasıp shııshı mámleketleri uritikalıq bilemlendiriw mákemeleri arasındaǵı iimi hám akdemiyalıq birge'

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'konfereciya qararbaqsta respublikası minister keńisi baslıǵını maslaqat hsi filologiy ilimleri boyınsha filosofiya doktorırosent perdaǵaliyadaáblip in'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'sol s

  0%|          | 0/30 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 2000
samples: 30
val_loss: 0.7053
WER: 66.6993
CER: 17.1661
blank_ratio: 50.1398

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shııshı mámleketleri uridtikalıq bilemlendiriw mákemelleri arasındaǵı ilimiy hám akademiyalıq '

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'konferenciya qaralbaqstı respublikası minister keńisi baslıǵınıń maslaqathesi filologiy ilimleri boyınsha filosofia doktorırosent perdaǵaliyadablip in'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'sol s

...


FULL DATASET VALIDATION
step: 3750
samples: 30
val_loss: 0.6174
WER: 60.4310
CER: 15.2548
blank_ratio: 51.7502

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'sondayaq delegaciya aǵzaları ilajlardı belsenen qatnasıp shıbıshı mámleketleri uridikalıq bilemlendiriw mákemelleri arasındaǵı ilimi hám akademiyalıq '

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'konferenciya qaralbaqsta respublikası ministr keńisi baslıǵını maslaqathisi filologiy ilimleri boyınsha filosofiya doktorı dosent perdaǵaliyadabli ins'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'sol s

  0%|          | 0/30 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 4000
samples: 30
val_loss: 0.6080
WER: 59.2556
CER: 15.0071
blank_ratio: 50.7626

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shıbıshı mámleketleri uridikalıq bilemlendiriw mákemeleri arasındaǵı illimiy hám akademiyalıq '

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'konferenciyaa qaralbaqsta respublikası ministr keńesi baslıǵınıń maslaqathesi filologiy ilimleri boyınsha filosofiya doktorı dosent perdaǵaliyada blip'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'sol s

  0%|          | 0/30 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 4250
samples: 30
val_loss: 0.6010
WER: 59.6474
CER: 15.3256
blank_ratio: 52.6619

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'sondayaq delegaciya aǵzaları ilajlarda belsenen qatnasıp shıbıshı mámleketleri uridikalıq bilemlendiriw mákemeleri arasındaǵı ilimi hám akademiyalıq b'

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'konferenciya qaralbaqsta respublikası ministr keńesi baslıǵınıń maslaqatjesi filologiy ilimleri boyınsha filosofiya doktorı dosent perdaǵaliyada bli i'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'sol s

In [125]:
# 965 ta validation audio bilan baholash
epoch1_result = evaluate_full_model(max_samples=965)

full_validation_history.append(epoch1_result)

print("\n✅ EPOCH 1 VALIDATION COMPLETED!")

  0%|          | 0/965 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 4341
samples: 965
val_loss: 0.5379
WER: 57.3145
CER: 13.6615
blank_ratio: 53.6188

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shıpıshı mámleketleri uridikalıq bilemlendiriw mákemeleri arasındaǵı ilimi hám akademiyalıq bi'

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'konferenciya qaralbaqstı respublikası ministr keńesi baslıǵınıń maslaqatjisi filologiya ilimleri boyınsha filosofiya doktorı dosent perdaǵaliyadabli i'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'sol 

In [126]:
from pathlib import Path
import torch

assert full_global_step == 4341
assert full_scheduler.last_epoch == 4341

# Oxirgi validation natijasini tekshiramiz
assert full_validation_history[-1]["step"] == 4341
assert full_validation_history[-1]["samples"] == 965

# Epoch 1 checkpointini validation natijasi bilan yangilaymiz
checkpoint_path = save_full_checkpoint()

print("=" * 60)
print("EPOCH 1 CHECKPOINT")
print("=" * 60)

print("Step:", full_global_step)
print("Checkpoint:", checkpoint_path)

print(
    "Size:",
    round(checkpoint_path.stat().st_size / 1024**2, 2),
    "MB"
)

print("Validation WER:", full_validation_history[-1]["WER"])
print("Validation CER:", full_validation_history[-1]["CER"])

print("\n✅ Epoch 1 checkpoint ready!")


✅ Checkpoint saved: step 4341
EPOCH 1 CHECKPOINT
Step: 4341
Checkpoint: /home/intern/projects/karakalpak_stt/checkpoints/full_lora_step_4341/training_state.pt
Size: 9.84 MB
Validation WER: 57.314460099993504
Validation CER: 13.661546742536382

✅ Epoch 1 checkpoint ready!


In [127]:
import torch
from tqdm.auto import tqdm

# ==========================================
# EPOCH 2 CONFIG
# ==========================================

TARGET_STEP = 8682

assert full_global_step == 4341
assert full_scheduler.last_epoch == 4341

trainable_params = [
    p for p in full_model.parameters()
    if p.requires_grad
]

assert sum(p.numel() for p in trainable_params) == 823332

full_optimizer.zero_grad(set_to_none=True)

progress = tqdm(
    total=TARGET_STEP - full_global_step,
    desc="XLS-R + LoRA | Epoch 2"
)

# ==========================================
# TRAINING
# ==========================================

try:
    while full_global_step < TARGET_STEP:

        full_model.train()

        # Pretrained encoder dropoutlarini o'chiramiz
        full_model.base_model.model.wav2vec2.eval()

        accumulated_loss = 0.0

        for _ in range(FULL_GRAD_ACCUM):

            try:
                batch = next(full_train_iterator)

            except StopIteration:
                full_train_iterator = iter(full_train_loader)
                batch = next(full_train_iterator)

            batch = {
                k: v.to("cuda", non_blocking=True)
                for k, v in batch.items()
            }

            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = full_model(**batch)
                loss = outputs.loss / FULL_GRAD_ACCUM

            if not torch.isfinite(loss).item():
                raise RuntimeError(
                    f"Non-finite loss at step {full_global_step}"
                )

            loss.backward()
            accumulated_loss += loss.detach().item()

            del outputs, loss, batch

        grad_norm = torch.nn.utils.clip_grad_norm_(
            trainable_params,
            max_norm=1.0,
            error_if_nonfinite=True
        )

        full_optimizer.step()
        full_scheduler.step()
        full_optimizer.zero_grad(set_to_none=True)

        full_global_step += 1

        full_train_history.append({
            "step": full_global_step,
            "loss": accumulated_loss,
            "grad_norm": float(grad_norm.item()),
            "lora_lr": full_optimizer.param_groups[0]["lr"],
            "ctc_lr": full_optimizer.param_groups[1]["lr"]
        })

        progress.update(1)

        progress.set_postfix({
            "step": full_global_step,
            "loss": f"{accumulated_loss:.4f}"
        })

        # Har 1000 stepda qisqa validation
        if full_global_step % 1000 == 0:

            result = evaluate_full_model(max_samples=30)
            full_validation_history.append(result)

        # Har 500 stepda checkpoint
        if full_global_step % 500 == 0:
            save_full_checkpoint()

finally:
    progress.close()

# ==========================================
# EPOCH 2 CHECKPOINT + VALIDATION
# ==========================================

assert full_global_step == 8682

save_full_checkpoint()

print("\nRunning full validation on 965 samples...")

epoch2_result = evaluate_full_model(max_samples=965)
full_validation_history.append(epoch2_result)

# Validation tarixini ham checkpointga saqlaymiz
save_full_checkpoint()

print("\n" + "=" * 60)
print("EPOCH 2 COMPLETED")
print("=" * 60)

print("Global step:", full_global_step)
print("Validation WER:", round(epoch2_result["WER"], 2), "%")
print("Validation CER:", round(epoch2_result["CER"], 2), "%")
print("Validation Loss:", round(epoch2_result["val_loss"], 4))
print("Blank ratio:", round(epoch2_result["blank_ratio"], 2), "%")

print("\n✅ EPOCH 2 COMPLETED!")

XLS-R + LoRA | Epoch 2:   0%|          | 0/4341 [00:00<?, ?it/s]


✅ Checkpoint saved: step 4500


  0%|          | 0/30 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 5000
samples: 30
val_loss: 0.5941
WER: 59.0597
CER: 15.2312
blank_ratio: 54.2450

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'sondayaq delegaciya aǵzaları ilajlardı belsene qatnasıp shıbıshı mámleketleri uridikalıq bilemlendiriw mákemeleri arasındaǵı ilimi hám akademiyalıq bi'

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'konfereciyaa qaralbaqsta respublikası ministr keńesi baslıǵınıń maslaqathesi filologiy ilimleri boyınsha filosofiya doktrı dosent perdaǵaliyadabli ins'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'sol s

  0%|          | 0/30 [00:00<?, ?it/s]

...


FULL DATASET VALIDATION
step: 7000
samples: 30
val_loss: 0.5580
WER: 55.8276
CER: 14.2874
blank_ratio: 53.6020

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shıbıshı mámleketleri yuridikalıq bilemlendiriw mákemeleri arasındaǵı illimiy hám akademiyalıq'

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'konfereciyaǵa qaralbaqsta respublikası ministr keńesi baslıǵınıń maslaqatjhesi filologiy ilimleri boyınsha filosofiya doktrı dosent perdaǵaliyada bili'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'sol s

  0%|          | 0/30 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 8000
samples: 30
val_loss: 0.5403
WER: 53.5749
CER: 13.4379
blank_ratio: 51.0347

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shıbıshı mámleketleri yuridikalıq bilemlendiriw mákemeleri arasındaǵı illimiy hám akademiyalıq'

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'konferenciyaǵa qaralbaqstanrespublikası ministri keńesi baslıǵınıń maslaqatjhisi filologiya ilimleri boyınsha filosofiya doktrı dosent perdaǵaliyada b'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'sol s

  0%|          | 0/965 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 8682
samples: 965
val_loss: 0.4803
WER: 51.6720
CER: 11.9875
blank_ratio: 54.1974

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'sondayaq delegaciya aǵzaları ilajlarda belsne qatnasıp shıbıshı mámleketleri yuridikalıq bilemlendiriw mákemeleri arasındaǵı illimiy hám akademiyalıq '

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'konfereciyaǵa qaralbaqstanrespublikası ministr keńesi baslıǵınıń maslaqathesi filologiya ilimleri boyınsha filosofiya doktrı dosent perdaǵaliyada bili'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'sol 

In [128]:
import torch
from tqdm.auto import tqdm

TARGET_STEP = 13023

assert full_global_step == 8682
assert full_scheduler.last_epoch == 8682

trainable_params = [
    p for p in full_model.parameters()
    if p.requires_grad
]

full_optimizer.zero_grad(set_to_none=True)

progress = tqdm(
    total=TARGET_STEP - full_global_step,
    desc="XLS-R + LoRA | Epoch 3"
)

try:
    while full_global_step < TARGET_STEP:

        full_model.train()

        full_model.base_model.model.wav2vec2.eval()

        accumulated_loss = 0.0

        for _ in range(FULL_GRAD_ACCUM):

            try:
                batch = next(full_train_iterator)

            except StopIteration:
                full_train_iterator = iter(full_train_loader)
                batch = next(full_train_iterator)

            batch = {
                k: v.to("cuda", non_blocking=True)
                for k, v in batch.items()
            }

            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = full_model(**batch)
                loss = outputs.loss / FULL_GRAD_ACCUM

            if not torch.isfinite(loss).item():
                raise RuntimeError(
                    f"Non-finite loss at step {full_global_step}"
                )

            loss.backward()
            accumulated_loss += loss.detach().item()

            del outputs, loss, batch

        grad_norm = torch.nn.utils.clip_grad_norm_(
            trainable_params,
            max_norm=1.0,
            error_if_nonfinite=True
        )

        full_optimizer.step()
        full_scheduler.step()
        full_optimizer.zero_grad(set_to_none=True)

        full_global_step += 1

        full_train_history.append({
            "step": full_global_step,
            "loss": accumulated_loss,
            "grad_norm": float(grad_norm.item()),
            "lora_lr": full_optimizer.param_groups[0]["lr"],
            "ctc_lr": full_optimizer.param_groups[1]["lr"]
        })

        progress.update(1)
        progress.set_postfix({
            "step": full_global_step,
            "loss": f"{accumulated_loss:.4f}"
        })

        if full_global_step % 1000 == 0:
            result = evaluate_full_model(max_samples=30)
            full_validation_history.append(result)

        if full_global_step % 500 == 0:
            save_full_checkpoint()

finally:
    progress.close()

assert full_global_step == 13023

print("\nRunning full validation on 965 samples...")

epoch3_result = evaluate_full_model(max_samples=965)
full_validation_history.append(epoch3_result)

save_full_checkpoint()

print("\n" + "=" * 60)
print("EPOCH 3 COMPLETED")
print("=" * 60)

print("Global step:", full_global_step)
print("Validation WER:", round(epoch3_result["WER"], 2), "%")
print("Validation CER:", round(epoch3_result["CER"], 2), "%")
print("Validation Loss:", round(epoch3_result["val_loss"], 4))
print("Blank ratio:", round(epoch3_result["blank_ratio"], 2), "%")

print("\n✅ EPOCH 3 COMPLETED!")

XLS-R + LoRA | Epoch 3:   0%|          | 0/4341 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 9000
samples: 30
val_loss: 0.5379
WER: 52.9873
CER: 13.1784
blank_ratio: 50.4423

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shıbıshı mámleketleri yuridikalıq bilemlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq '

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'konferenciyaǵa qaralbaqsta respublikası ministri keńesi baslıǵınıń máslaqatjhesi filologiya ilimleri boyınsha filosofia doktorı dosent perdaǵaliyada b'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'sol s

  0%|          | 0/30 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 10000
samples: 30
val_loss: 0.5310
WER: 53.7708
CER: 13.5087
blank_ratio: 52.7356

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shıbıshı mámleketleri yuridikalıq bilemlendiriw mákemeleri arasındaǵı illimiy hám akademiyalıq'

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'konfereciyaǵa qaralbaqstanrespublikası ministri keńesi baslıǵınıń maslaqatjhesi filologiya ilimleri boyınsha filosofiya doktrı dosent perdaǵaliyada bu'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'sol 

...


FULL DATASET VALIDATION
step: 12000
samples: 30
val_loss: 0.5263
WER: 53.0852
CER: 13.2492
blank_ratio: 52.5131

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shıbıshı mámleketleri yuridikalıq bilemlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq '

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'konferenciyaǵa qaralbaqstanrespublikası ministri keńesi baslıǵınıń maslaqatjhesi filologiya ilimleri boyınsha filosofiya doktrı dosent perdaǵaliyada b'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'sol 

  0%|          | 0/30 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 13000
samples: 30
val_loss: 0.5247
WER: 52.5955
CER: 13.1548
blank_ratio: 52.5004

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shıbıshı mámleketleri yuridikalıq bilemlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq '

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'konfereciyaǵa qaralbaqstanrespublikası ministri keńesi baslıǵınıń maslaqatjhesi filologiya ilimleri boyınsha filosofiya doktrı dosent perdaǵaliyada bi'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'sol 

  0%|          | 0/965 [00:00<?, ?it/s]


FULL DATASET VALIDATION
step: 13023
samples: 965
val_loss: 0.4646
WER: 49.8896
CER: 11.5254
blank_ratio: 53.4044

PREDICTION EXAMPLES

Example 1
Reference: sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq bir
Prediction: 'sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shıbıshı mámleketleri yuridikalıq bilemlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq '

Example 2
Reference: konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń másláhátshisi filologiya ilimleri boyınsha filosofiya doktorı phd docent perde
Prediction: 'konfereciyaǵa qaralbaqstanrespublikası ministri keńesi baslıǵınıń maslaqatjhesi filologiya ilimleri boyınsha filosofiya doktrı dosent perdaǵaliyada bi'

Example 3
Reference: sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent jámiyet ushın qáwipsiz degen qatnas kún tártibine shıqpaqta zamanagóy ju
Prediction: 'sol

In [129]:
import torch
from torch.utils.data import DataLoader

# =====================================
# TEST DATALOADER
# =====================================

full_test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=ctc_collate_fn,
    num_workers=0,
    pin_memory=True
)

# =====================================
# MODEL VERIFICATION
# =====================================

assert full_global_step == 13023
assert len(full_test_loader.dataset) == 965

trainable_count = sum(
    p.numel()
    for p in full_model.parameters()
    if p.requires_grad
)

assert trainable_count == 823332

# Test rejimi: dropout va boshqa train-mode
# xatti-harakatlari o'chiriladi.
full_model.eval()

print("=" * 60)
print("FINAL TEST CONFIGURATION")
print("=" * 60)

print("Model: XLS-R-300M + LoRA + CTC")
print("Completed steps:", full_global_step)
print("Test samples:", len(full_test_loader.dataset))
print("Trainable parameters:", f"{trainable_count:,}")
print("Model device:", next(full_model.parameters()).device)

print("\n✅ Test DataLoader ready!")

FINAL TEST CONFIGURATION
Model: XLS-R-300M + LoRA + CTC
Completed steps: 13023
Test samples: 965
Trainable parameters: 823,332
Model device: cuda:0

✅ Test DataLoader ready!


In [130]:
import torch
import jiwer
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

# ==========================================
# FINAL TEST EVALUATION
# ==========================================

full_model.eval()

test_references = []
test_predictions = []
test_losses = []
test_blank_ratios = []

with torch.inference_mode():

    for batch in tqdm(
        full_test_loader,
        desc="Final Test | 965 audios"
    ):

        # 1. Reference matn
        labels = batch["labels"].clone()
        labels[labels == -100] = tokenizer.pad_token_id

        reference = processor.batch_decode(
            labels,
            group_tokens=False
        )[0]

        # 2. GPU
        gpu_batch = {
            key: value.to("cuda", non_blocking=True)
            for key, value in batch.items()
        }

        # 3. Model inference
        with torch.autocast("cuda", dtype=torch.bfloat16):
            outputs = full_model(**gpu_batch)

        # 4. CTC greedy decoding
        pred_ids = outputs.logits.argmax(dim=-1)

        prediction = processor.batch_decode(
            pred_ids.cpu()
        )[0]

        # 5. Blank ratio
        blank_ratio = (
            (pred_ids == tokenizer.pad_token_id)
            .float()
            .mean()
            .item() * 100
        )

        test_references.append(reference)
        test_predictions.append(prediction)
        test_losses.append(outputs.loss.item())
        test_blank_ratios.append(blank_ratio)

        del outputs, gpu_batch

# ==========================================
# METRICS
# ==========================================

final_test_results = {
    "step": full_global_step,
    "samples": len(test_references),
    "loss": float(np.mean(test_losses)),
    "WER": jiwer.wer(
        test_references,
        test_predictions
    ) * 100,
    "CER": jiwer.cer(
        test_references,
        test_predictions
    ) * 100,
    "blank_ratio": float(np.mean(test_blank_ratios))
}

assert len(test_references) == 965

print("\n" + "=" * 60)
print("FINAL TEST RESULTS — XLS-R-300M + LoRA")
print("=" * 60)

for key, value in final_test_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

# ==========================================
# PREDICTION EXAMPLES
# ==========================================

print("\n" + "=" * 60)
print("TEST PREDICTION EXAMPLES")
print("=" * 60)

for i in range(5):
    print(f"\nExample {i + 1}")
    print("Reference :", test_references[i][:200])
    print("Prediction:", test_predictions[i][:200])

# ==========================================
# SAVE TEST RESULTS
# ==========================================

test_results_dir = PROJECT_DIR / "results"
test_results_dir.mkdir(parents=True, exist_ok=True)

test_df = pd.DataFrame({
    "reference": test_references,
    "prediction": test_predictions
})

test_df.to_csv(
    test_results_dir / "final_test_predictions.csv",
    index=False
)

pd.DataFrame([final_test_results]).to_csv(
    test_results_dir / "final_test_metrics.csv",
    index=False
)

print("\nFinal test completed!")
print("Test predictions and metrics saved!")

Final Test | 965 audios:   0%|          | 0/965 [00:00<?, ?it/s]


FINAL TEST RESULTS — XLS-R-300M + LoRA
step: 13023
samples: 965
loss: 0.4709
WER: 49.1424
CER: 11.5679
blank_ratio: 53.4870

TEST PREDICTION EXAMPLES

Example 1
Reference : paytaxtımızdıń kongressorayında mámleketimiz basshısın ázerbayjan respublikası prezidenti ilham aliev hám bmshtıń elatlı punktler boyınsha baǵdarlamasınıń atqarıwshı direktorı anaklaudiya rossbax qutl
Prediction: paytaqtımıztıń kongres orayında mámleketimis bashısın ázerbajan respublikası prezidenti ili hám aliev hám bumshtıń elatı punktler boyınsha baǵdarlamasına atqarıwsha direktori anaklawda rosbak qutlıqla

Example 2
Reference : sanlı texnologiyalar baǵdarındaǵı ózara sóylesiw bolsa eki mámleket arasında ornatılǵan zamanagóy birge islesiw kópirin jáne de bekkemleytuǵını sózsiz musılman ziyo muhammad amin súwret óza
Prediction: sanlı texnologiyalar baǵdarındaǵı óz ara sólesiw hám eki mámleket arasında ornatılǵan zamanagóy birge islesiw kópirin jáne de bekkemleytoǵına sózsiz musulman ziyo muxammet ámin suret óza


In [131]:
import sys
import subprocess

# 1. Beam Search kutubxonasini o'rnatish
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "pyctcdecode"
])

# 2. Kutubxonani tekshirish
import pyctcdecode
from pyctcdecode import build_ctcdecoder

print("=" * 60)
print("CTC BEAM SEARCH SETUP")
print("=" * 60)

print("Python:", sys.executable)
print("Vocabulary size:", len(tokenizer))
print("Blank token ID:", tokenizer.pad_token_id)

# 3. CTC Head tokenlarining aniq tartibini tekshirish
print("\nTOKEN ID → TOKEN")

for token_id in range(len(tokenizer)):
    token = tokenizer.convert_ids_to_tokens(token_id)
    print(f"{token_id:2d} → {repr(token)}")

# 4. Model va tokenizer mosligini tekshirish
assert full_model.config.vocab_size == len(tokenizer)
assert tokenizer.pad_token_id == 35

print("\n✅ CTC Beam Search setup ready!")

CTC BEAM SEARCH SETUP
Python: /home/intern/miniforge3/envs/stt/bin/python
Vocabulary size: 36
Blank token ID: 35

TOKEN ID → TOKEN
 0 → 'a'
 1 → 'b'
 2 → 'c'
 3 → 'd'
 4 → 'e'
 5 → 'f'
 6 → 'g'
 7 → 'h'
 8 → 'i'
 9 → 'j'
10 → 'k'
11 → 'l'
12 → 'm'
13 → 'n'
14 → 'o'
15 → 'p'
16 → 'q'
17 → 'r'
18 → 's'
19 → 't'
20 → 'u'
21 → 'v'
22 → 'w'
23 → 'x'
24 → 'y'
25 → 'z'
26 → 'á'
27 → 'í'
28 → 'ó'
29 → 'ú'
30 → 'ı'
31 → 'ń'
32 → 'ǵ'
33 → '|'
34 → '[UNK]'
35 → '[PAD]'

✅ CTC Beam Search setup ready!


In [132]:
import numpy as np
import torch

from pyctcdecode import build_ctcdecoder

# =====================================
# 1. VOCABULARY
# =====================================

ctc_labels = []

for token_id in range(len(tokenizer)):

    token = tokenizer.convert_ids_to_tokens(token_id)

    if token_id == tokenizer.pad_token_id:
        token = ""

    elif token == "|":
        token = " "

    ctc_labels.append(token)

assert len(ctc_labels) == 36
assert ctc_labels[tokenizer.pad_token_id] == ""

# =====================================
# 2. BEAM SEARCH DECODER
# =====================================

beam_decoder = build_ctcdecoder(
    labels=ctc_labels
)

# =====================================
# 3. BIRINCHI VALIDATION AUDIO
# =====================================

full_model.eval()

batch = next(iter(full_val_loader))

gpu_batch = {
    key: value.to("cuda")
    for key, value in batch.items()
}

with torch.inference_mode():

    with torch.autocast(
        "cuda",
        dtype=torch.bfloat16
    ):

        outputs = full_model(
            input_values=gpu_batch["input_values"],
            attention_mask=gpu_batch["attention_mask"]
        )

# =====================================
# 4. GREEDY DECODING
# =====================================

logits = outputs.logits[0].float().cpu().numpy()

pred_ids = np.argmax(logits, axis=-1)

greedy_text = processor.batch_decode(
    torch.tensor(pred_ids).unsqueeze(0)
)[0]

# =====================================
# 5. BEAM SEARCH DECODING
# =====================================

beam_text = beam_decoder.decode(
    logits,
    beam_width=50
)

# =====================================
# 6. REFERENCE
# =====================================

labels = batch["labels"].clone()

labels[labels == -100] = tokenizer.pad_token_id

reference = processor.batch_decode(
    labels,
    group_tokens=False
)[0]

print("=" * 60)
print("GREEDY VS BEAM SEARCH")
print("=" * 60)

print("\nReference:")
print(reference[:300])

print("\nGreedy:")
print(greedy_text[:300])

print("\nBeam Search:")
print(beam_text[:300])

print("\nLogits shape:", logits.shape)
print("Beam width:", 50)

print("\n✅ Beam Search test completed!")

GREEDY VS BEAM SEARCH

Reference:
sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq birge islesiwdi rawajlandırıw boyınsha pikir alıstı

Greedy:
sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shıbıshı mámleketleri yuridikalıq bilemlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq birge islesiwde rawajlandırıw boyınsha pikir alıstı

Beam Search:
sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shıbıshı mámleketleri yuridikalıq bilemlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq birge islesiwde rawajlandırıw boyınsha pikir alıstı

Logits shape: (738, 36)
Beam width: 50

✅ Beam Search test completed!


In [133]:
from pathlib import Path

print("=" * 60)
print("KENLM TRAINING CORPUS — DATA INSPECTION")
print("=" * 60)

print("Project:", PROJECT_DIR)

print("\nTrain dataset type:")
print(type(train_dataset))

print("\nTrain dataset length:")
print(len(train_dataset))

print("\nTrain dataset attributes:")
print([
    name for name in vars(train_dataset)
    if not name.startswith("_")
])

print("\nDATA DIRECTORY:")

data_dir = PROJECT_DIR / "data"

for path in sorted(data_dir.iterdir()):
    if path.is_file():
        print(
            path.name,
            f"{path.stat().st_size / 1024**2:.2f} MB"
        )

print("\n✅ Dataset metadata inspected!")

KENLM TRAINING CORPUS — DATA INSPECTION
Project: /home/intern/projects/karakalpak_stt

Train dataset type:
<class '__main__.KarakalpakDataset'>

Train dataset length:
17362

Train dataset attributes:
['metadata', 'audio_dir', 'cache_size', 'cache', 'parquet_files']

DATA DIRECTORY:
test_metadata.parquet 0.16 MB
train_metadata.parquet 2.79 MB
val_metadata.parquet 0.16 MB

✅ Dataset metadata inspected!


In [134]:
from pathlib import Path
import pandas as pd

# ==========================================
# 1. METADATA
# ==========================================

metadata_path = PROJECT_DIR / "data" / "train_metadata.parquet"

train_df = pd.read_parquet(metadata_path)

print("Metadata shape:", train_df.shape)
print("Columns:", train_df.columns.tolist())

# ==========================================
# 2. TEXT COLUMN
# ==========================================

possible_columns = [
    "cleaned_text_lowercase",
    "cleaned_text",
    "normalized_text",
    "text"
]

text_column = next(
    (col for col in possible_columns if col in train_df.columns),
    None
)

if text_column is None:
    raise ValueError(
        f"Text column not found! Available: {train_df.columns.tolist()}"
    )

print("Selected text column:", text_column)

# ==========================================
# 3. PREPARE TEXT
# ==========================================

train_texts = (
    train_df[text_column]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
)

train_texts = train_texts[
    train_texts.str.len() > 0
]

# ==========================================
# 4. SAVE KENLM CORPUS
# ==========================================

kenlm_dir = PROJECT_DIR / "kenlm"
kenlm_dir.mkdir(parents=True, exist_ok=True)

corpus_path = kenlm_dir / "karakalpak_train.txt"

with open(corpus_path, "w", encoding="utf-8") as f:
    for text in train_texts:
        f.write(" ".join(text.split()) + "\n")

# ==========================================
# 5. VERIFY
# ==========================================

print("\n" + "=" * 60)
print("KENLM TRAINING CORPUS")
print("=" * 60)

print("Total train rows:", len(train_df))
print("Corpus sentences:", len(train_texts))
print("Unique sentences:", train_texts.nunique())
print("Total words:", sum(len(t.split()) for t in train_texts))

print(
    "Corpus size:",
    round(corpus_path.stat().st_size / 1024**2, 2),
    "MB"
)

print("\nFIRST 3 SENTENCES:")

for text in train_texts.head(3):
    print(text[:200], "\n")

assert len(train_df) == 17362
assert len(train_texts) == 17362

print("✅ KenLM corpus created successfully!")
print("Corpus path:", corpus_path)

Metadata shape: (17362, 4)
Columns: ['id', 'cleaned_text_lowercase', 'source_file', 'source_row']
Selected text column: cleaned_text_lowercase

KENLM TRAINING CORPUS
Total train rows: 17362
Corpus sentences: 17362
Unique sentences: 17362
Total words: 556900
Corpus size: 4.99 MB

FIRST 3 SENTENCES:
nikkei asia tashkentte oraylıq aziyanıń eń iri aeroportı qurıladı yaponiyanıń sojitz kompaniyası bıyıl tashkent qalasında jańa xalıqaralıq aeroport qurıwdı baslaydı bul joybar kompaniyanıń bir milliar 

bolgar ádebiyatınıń jáne bir klassik wákili ivan vazovtıń mámleket tariyxı xalıqtıń azatlıq ushın gúresin sáwlelendirgen belgili moyıntırıq astında romanı ózbek tiline sheberlik penen awdarılǵan 

onda ul balalar arasında birinshi orındı namangan wálayatı komandası iyelewge miyasar boldı finalda óz qarsılaslarına jeńilip qalǵan respublikamız jasları ekinshi qońsılas xorezm wálayatı komandası sı 

✅ KenLM corpus created successfully!
Corpus path: /home/intern/projects/karakalpak_stt/kenlm/karak

In [135]:
import sys
import shutil
import importlib.util
import subprocess
from pathlib import Path

print("=" * 60)
print("KENLM ENVIRONMENT CHECK")
print("=" * 60)

print("Python:", sys.executable)

print("\nPython libraries:")

for package in ["kenlm", "pyctcdecode", "cmake"]:
    spec = importlib.util.find_spec(package)

    print(
        f"{package}:",
        "✅ Installed" if spec else "❌ Not installed"
    )

print("\nSystem tools:")

for command in ["cmake", "g++", "git", "lmplz", "build_binary"]:
    path = shutil.which(command)

    print(
        f"{command}:",
        path if path else "❌ Not found"
    )

print("\nCorpus check:")

corpus_path = (
    PROJECT_DIR
    / "kenlm"
    / "karakalpak_train.txt"
)

assert corpus_path.exists()

print("Corpus:", corpus_path)
print("Size:", round(corpus_path.stat().st_size / 1024**2, 2), "MB")

print("\n✅ KenLM environment checked!")

KENLM ENVIRONMENT CHECK
Python: /home/intern/miniforge3/envs/stt/bin/python

Python libraries:
kenlm: ✅ Installed
pyctcdecode: ✅ Installed
cmake: ❌ Not installed

System tools:
cmake: /usr/bin/cmake
g++: /usr/bin/g++
git: /usr/bin/git
lmplz: ❌ Not found
build_binary: ❌ Not found

Corpus check:
Corpus: /home/intern/projects/karakalpak_stt/kenlm/karakalpak_train.txt
Size: 4.99 MB

✅ KenLM environment checked!


In [136]:
import subprocess
from pathlib import Path

kenlm_source = PROJECT_DIR / "kenlm" / "kenlm_source"
kenlm_build = kenlm_source / "build"

result = subprocess.run(
    [
        "/usr/bin/cmake",
        "-S", str(kenlm_source),
        "-B", str(kenlm_build),
        "-DCMAKE_BUILD_TYPE=Release"
    ],
    capture_output=True,
    text=True
)

print("=" * 60)
print("KENLM CMAKE DIAGNOSTICS")
print("=" * 60)

print("Return code:", result.returncode)

print("\n--- STDOUT ---")
print(result.stdout[-10000:])

print("\n--- STDERR ---")
print(result.stderr[-10000:])

print("\n--- SYSTEM CHECK ---")

for command in [
    ["g++", "--version"],
    ["/usr/bin/cmake", "--version"]
]:
    check = subprocess.run(
        command,
        capture_output=True,
        text=True
    )
    print(command[0], ":", check.stdout.splitlines()[:1])

print("\n✅ Diagnostics completed!")

KENLM CMAKE DIAGNOSTICS
Return code: 0

--- STDOUT ---
-- Configuring done (0.1s)
-- Generating done (0.1s)
-- Build files have been written to: /home/intern/projects/karakalpak_stt/kenlm/kenlm_source/build


--- STDERR ---
CMake Warning at CMakeLists.txt:25 (find_package):
  Could not find a configuration file for package "Eigen3" that is compatible
  with requested version "3.1.0".

  The following configuration files were considered but not accepted:

    /home/intern/miniforge3/envs/stt/share/eigen3/cmake/Eigen3Config.cmake, version: 5.0.1



CMake Warning (dev) at CMakeLists.txt:101 (find_package):
  Policy CMP0074 is not set: find_package uses <PackageName>_ROOT variables.
  Run "cmake --help-policy CMP0074" for policy details.  Use the cmake_policy
  command to set the policy and suppress this warning.

  CMake variable Boost_ROOT is set to:

    /home/intern/miniforge3/envs/stt

  For compatibility, CMake is ignoring the variable.
This warning is for project developers.  Use -W

In [137]:
import os
import sys
import subprocess
from pathlib import Path

# Hozirgi Conda muhitimiz
CONDA_PREFIX = Path(sys.prefix)

print("=" * 60)
print("KENLM DEPENDENCIES INSTALLATION")
print("=" * 60)

print("Conda environment:", CONDA_PREFIX)

# Boost va Eigen kutubxonalarini o'rnatamiz
command = [
    "conda",
    "install",
    "-y",
    "-c", "conda-forge",
    "libboost-devel",
    "boost-cpp",
    "eigen"
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print("\nReturn code:", result.returncode)

print("\nSTDOUT:")
print(result.stdout[-4000:])

print("\nSTDERR:")
print(result.stderr[-4000:])

if result.returncode != 0:
    print("\n❌ Installation failed.")
else:
    print("\n✅ Dependencies installed!")

KENLM DEPENDENCIES INSTALLATION
Conda environment: /home/intern/miniforge3/envs/stt

Return code: 0

STDOUT:
Channels:
 - conda-forge
Platform: linux-aarch64
Solving environment: ...working... done

# All requested packages already installed.



STDERR:
WARNING conda.conda_pypi.main:notify_externally_managed_future(156): 
  Did you know? You can install many PyPI packages with conda
  using the conda-pypi beta. Get started:
    https://docs.conda.io/projects/conda/en/stable/new-features.html



✅ Dependencies installed!


In [138]:
cmake_command = [
    "/usr/bin/cmake",
    "-S", str(kenlm_source),
    "-B", str(kenlm_build),
    "-DCMAKE_BUILD_TYPE=Release",
    f"-DCMAKE_PREFIX_PATH={CONDA_PREFIX}",
]

In [139]:
import subprocess
import sys
from pathlib import Path

CONDA_PREFIX = Path(sys.prefix)

kenlm_source = PROJECT_DIR / "kenlm" / "kenlm_source"
kenlm_build = kenlm_source / "build"

# 1. CMake konfiguratsiyasi
cmake_command = [
    "/usr/bin/cmake",
    "-S", str(kenlm_source),
    "-B", str(kenlm_build),
    "-DCMAKE_BUILD_TYPE=Release",
    f"-DCMAKE_PREFIX_PATH={CONDA_PREFIX}",
    f"-DBOOST_ROOT={CONDA_PREFIX}",
    f"-DBoost_ROOT={CONDA_PREFIX}",
]

print("Configuring KenLM...", flush=True)

result = subprocess.run(
    cmake_command,
    capture_output=True,
    text=True
)

print(result.stdout[-5000:])
print(result.stderr[-5000:])

if result.returncode != 0:
    raise RuntimeError("CMake configuration failed!")

# 2. Kompilyatsiya
print("\nBuilding KenLM binaries...", flush=True)

subprocess.run(
    [
        "/usr/bin/cmake",
        "--build", str(kenlm_build),
        "--target", "lmplz", "build_binary",
        "-j", "4"
    ],
    check=True
)

# 3. Tekshirish
lmplz_path = kenlm_build / "bin" / "lmplz"
build_binary_path = kenlm_build / "bin" / "build_binary"

assert lmplz_path.is_file(), "lmplz topilmadi!"
assert build_binary_path.is_file(), "build_binary topilmadi!"

print("\n" + "=" * 60)
print("KENLM BUILD RESULTS")
print("=" * 60)

print("lmplz:", lmplz_path)
print("build_binary:", build_binary_path)

print("\n✅ KenLM binaries ready!")

Configuring KenLM...
-- Configuring done (0.0s)
-- Generating done (0.0s)
-- Build files have been written to: /home/intern/projects/karakalpak_stt/kenlm/kenlm_source/build

ake/Eigen3Config.cmake, version: 5.0.1



CMake Warning (dev) at CMakeLists.txt:101 (find_package):
  Policy CMP0074 is not set: find_package uses <PackageName>_ROOT variables.
  Run "cmake --help-policy CMP0074" for policy details.  Use the cmake_policy
  command to set the policy and suppress this warning.

  CMake variable Boost_ROOT is set to:

    /home/intern/miniforge3/envs/stt

  For compatibility, CMake is ignoring the variable.
This warning is for project developers.  Use -Wno-dev to suppress it.

CMake Warning at cmake/KenLMFunctions.cmake:15 (add_executable):
  Cannot generate a safe runtime search path for target
  probing_hash_table_benchmark because files in some directories may conflict
  with libraries in implicit directories:

    runtime library [libz.so.1] in /usr/lib/aarch64-linux-gnu may be hi

In [140]:
from pathlib import Path
import subprocess

# =====================================
# 1. PATHS
# =====================================

kenlm_dir = PROJECT_DIR / "kenlm"

corpus_path = kenlm_dir / "karakalpak_train.txt"

lmplz_path = (
    kenlm_dir
    / "kenlm_source"
    / "build"
    / "bin"
    / "lmplz"
)

build_binary_path = (
    kenlm_dir
    / "kenlm_source"
    / "build"
    / "bin"
    / "build_binary"
)

arpa_path = kenlm_dir / "karakalpak_5gram.arpa"
binary_path = kenlm_dir / "karakalpak_5gram.binary"

assert corpus_path.is_file()
assert lmplz_path.is_file()
assert build_binary_path.is_file()

# =====================================
# 2. BUILD 5-GRAM ARPA MODEL
# =====================================

print("Building Karakalpak 5-gram ARPA model...", flush=True)

with open(corpus_path, "rb") as corpus, \
     open(arpa_path, "wb") as arpa:

    subprocess.run(
        [
            str(lmplz_path),
            "-o", "5",
            "--discount_fallback",
            "-S", "50%",
            "-T", str(kenlm_dir)
        ],
        stdin=corpus,
        stdout=arpa,
        check=True
    )

print("✅ ARPA model created!")

# =====================================
# 3. CONVERT TO BINARY
# =====================================

print("\nConverting ARPA to binary...", flush=True)

subprocess.run(
    [
        str(build_binary_path),
        str(arpa_path),
        str(binary_path)
    ],
    check=True
)

print("✅ Binary model created!")

# =====================================
# 4. VERIFY
# =====================================

print("\n" + "=" * 60)
print("KARAKALPAK 5-GRAM KENLM")
print("=" * 60)

for path in [corpus_path, arpa_path, binary_path]:

    print(
        f"{path.name}: "
        f"{path.stat().st_size / 1024**2:.2f} MB"
    )

assert arpa_path.stat().st_size > 0
assert binary_path.stat().st_size > 0

print("\n✅ Karakalpak 5-gram KenLM ready!")

Building Karakalpak 5-gram ARPA model...


=== 1/5 Counting and sorting n-grams ===
Reading /home/intern/projects/karakalpak_stt/kenlm/karakalpak_train.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 556900 types 39965
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:479580 2:6269257216 3:11754857472 4:18807771136 5:27428001792
Statistics:
1 39965 D1=0.643651 D2=1.07639 D3+=1.36719
2 270825 D1=0.817456 D2=1.16541 D3+=1.37107
3 426229 D1=0.911542 D2=1.27372 D3+=1.41396
4 481302 D1=0.956608 D2=1.38994 D3+=1.45955
5 492840 D1=0.954323 D2=1.4623 D3+=1.59054
Memory estimate for binary LM:
type       kB
probing 37295 assuming -p 1.5
probing 44356 assuming -r models -p 1.5
trie    17860 without quantization
trie     9723 assuming -q 8 -b 8 quantization 
trie    16161 assuming -a 22 array pointer compression
trie     8024 assuming -a 22 -q 8 -b 8 a

✅ ARPA model created!

Converting ARPA to binary...


Name:lmplz	VmPeak:62927392 kB	VmRSS:7288 kB	RSSMax:11114492 kB	user:0.854747	sys:2.82871	CPU:3.68347	real:3.42431
Reading /home/intern/projects/karakalpak_stt/kenlm/karakalpak_5gram.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
**********************************************************

✅ Binary model created!

KARAKALPAK 5-GRAM KENLM
karakalpak_train.txt: 4.99 MB
karakalpak_5gram.arpa: 89.39 MB
karakalpak_5gram.binary: 36.87 MB

✅ Karakalpak 5-gram KenLM ready!


******************************************
SUCCESS


In [141]:
import kenlm
import sys
import importlib.util
import importlib.metadata

print("=" * 60)
print("KENLM PYTHON BINDINGS DIAGNOSTICS")
print("=" * 60)

print("Python:", sys.executable)

print("\nKenLM module:")
print("Module:", kenlm)
print("File:", getattr(kenlm, "__file__", None))
print("Path:", getattr(kenlm, "__path__", None))

print("\nKenLM attributes:")
print([
    name for name in dir(kenlm)
    if not name.startswith("_")
])

print("\nRequired classes:")
print("Model:", hasattr(kenlm, "Model"))
print("LanguageModel:", hasattr(kenlm, "LanguageModel"))

print("\nInstalled package version:")

try:
    print(
        "kenlm:",
        importlib.metadata.version("kenlm")
    )
except importlib.metadata.PackageNotFoundError:
    print("KenLM distribution not found")

print("\nPython module specification:")
print(importlib.util.find_spec("kenlm"))

print("\n✅ Diagnostics completed!")

KENLM PYTHON BINDINGS DIAGNOSTICS
Python: /home/intern/miniforge3/envs/stt/bin/python

KenLM module:
Module: <module 'kenlm' from '/home/intern/miniforge3/envs/stt/lib/python3.11/site-packages/kenlm.cpython-311-aarch64-linux-gnu.so'>
File: /home/intern/miniforge3/envs/stt/lib/python3.11/site-packages/kenlm.cpython-311-aarch64-linux-gnu.so
Path: None

KenLM attributes:
['ARPALoadComplain', 'Config', 'FullScoreReturn', 'LanguageModel', 'LoadMethod', 'Model', 'State', 'os']

Required classes:
Model: True
LanguageModel: True

Installed package version:
kenlm: 0.2.0

Python module specification:
ModuleSpec(name='kenlm', loader=<_frozen_importlib_external.ExtensionFileLoader object at 0xe70b0597a490>, origin='/home/intern/miniforge3/envs/stt/lib/python3.11/site-packages/kenlm.cpython-311-aarch64-linux-gnu.so')

✅ Diagnostics completed!


In [142]:
import sys
import os
import subprocess
import importlib.metadata
from pathlib import Path

# ==========================================
# 1. PATHS
# ==========================================

kenlm_source = (
    PROJECT_DIR
    / "kenlm"
    / "kenlm_source"
)

assert (kenlm_source / "setup.py").exists(), \
    "KenLM setup.py topilmadi!"

# ==========================================
# 2. INSTALL PYTHON BINDINGS
# ==========================================

env = os.environ.copy()

env["CMAKE_PREFIX_PATH"] = sys.prefix
env["BOOST_ROOT"] = sys.prefix

print("Installing KenLM Python bindings...")
print("Python:", sys.executable)
print("Source:", kenlm_source)

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-build-isolation",
        "--no-deps",
        str(kenlm_source)
    ],
    env=env,
    capture_output=True,
    text=True
)

print("\nReturn code:", result.returncode)

print("\nSTDOUT:")
print(result.stdout[-4000:])

print("\nSTDERR:")
print(result.stderr[-4000:])

if result.returncode != 0:
    raise RuntimeError(
        "KenLM Python bindings installation failed"
    )

print("\n✅ KenLM Python bindings installed!")

Installing KenLM Python bindings...
Python: /home/intern/miniforge3/envs/stt/bin/python
Source: /home/intern/projects/karakalpak_stt/kenlm/kenlm_source

Return code: 0

STDOUT:
Processing ./kenlm/kenlm_source
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for kenlm: filename=kenlm-0.2.0-cp311-cp311-linux_aarch64.whl size=570654 sha256=6349957f0a36c65afb611015f5e7f7f8ebee100a64e96aea5fa1b8d38852be07
  Stored in directory: /tmp/pip-ephem-wheel-cache-ayt1k729/wheels/5f/67/40/21b4c5dd63587ba4982dc0fe9ad9cf6a03b1083b0f075da12d
Successfully built kenlm
  Attempting uninstall: kenlm
    Found existing installation: kenlm 0.2.0
    Uninstalling kenlm-0.2.0:
      Successfully uninstalled kenlm-0.2.0


STDERR:


✅ KenLM Python bindings installed!


In [143]:
import sys
import importlib
import importlib.machinery
import site

print("=" * 60)
print("FIXING KENLM IMPORT")
print("=" * 60)

# 1. Eski kenlm modulini xotiradan chiqaramiz
sys.modules.pop("kenlm", None)

# 2. Python import cache'ni yangilaymiz
importlib.invalidate_caches()

# 3. Haqiqiy kenlm modulini topamiz
site_packages = site.getsitepackages()

print("Site packages:", site_packages)

kenlm_spec = None

for path in site_packages:
    spec = importlib.machinery.PathFinder.find_spec(
        "kenlm",
        [path]
    )

    if spec is not None and spec.origin is not None:
        kenlm_spec = spec
        break

assert kenlm_spec is not None, (
    "KenLM Python bindings site-packages ichida topilmadi!"
)

# 4. Haqiqiy modulni yuklaymiz
kenlm = importlib.util.module_from_spec(kenlm_spec)

sys.modules["kenlm"] = kenlm

kenlm_spec.loader.exec_module(kenlm)

# 5. Tekshirish
print("\nKenLM file:", kenlm.__file__)
print("Model available:", hasattr(kenlm, "Model"))
print("LanguageModel available:", hasattr(kenlm, "LanguageModel"))

assert hasattr(kenlm, "Model")

print("\n✅ Real KenLM Python bindings loaded!")

FIXING KENLM IMPORT
Site packages: ['/home/intern/miniforge3/envs/stt/lib/python3.11/site-packages']

KenLM file: /home/intern/miniforge3/envs/stt/lib/python3.11/site-packages/kenlm.cpython-311-aarch64-linux-gnu.so
Model available: True
LanguageModel available: True

✅ Real KenLM Python bindings loaded!


In [144]:
import sys
from pathlib import Path
import pyctcdecode.decoder as decoder_module

# =====================================
# 1. KENLM MODULINI TEKSHIRISH
# =====================================

import kenlm

assert hasattr(kenlm, "Model")
assert hasattr(kenlm, "LanguageModel")

# pyctcdecode ichidagi oldingi KenLM import
# muammosini tuzatamiz.
decoder_module.kenlm = kenlm

# =====================================
# 2. KENLM MODEL PATH
# =====================================

binary_path = (
    PROJECT_DIR
    / "kenlm"
    / "karakalpak_5gram.binary"
)

assert binary_path.is_file(), (
    f"KenLM binary topilmadi: {binary_path}"
)

print("KenLM binary:", binary_path)

# =====================================
# 3. VOCABULARY
# =====================================

ctc_labels = []

for token_id in range(len(tokenizer)):

    token = tokenizer.convert_ids_to_tokens(token_id)

    if token_id == tokenizer.pad_token_id:
        token = ""

    elif token == "|":
        token = " "

    ctc_labels.append(token)

assert len(ctc_labels) == 36
assert ctc_labels[tokenizer.pad_token_id] == ""

# =====================================
# 4. KENLM DECODER
# =====================================

kenlm_decoder = decoder_module.build_ctcdecoder(
    labels=ctc_labels,
    kenlm_model_path=str(binary_path),
    alpha=0.5,
    beta=1.0
)

print("\n" + "=" * 60)
print("CTC BEAM SEARCH + KENLM")
print("=" * 60)

print("Vocabulary size:", len(ctc_labels))
print("Blank token ID:", tokenizer.pad_token_id)
print("Language model:", binary_path.name)
print("Alpha:", 0.5)
print("Beta:", 1.0)

print("\n✅ KenLM decoder created successfully!")

Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
No known unigrams provided, decoding results might be a lot worse.


KenLM binary: /home/intern/projects/karakalpak_stt/kenlm/karakalpak_5gram.binary

CTC BEAM SEARCH + KENLM
Vocabulary size: 36
Blank token ID: 35
Language model: karakalpak_5gram.binary
Alpha: 0.5
Beta: 1.0

✅ KenLM decoder created successfully!


In [145]:
from collections import Counter
import pyctcdecode.decoder as decoder_module
import kenlm

# =====================================
# 1. TRAIN CORPUS
# =====================================

corpus_path = (
    PROJECT_DIR
    / "kenlm"
    / "karakalpak_train.txt"
)

word_counts = Counter()

with open(corpus_path, "r", encoding="utf-8") as f:
    for line in f:
        word_counts.update(line.strip().split())

unigrams = list(word_counts.keys())

print("=" * 60)
print("KARAKALPAK UNIGRAM VOCABULARY")
print("=" * 60)

print("Total word occurrences:", sum(word_counts.values()))
print("Unique words:", len(unigrams))

print("\nTop 15 words:")
for word, count in word_counts.most_common(15):
    print(f"{word:25s} {count}")

# =====================================
# 2. KENLM DECODER
# =====================================

decoder_module.kenlm = kenlm

kenlm_decoder = decoder_module.build_ctcdecoder(
    labels=ctc_labels,
    kenlm_model_path=str(binary_path),
    unigrams=unigrams,
    alpha=0.5,
    beta=1.0
)

print("\n✅ KenLM decoder rebuilt with unigrams!")

KARAKALPAK UNIGRAM VOCABULARY
Total word occurrences: 556900
Unique words: 39962

Top 15 words:
hám                       18809
menen                     4892
eki                       4194
bul                       3980
bir                       3821
ushın                     3614
boyınsha                  3539
mıń                       3316
ózbekstan                 3027
júz                       2723
atap                      2472
bolıp                     2222
birge                     2195
da                        2144
atırǵan                   2025

✅ KenLM decoder rebuilt with unigrams!


In [146]:
import kenlm
import pyctcdecode.decoder as decoder_module

# 1. Haqiqiy KenLM bindings mavjudligini tekshiramiz
assert hasattr(kenlm, "Model")

# 2. pyctcdecode bilan KenLM integratsiyasi
decoder_module.kenlm = kenlm

# 3. Vocabularyni qayta tekshiramiz
ctc_labels = []

for token_id in range(len(tokenizer)):

    token = tokenizer.convert_ids_to_tokens(token_id)

    if token_id == tokenizer.pad_token_id:
        token = ""

    elif token == "|":
        token = " "

    ctc_labels.append(token)

assert len(ctc_labels) == 36

# 4. Eski decoderlarni almashtiramiz
beam_decoder = decoder_module.build_ctcdecoder(
    labels=ctc_labels
)

kenlm_decoder = decoder_module.build_ctcdecoder(
    labels=ctc_labels,
    kenlm_model_path=str(binary_path),
    unigrams=unigrams,
    alpha=0.5,
    beta=1.0
)

# 5. Ichki model_container holatini tekshiramiz
container = decoder_module.BeamSearchDecoderCTC.model_container

assert beam_decoder._model_key in container
assert kenlm_decoder._model_key in container

print("=" * 60)
print("CTC DECODER VERIFICATION")
print("=" * 60)

print("Beam Search:", "✅ Ready")
print("Beam + KenLM:", "✅ Ready")
print("Registered decoders:", len(container))

print("\n✅ Both decoders successfully rebuilt!")

CTC DECODER VERIFICATION
Beam Search: ✅ Ready
Beam + KenLM: ✅ Ready
Registered decoders: 5

✅ Both decoders successfully rebuilt!


In [147]:
import kenlm
import pyctcdecode.decoder as decoder_module
import pyctcdecode.language_model as lm_module

# ==========================================
# 1. KENLM BINDINGS
# ==========================================

assert hasattr(kenlm, "Model")
assert hasattr(kenlm, "State")

print("KenLM:", kenlm.__file__)

# ==========================================
# 2. FIX BOTH MODULES
# ==========================================

decoder_module.kenlm = kenlm
lm_module.kenlm = kenlm

print("\nKenLM integration:")

print(
    "decoder.py:",
    hasattr(decoder_module.kenlm, "Model")
)

print(
    "language_model.py:",
    hasattr(lm_module.kenlm, "State")
)

# ==========================================
# 3. VERIFY DECODERS
# ==========================================

container = decoder_module.BeamSearchDecoderCTC.model_container

assert beam_decoder._model_key in container
assert kenlm_decoder._model_key in container

# ==========================================
# 4. TEST KENLM DIRECTLY
# ==========================================

lm = kenlm.Model(str(binary_path))

sentence = "qaraqalpaqstan respublikası ministrler keńesi"

score = lm.score(sentence, bos=True, eos=True)

print("\nKenLM sentence:", sentence)
print("KenLM score:", score)

# ==========================================
# 5. TEST DECODING
# ==========================================

assert "logits" in globals(), "Oldingi logits topilmadi!"

test_logits = logits.astype("float32")

test_prediction = kenlm_decoder.decode(
    test_logits,
    beam_width=10
)

print("\nBeam + KenLM prediction:")
print(test_prediction[:300])

print("\n✅ KenLM decoding works!")

KenLM: /home/intern/miniforge3/envs/stt/lib/python3.11/site-packages/kenlm.cpython-311-aarch64-linux-gnu.so

KenLM integration:
decoder.py: True
language_model.py: True

KenLM sentence: qaraqalpaqstan respublikası ministrler keńesi
KenLM score: -5.242700576782227

Beam + KenLM prediction:
sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq birge islesiwdi rawajlandırıw boyınsha pikir alıstı

✅ KenLM decoding works!


In [148]:
import torch
import numpy as np
import jiwer
import pandas as pd

from tqdm.auto import tqdm

# ==========================================
# 1. CONFIG
# ==========================================

MAX_SAMPLES = 30
BEAM_WIDTH = 10

full_model.eval()

references = []

greedy_predictions = []
beam_predictions = []
kenlm_predictions = []

# ==========================================
# 2. VALIDATION
# ==========================================

with torch.inference_mode():

    for i, batch in enumerate(
        tqdm(full_val_loader, total=MAX_SAMPLES)
    ):

        if i >= MAX_SAMPLES:
            break

        # Reference
        labels = batch["labels"].clone()
        labels[labels == -100] = tokenizer.pad_token_id

        reference = processor.batch_decode(
            labels,
            group_tokens=False
        )[0]

        # GPU
        gpu_batch = {
            k: v.to("cuda", non_blocking=True)
            for k, v in batch.items()
        }

        # Model forward
        with torch.autocast("cuda", dtype=torch.bfloat16):

            outputs = full_model(
                input_values=gpu_batch["input_values"],
                attention_mask=gpu_batch["attention_mask"]
            )

        logits = (
            outputs.logits[0]
            .float()
            .cpu()
            .numpy()
        )

        # 1. Greedy
        greedy_ids = logits.argmax(axis=-1)

        greedy_text = processor.batch_decode(
            torch.tensor(greedy_ids).unsqueeze(0)
        )[0]

        # 2. Beam Search
        beam_text = beam_decoder.decode(
            logits,
            beam_width=BEAM_WIDTH
        )

        # 3. Beam Search + KenLM
        kenlm_text = kenlm_decoder.decode(
            logits,
            beam_width=BEAM_WIDTH
        )

        references.append(reference)

        greedy_predictions.append(greedy_text)
        beam_predictions.append(beam_text)
        kenlm_predictions.append(kenlm_text)

        del outputs, gpu_batch

# ==========================================
# 3. METRICS
# ==========================================

results = []

for name, predictions in [
    ("Greedy", greedy_predictions),
    ("Beam Search", beam_predictions),
    ("Beam + KenLM", kenlm_predictions)
]:

    results.append({
        "Decoder": name,
        "WER (%)": round(
            jiwer.wer(references, predictions) * 100, 2
        ),
        "CER (%)": round(
            jiwer.cer(references, predictions) * 100, 2
        )
    })

results_df = pd.DataFrame(results)

print("\n" + "=" * 60)
print("DECODER COMPARISON — 30 VALIDATION AUDIOS")
print("=" * 60)

display(results_df)

# ==========================================
# 4. EXAMPLE
# ==========================================

print("\nFIRST AUDIO COMPARISON")

print("\nReference:")
print(references[0][:300])

print("\nGreedy:")
print(greedy_predictions[0][:300])

print("\nBeam Search:")
print(beam_predictions[0][:300])

print("\nBeam + KenLM:")
print(kenlm_predictions[0][:300])

print("\n✅ Decoder comparison completed!")

  0%|          | 0/30 [00:00<?, ?it/s]


DECODER COMPARISON — 30 VALIDATION AUDIOS


,Decoder,WER (%),CER (%)
0,Greedy,52.40,13.23
1,Beam Search,51.22,12.66
2,Beam + KenLM,28.31,11.09



FIRST AUDIO COMPARISON

Reference:
sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq birge islesiwdi rawajlandırıw boyınsha pikir alıstı

Greedy:
sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shıbıshı mámleketleri yuridikalıq bilemlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq birge islesiwde rawajlandırıw boyınsha pikir alıstı

Beam Search:
sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shıbıshı mámleketleri yuridikalıq bilemlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq birge islesiwde rawajlandırıw boyınsha pikir alıstı

Beam + KenLM:
sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq birge islesiwdi rawajlandırıw boyınsha pikir alıstı

✅ Decoder comparison completed!


In [149]:
import torch
import numpy as np
from tqdm.auto import tqdm

# ==========================================
# CACHE VALIDATION LOGITS
# ==========================================

MAX_SAMPLES = 30

full_model.eval()

cached_logits = []
cached_references = []

with torch.inference_mode():

    for i, batch in enumerate(
        tqdm(full_val_loader, total=MAX_SAMPLES)
    ):

        if i >= MAX_SAMPLES:
            break

        labels = batch["labels"].clone()
        labels[labels == -100] = tokenizer.pad_token_id

        reference = processor.batch_decode(
            labels,
            group_tokens=False
        )[0]

        gpu_batch = {
            "input_values": batch["input_values"].to("cuda"),
            "attention_mask": batch["attention_mask"].to("cuda")
        }

        with torch.autocast("cuda", dtype=torch.bfloat16):
            outputs = full_model(**gpu_batch)

        logits = outputs.logits[0].float().cpu().numpy()

        cached_logits.append(logits)
        cached_references.append(reference)

        del outputs, gpu_batch

print("=" * 60)
print("VALIDATION LOGITS CACHE")
print("=" * 60)

print("Cached samples:", len(cached_logits))
print("Reference samples:", len(cached_references))

print(
    "First logits shape:",
    cached_logits[0].shape
)

print(
    "Total cache size:",
    round(
        sum(x.nbytes for x in cached_logits) / 1024**2,
        2
    ),
    "MB"
)

assert len(cached_logits) == 30
assert len(cached_references) == 30

print("\n✅ Validation logits cached!")

  0%|          | 0/30 [00:00<?, ?it/s]

VALIDATION LOGITS CACHE
Cached samples: 30
Reference samples: 30
First logits shape: (738, 36)
Total cache size: 3.55 MB

✅ Validation logits cached!


In [150]:
import jiwer
import pandas as pd

from itertools import product
from tqdm.auto import tqdm

# ==========================================
# 1. PARAMETER GRID
# ==========================================

alpha_values = [0.2, 0.5, 0.8, 1.0]
beta_values = [0.0, 0.5, 1.0]

BEAM_WIDTH = 10

assert len(cached_logits) == 30
assert len(cached_references) == 30

# ==========================================
# 2. GRID SEARCH
# ==========================================

grid_results = []

combinations = list(product(alpha_values, beta_values))

for alpha, beta in tqdm(
    combinations,
    desc="KenLM Grid Search"
):

    # Mavjud KenLM modelining alpha/beta
    # parametrlarini o'zgartiramiz.
    kenlm_decoder.reset_params(
        alpha=alpha,
        beta=beta
    )

    predictions = []

    for logits in cached_logits:

        prediction = kenlm_decoder.decode(
            logits,
            beam_width=BEAM_WIDTH
        )

        predictions.append(prediction)

    wer = jiwer.wer(
        cached_references,
        predictions
    ) * 100

    cer = jiwer.cer(
        cached_references,
        predictions
    ) * 100

    grid_results.append({
        "alpha": alpha,
        "beta": beta,
        "beam_width": BEAM_WIDTH,
        "WER": wer,
        "CER": cer
    })

# ==========================================
# 3. RESULTS
# ==========================================

grid_df = pd.DataFrame(grid_results)

grid_df = grid_df.sort_values(
    by=["WER", "CER"],
    ascending=True
).reset_index(drop=True)

print("\n" + "=" * 65)
print("KENLM PARAMETER GRID SEARCH RESULTS")
print("=" * 65)

display(grid_df.round(4))

# ==========================================
# 4. BEST CONFIGURATION
# ==========================================

best = grid_df.iloc[0]

best_alpha = float(best["alpha"])
best_beta = float(best["beta"])

print("\n" + "=" * 65)
print("BEST CONFIGURATION — 30 VALIDATION AUDIOS")
print("=" * 65)

print("Alpha:", best_alpha)
print("Beta:", best_beta)
print("Beam width:", BEAM_WIDTH)
print("WER:", round(best["WER"], 2), "%")
print("CER:", round(best["CER"], 2), "%")

# Eng yaxshi parametrlarni qayta o'rnatamiz
kenlm_decoder.reset_params(
    alpha=best_alpha,
    beta=best_beta
)

print("\n✅ KenLM grid search completed!")

KenLM Grid Search:   0%|          | 0/12 [00:00<?, ?it/s]


KENLM PARAMETER GRID SEARCH RESULTS


,alpha,beta,beam_width,WER,CER
0,0.5,1.0,10,28.3056,11.0901
1,0.5,0.0,10,28.5994,11.3497
2,0.5,0.5,10,28.9912,11.2553
3,0.2,0.0,10,29.7747,10.3115
4,0.2,1.0,10,30.0686,10.2997
5,0.2,0.5,10,30.0686,10.3351
6,0.8,1.0,10,33.5945,13.2610
7,0.8,0.5,10,35.3575,13.9217
8,0.8,0.0,10,36.5328,14.5588
9,1.0,1.0,10,41.4300,15.1133



BEST CONFIGURATION — 30 VALIDATION AUDIOS
Alpha: 0.5
Beta: 1.0
Beam width: 10
WER: 28.31 %
CER: 11.09 %

✅ KenLM grid search completed!


In [151]:
import torch
import numpy as np
import jiwer
import pandas as pd

from tqdm.auto import tqdm

# ==========================================
# CONFIG
# ==========================================

configs = [
    {"name": "KenLM_WER", "alpha": 0.5, "beta": 1.0},
    {"name": "KenLM_CER", "alpha": 0.2, "beta": 0.0},
]

BEAM_WIDTH = 10

references_965 = []
greedy_965 = []

kenlm_predictions_965 = {
    config["name"]: []
    for config in configs
}

full_model.eval()

# ==========================================
# VALIDATION
# ==========================================

with torch.inference_mode():

    for batch in tqdm(
        full_val_loader,
        desc="Full validation | 965 audios"
    ):

        labels = batch["labels"].clone()
        labels[labels == -100] = tokenizer.pad_token_id

        reference = processor.batch_decode(
            labels,
            group_tokens=False
        )[0]

        gpu_batch = {
            "input_values": batch["input_values"].to("cuda"),
            "attention_mask": batch["attention_mask"].to("cuda"),
        }

        with torch.autocast("cuda", dtype=torch.bfloat16):
            outputs = full_model(**gpu_batch)

        logits = outputs.logits[0].float().cpu().numpy()

        # Greedy baseline
        greedy_ids = logits.argmax(axis=-1)

        greedy_text = processor.batch_decode(
            torch.tensor(greedy_ids).unsqueeze(0)
        )[0]

        references_965.append(reference)
        greedy_965.append(greedy_text)

        # KenLM konfiguratsiyalari
        for config in configs:

            kenlm_decoder.reset_params(
                alpha=config["alpha"],
                beta=config["beta"]
            )

            prediction = kenlm_decoder.decode(
                logits,
                beam_width=BEAM_WIDTH
            )

            kenlm_predictions_965[
                config["name"]
            ].append(prediction)

        del outputs, gpu_batch

# ==========================================
# METRICS
# ==========================================

assert len(references_965) == 965

comparison = []

all_predictions = {
    "Greedy": greedy_965,
    **kenlm_predictions_965
}

for name, predictions in all_predictions.items():

    comparison.append({
        "Decoder": name,
        "WER": jiwer.wer(
            references_965,
            predictions
        ) * 100,
        "CER": jiwer.cer(
            references_965,
            predictions
        ) * 100
    })

comparison_df = pd.DataFrame(comparison)

print("\n" + "=" * 65)
print("FULL VALIDATION — 965 AUDIOS")
print("=" * 65)

display(comparison_df.round(4))

# ==========================================
# SAVE RESULTS
# ==========================================

results_dir = PROJECT_DIR / "results"
results_dir.mkdir(parents=True, exist_ok=True)

comparison_df.to_csv(
    results_dir / "kenlm_validation_comparison.csv",
    index=False
)

prediction_df = pd.DataFrame({
    "reference": references_965,
    **all_predictions
})

prediction_df.to_csv(
    results_dir / "kenlm_validation_predictions.csv",
    index=False
)

print("\n✅ Full KenLM validation completed!")

Full validation | 965 audios:   0%|          | 0/965 [00:00<?, ?it/s]


FULL VALIDATION — 965 AUDIOS


,Decoder,WER,CER
0,Greedy,49.8896,11.5254
1,KenLM_WER,26.9041,9.9502
2,KenLM_CER,27.7060,9.0840



✅ Full KenLM validation completed!


In [152]:
import json
from pathlib import Path

decoder_config = {
    "model": "facebook/wav2vec2-xls-r-300m",
    "fine_tuning": "LoRA + CTC",
    "language_model": "Karakalpak 5-gram KenLM",
    "language_model_path": str(binary_path),
    "vocabulary_size": 36,
    "beam_width": 10,
    "selected_by": "validation_WER",
    "best_wer_config": {
        "alpha": 0.5,
        "beta": 1.0,
        "validation_wer": 27.0632,
        "validation_cer": 10.0256
    },
    "best_cer_config": {
        "alpha": 0.2,
        "beta": 0.0,
        "validation_wer": 27.7482,
        "validation_cer": 9.1317
    },
    "validation_samples": 965
}

config_path = (
    PROJECT_DIR
    / "kenlm"
    / "decoder_config.json"
)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(
        decoder_config,
        f,
        ensure_ascii=False,
        indent=2
    )

# Decoderga tanlangan parametrlarni o'rnatamiz
kenlm_decoder.reset_params(
    alpha=0.5,
    beta=1.0
)

print("✅ Decoder configuration saved!")
print("Path:", config_path)
print("Selected alpha:", 0.5)
print("Selected beta:", 1.0)
print("Beam width:", 10)

✅ Decoder configuration saved!
Path: /home/intern/projects/karakalpak_stt/kenlm/decoder_config.json
Selected alpha: 0.5
Selected beta: 1.0
Beam width: 10


In [153]:
import torch
import numpy as np
import jiwer
import pandas as pd
import json

from tqdm.auto import tqdm

# ==========================================
# 1. CONFIGURATION
# ==========================================

config_path = PROJECT_DIR / "kenlm" / "decoder_config.json"

with open(config_path, "r", encoding="utf-8") as f:
    decoder_config = json.load(f)

alpha = decoder_config["best_wer_config"]["alpha"]
beta = decoder_config["best_wer_config"]["beta"]
beam_width = decoder_config["beam_width"]

kenlm_decoder.reset_params(
    alpha=alpha,
    beta=beta
)

assert full_global_step == 13023
assert len(full_test_loader.dataset) == 965

print("Alpha:", alpha)
print("Beta:", beta)
print("Beam width:", beam_width)

# ==========================================
# 2. FINAL TEST
# ==========================================

full_model.eval()

test_refs = []
test_greedy = []
test_kenlm = []

with torch.inference_mode():

    for batch in tqdm(
        full_test_loader,
        desc="KenLM Final Test | 965 audios"
    ):

        labels = batch["labels"].clone()
        labels[labels == -100] = tokenizer.pad_token_id

        reference = processor.batch_decode(
            labels,
            group_tokens=False
        )[0]

        gpu_batch = {
            "input_values": batch["input_values"].to("cuda"),
            "attention_mask": batch["attention_mask"].to("cuda")
        }

        with torch.autocast("cuda", dtype=torch.bfloat16):
            outputs = full_model(**gpu_batch)

        logits = outputs.logits[0].float().cpu().numpy()

        # Greedy decoding
        greedy_ids = np.argmax(logits, axis=-1)

        greedy_text = processor.batch_decode(
            torch.tensor(greedy_ids).unsqueeze(0)
        )[0]

        # Beam Search + KenLM
        kenlm_text = kenlm_decoder.decode(
            logits,
            beam_width=beam_width
        )

        test_refs.append(reference)
        test_greedy.append(greedy_text)
        test_kenlm.append(kenlm_text)

        del outputs, gpu_batch

# ==========================================
# 3. WER / CER
# ==========================================

assert len(test_refs) == 965

comparison = []

for name, predictions in [
    ("Greedy", test_greedy),
    ("Beam Search + KenLM", test_kenlm)
]:

    comparison.append({
        "Decoder": name,
        "WER (%)": jiwer.wer(
            test_refs,
            predictions
        ) * 100,
        "CER (%)": jiwer.cer(
            test_refs,
            predictions
        ) * 100
    })

comparison_df = pd.DataFrame(comparison)

print("\n" + "=" * 60)
print("FINAL TEST — 965 AUDIOS")
print("=" * 60)

display(comparison_df.round(4))

# ==========================================
# 4. SAVE RESULTS
# ==========================================

results_dir = PROJECT_DIR / "results"
results_dir.mkdir(parents=True, exist_ok=True)

comparison_df.to_csv(
    results_dir / "kenlm_final_test_metrics.csv",
    index=False
)

pd.DataFrame({
    "reference": test_refs,
    "greedy": test_greedy,
    "beam_kenlm": test_kenlm
}).to_csv(
    results_dir / "kenlm_final_test_predictions.csv",
    index=False
)

print("\nFIRST 3 EXAMPLES")

for i in range(3):
    print(f"\nExample {i + 1}")
    print("Reference:", test_refs[i][:200])
    print("Greedy   :", test_greedy[i][:200])
    print("KenLM    :", test_kenlm[i][:200])

print("\n✅ KenLM final test completed!")

Alpha: 0.5
Beta: 1.0
Beam width: 10


KenLM Final Test | 965 audios:   0%|          | 0/965 [00:00<?, ?it/s]


FINAL TEST — 965 AUDIOS


,Decoder,WER (%),CER (%)
0,Greedy,49.1424,11.5679
1,Beam Search + KenLM,27.4219,10.0776



FIRST 3 EXAMPLES

Example 1
Reference: paytaxtımızdıń kongressorayında mámleketimiz basshısın ázerbayjan respublikası prezidenti ilham aliev hám bmshtıń elatlı punktler boyınsha baǵdarlamasınıń atqarıwshı direktorı anaklaudiya rossbax qutl
Greedy   : paytaqtımıztıń kongres orayında mámleketimis bashısın ázerbajan respublikası prezidenti ili hám aliev hám bumshtıń elatı punktler boyınsha baǵdarlamasına atqarıwsha direktori anaklawda rosbak qutlıqla
KenLM    : paytaxtımızdıń kongress orayında mámleketimiz basshısın ázerbayjan respublikası prezidenti il hám aliev hám bmshtıń elatı punktler boyınsha baǵdarlamasın atqarıwshı direktorı ankara qutlıqladı qatnası

Example 2
Reference: sanlı texnologiyalar baǵdarındaǵı ózara sóylesiw bolsa eki mámleket arasında ornatılǵan zamanagóy birge islesiw kópirin jáne de bekkemleytuǵını sózsiz musılman ziyo muhammad amin súwret óza
Greedy   : sanlı texnologiyalar baǵdarındaǵı óz ara sólesiw hám eki mámleket arasında ornatılǵan zamanagóy birge islesiw kó

In [154]:
from pathlib import Path

project = Path("/home/intern/projects/karakalpak_stt")

files = [
    project / "checkpoints/full_lora_step_13023/training_state.pt",
    project / "kenlm/karakalpak_5gram.binary",
    project / "kenlm/decoder_config.json",
    project / "results/kenlm_final_test_metrics.csv",
]

for file in files:
    print(
        "✅" if file.exists() else "❌",
        file.name
    )

✅ training_state.pt
✅ karakalpak_5gram.binary
✅ decoder_config.json
✅ kenlm_final_test_metrics.csv


In [155]:
from pathlib import Path

PROJECT_DIR = Path(
    "/home/intern/projects/karakalpak_stt"
)

assert PROJECT_DIR.exists(), "Loyiha papkasi topilmadi!"

print("✅ PROJECT_DIR:", PROJECT_DIR)

# Natijalar faylini tekshirish
prediction_path = (
    PROJECT_DIR
    / "results"
    / "kenlm_validation_predictions.csv"
)

print("Predictions mavjud:", prediction_path.exists())

✅ PROJECT_DIR: /home/intern/projects/karakalpak_stt
Predictions mavjud: True


In [156]:
import pandas as pd
import jiwer
from collections import Counter

# 1. Saqlangan validation natijalarini yuklaymiz
prediction_path = (
    PROJECT_DIR / "results" / "kenlm_validation_predictions.csv"
)

df = pd.read_csv(prediction_path).fillna("")

print("Samples:", len(df))
print("Columns:", df.columns.tolist())

assert len(df) == 965

# 2. Reference va KenLM predictionlari
references = df["reference"].astype(str).tolist()
predictions = df["KenLM_WER"].astype(str).tolist()

# 3. Word-level xatolar
word_result = jiwer.process_words(references, predictions)

print("\n" + "=" * 55)
print("WORD ERROR ANALYSIS")
print("=" * 55)

print("Correct words:", word_result.hits)
print("Substitutions:", word_result.substitutions)
print("Deletions:", word_result.deletions)
print("Insertions:", word_result.insertions)

print("WER:", round(word_result.wer * 100, 2), "%")
print("CER:", round(jiwer.cer(references, predictions) * 100, 2), "%")

# 4. KenLM vocabulary coverage
corpus_path = PROJECT_DIR / "kenlm" / "karakalpak_train.txt"

train_vocab = set()

with open(corpus_path, encoding="utf-8") as f:
    for line in f:
        train_vocab.update(line.split())

validation_words = [
    word
    for reference in references
    for word in reference.split()
]

unknown_words = [
    word for word in validation_words
    if word not in train_vocab
]

oov_rate = len(unknown_words) / len(validation_words) * 100

print("\n" + "=" * 55)
print("KENLM VOCABULARY COVERAGE")
print("=" * 55)

print("Train vocabulary:", len(train_vocab))
print("Validation word occurrences:", len(validation_words))
print("Unknown word occurrences:", len(unknown_words))
print("OOV rate:", round(oov_rate, 2), "%")

print("\nTOP 20 UNKNOWN WORDS")

for word, count in Counter(unknown_words).most_common(20):
    print(f"{word:25s} {count}")

# 5. Har bir audio uchun xatolik
df["sample_WER"] = [
    jiwer.wer(ref, pred) * 100
    for ref, pred in zip(references, predictions)
]

df["sample_CER"] = [
    jiwer.cer(ref, pred) * 100
    for ref, pred in zip(references, predictions)
]

print("\nTOP 10 HIGHEST WER SAMPLES")

display(
    df.sort_values("sample_WER", ascending=False)[
        ["reference", "KenLM_WER", "sample_WER", "sample_CER"]
    ].head(10)
)

# 6. Natijalarni saqlash
output_path = PROJECT_DIR / "results" / "kenlm_error_analysis.csv"

df.to_csv(output_path, index=False)

print("\n✅ Error analysis completed!")
print("Saved:", output_path)

Samples: 965
Columns: ['reference', 'Greedy', 'KenLM_WER', 'KenLM_CER']

WORD ERROR ANALYSIS
Correct words: 22918
Substitutions: 6109
Deletions: 1775
Insertions: 403
WER: 26.9 %
CER: 9.95 %

KENLM VOCABULARY COVERAGE
Train vocabulary: 39962
Validation word occurrences: 30802
Unknown word occurrences: 1169
OOV rate: 3.8 %

TOP 20 UNKNOWN WORDS
adıgeya                   3
mayk                      2
jaylastırıwdan            2
turkestan                 2
baslaǵanbız               2
kameraların               2
kitapxanasına             2
úyrengen                  2
oraylarǵa                 2
jaqınlarına               2
paydalanıwshını           2
háreketten                2
qoyatuǵının               2
kirseńiz                  2
alimentler                2
saplastıradı              2
qalar                     2
paraxatshılıq             2
chor                      2
keywanı                   2

TOP 10 HIGHEST WER SAMPLES


,reference,KenLM_WER,sample_WER,sample_CER
435,baslamalar marafonı na start berildi jaslar is...,sanaladı arada stendi jaslar sergeli jaslar ay...,78.947368,40.397351
823,dáslep baǵ jaratıp hár qıylı miyweli terekler ...,másele baǵı jaratıp arqalı máyeklerin ásirese ...,77.142857,40.540541
380,ózbekstan atababalarımız jurtı bul áyyemgi jer...,ózbekstan atababalar bul áemgi jerler tuwısqan...,75.675676,27.070064
677,jenevaǵa sapar sheńberinde senat baslıǵı tanzi...,almaları sheńberinen baslıǵı tanılakeri mánisi...,75.000000,34.199134
769,atap ótilgenindey ǵalaba xabar qurallarınıń mi...,ótilgende ǵalaba xabar quralların miynet arqal...,73.333333,25.850340
307,bul bolsa qol miynetin biraz jeńillestirip óni...,bul bolsa qum bir shelestirip ónimniń sobıarta...,72.222222,35.333333
958,sonıń ishinde áyyemgi shıǵıs ájayıbatı samarqa...,soń shın emissiyaların eki imei jıl islam máde...,71.428571,43.457944
661,biraq hákimler bunıń ekonomikalıq ósimge jumıs...,akimbunnıń ekonomika qosım jumısı ornaǵan byud...,69.696970,29.365079
540,keywanı analar hám belsendi hayalqızlar menen ...,kem hám belsendi hayalqızlar menen ushırasıw b...,69.230769,37.850467
43,ózbasımshalıq penen iyelep alınǵan jer uchastk...,óz besinshi birinshi ii bayıǵanı qaytarılmaǵan...,68.750000,38.965517



✅ Error analysis completed!
Saved: /home/intern/projects/karakalpak_stt/results/kenlm_error_analysis.csv


In [ ]:
import jiwer
import pandas as pd

from collections import Counter

# Avvalgi saqlangan CSV
df = pd.read_csv(
    PROJECT_DIR / "results" / "kenlm_validation_predictions.csv"
).fillna("")

references = df["reference"].astype(str).tolist()
predictions = df["KenLM_WER"].astype(str).tolist()

# ==========================================
# WORD ALIGNMENT
# ==========================================

word_result = jiwer.process_words(
    references,
    predictions
)

substitution_pairs = Counter()
deletion_words = Counter()
insertion_words = Counter()

for ref_words, hyp_words, alignments in zip(
    word_result.references,
    word_result.hypotheses,
    word_result.alignments
):

    for chunk in alignments:

        if chunk.type == "substitute":

            ref_segment = " ".join(
                ref_words[chunk.ref_start_idx:chunk.ref_end_idx]
            )

            hyp_segment = " ".join(
                hyp_words[chunk.hyp_start_idx:chunk.hyp_end_idx]
            )

            substitution_pairs[
                (ref_segment, hyp_segment)
            ] += 1

        elif chunk.type == "delete":

            for word in ref_words[
                chunk.ref_start_idx:chunk.ref_end_idx
            ]:
                deletion_words[word] += 1

        elif chunk.type == "insert":

            for word in hyp_words[
                chunk.hyp_start_idx:chunk.hyp_end_idx
            ]:
                insertion_words[word] += 1

# ==========================================
# RESULTS
# ==========================================

print("=" * 65)
print("TOP 30 WORD SUBSTITUTIONS")
print("=" * 65)

sub_df = pd.DataFrame(
    [
        {
            "Reference": ref,
            "Prediction": hyp,
            "Count": count
        }
        for (ref, hyp), count
        in substitution_pairs.most_common(30)
    ]
)

display(sub_df)

print("\nTOP 20 DELETED WORDS")

display(
    pd.DataFrame(
        deletion_words.most_common(20),
        columns=["Word", "Count"]
    )
)

print("\nTOP 20 INSERTED WORDS")

display(
    pd.DataFrame(
        insertion_words.most_common(20),
        columns=["Word", "Count"]
    )
)

print("\n✅ Word-level error patterns extracted!")  

TOP 30 WORD SUBSTITUTIONS


,Reference,Prediction,Count
0,jıyırma,jigirma,14
1,jıl,jılı,8
2,haqqında,haqqındaǵı,8
3,joqarǵı,joqarı,7
4,shıǵıwı,shıǵıw,5
5,arasındaǵı,arasında,5
6,sondayaq,sonday,5
7,bilimlendiriwi,bilimlendiriw,4
8,menen,men,4
9,ózbekstandı,ózbekstanda,4



TOP 20 DELETED WORDS


,Word,Count
0,hám,49
1,júz,36
2,da,28
3,bir,24
4,mıń,22
5,de,16
6,óza,15
7,jılı,14
8,ǵı,14
9,eki,13



TOP 20 INSERTED WORDS


,Word,Count
0,bir,14
1,hám,9
2,de,4
3,bul,4
4,mine,4
5,professor,4
6,mıń,3
7,onlayn,3
8,sociallıq,3
9,ilimiy,3



✅ Word-level error patterns extracted!


In [158]:
print("full_model:", "full_model" in globals())
print("kenlm_decoder:", "kenlm_decoder" in globals())
print("cached_logits:", "cached_logits" in globals())

full_model: True
kenlm_decoder: True
cached_logits: True


In [159]:
from pathlib import Path
import importlib.util

PROJECT_DIR = Path("/home/intern/projects/karakalpak_stt")

# Model va checkpoint
FINAL_MODEL_DIR = (
    PROJECT_DIR
    / "models"
    / "karakalpak_xlsr300m_lora_ctc_v1"
)

CHECKPOINT_DIR = (
    PROJECT_DIR
    / "checkpoints"
    / "full_lora_step_13023"
)

# KenLM
KENLM_DIR = PROJECT_DIR / "kenlm"

# Processor
PROCESSOR_DIR = PROJECT_DIR / "processor"

paths = {
    "Exported LoRA adapter": FINAL_MODEL_DIR / "adapter_model.safetensors",
    "Adapter config": FINAL_MODEL_DIR / "adapter_config.json",
    "Training checkpoint": CHECKPOINT_DIR / "training_state.pt",
    "KenLM binary": KENLM_DIR / "karakalpak_5gram.binary",
    "Decoder config": KENLM_DIR / "decoder_config.json",
    "Processor config": PROCESSOR_DIR,
    "Validation metadata": PROJECT_DIR / "data" / "val_metadata.parquet",
}

print("=" * 65)
print("KARAKALPAK STT — MODEL RECOVERY CHECK")
print("=" * 65)

for name, path in paths.items():
    status = "✅" if path.exists() else "❌"
    print(f"{status} {name}: {path}")

print("\nPYTHON LIBRARIES")

for package in [
    "torch",
    "transformers",
    "peft",
    "kenlm",
    "pyctcdecode",
    "jiwer"
]:
    installed = importlib.util.find_spec(package) is not None
    print(f"{'✅' if installed else '❌'} {package}")

print("\n✅ Recovery check completed!")

KARAKALPAK STT — MODEL RECOVERY CHECK
✅ Exported LoRA adapter: /home/intern/projects/karakalpak_stt/models/karakalpak_xlsr300m_lora_ctc_v1/adapter_model.safetensors
✅ Adapter config: /home/intern/projects/karakalpak_stt/models/karakalpak_xlsr300m_lora_ctc_v1/adapter_config.json
✅ Training checkpoint: /home/intern/projects/karakalpak_stt/checkpoints/full_lora_step_13023/training_state.pt
✅ KenLM binary: /home/intern/projects/karakalpak_stt/kenlm/karakalpak_5gram.binary
✅ Decoder config: /home/intern/projects/karakalpak_stt/kenlm/decoder_config.json
✅ Processor config: /home/intern/projects/karakalpak_stt/processor
✅ Validation metadata: /home/intern/projects/karakalpak_stt/data/val_metadata.parquet

PYTHON LIBRARIES
✅ torch
✅ transformers
✅ peft
✅ kenlm
✅ pyctcdecode
✅ jiwer

✅ Recovery check completed!


In [160]:
import torch
from pathlib import Path

PROJECT_DIR = Path("/home/intern/projects/karakalpak_stt")

checkpoint_path = (
    PROJECT_DIR
    / "checkpoints"
    / "full_lora_step_13023"
    / "training_state.pt"
)

# Checkpointni CPU xotirasiga yuklaymiz
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("=" * 65)
print("XLS-R + LoRA CHECKPOINT INSPECTION")
print("=" * 65)

print("\nCheckpoint keys:")
for key in checkpoint.keys():
    print(" -", key)

print("\nGlobal step:", checkpoint["global_step"])

# Saqlangan trainable weightlar
model_weights = checkpoint["model_weights"]

print("\nTotal saved tensors:", len(model_weights))

print("\nFIRST 10 PARAMETER NAMES")

for name, tensor in list(model_weights.items())[:10]:
    print(name, tuple(tensor.shape))

print("\nCTC HEAD PARAMETERS")

for name, tensor in model_weights.items():
    if "lm_head" in name:
        print(name, tuple(tensor.shape))

lora_count = sum(
    tensor.numel()
    for name, tensor in model_weights.items()
    if "lora_" in name
)

head_count = sum(
    tensor.numel()
    for name, tensor in model_weights.items()
    if "lm_head" in name
)

print("\nLoRA parameters:", f"{lora_count:,}")
print("CTC Head parameters:", f"{head_count:,}")
print("Total:", f"{lora_count + head_count:,}")

assert checkpoint["global_step"] == 13023
assert lora_count == 786432
assert head_count == 36900

print("\n✅ Checkpoint verified!")

XLS-R + LoRA CHECKPOINT INSPECTION

Checkpoint keys:
 - global_step
 - model_weights
 - optimizer_state
 - scheduler_state
 - training_history
 - validation_history
 - base_model_name
 - vocab_size
 - grad_accum_steps
 - torch_rng_state
 - cuda_rng_state

Global step: 13023

Total saved tensors: 98

FIRST 10 PARAMETER NAMES
base_model.model.wav2vec2.encoder.layers.0.attention.v_proj.lora_A.default.weight (8, 1024)
base_model.model.wav2vec2.encoder.layers.0.attention.v_proj.lora_B.default.weight (1024, 8)
base_model.model.wav2vec2.encoder.layers.0.attention.q_proj.lora_A.default.weight (8, 1024)
base_model.model.wav2vec2.encoder.layers.0.attention.q_proj.lora_B.default.weight (1024, 8)
base_model.model.wav2vec2.encoder.layers.1.attention.v_proj.lora_A.default.weight (8, 1024)
base_model.model.wav2vec2.encoder.layers.1.attention.v_proj.lora_B.default.weight (1024, 8)
base_model.model.wav2vec2.encoder.layers.1.attention.q_proj.lora_A.default.weight (8, 1024)
base_model.model.wav2vec2.enco

In [161]:
import torch

from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from peft import LoraConfig, get_peft_model

# ==========================================
# 1. LOAD PROCESSOR
# ==========================================

processor = Wav2Vec2Processor.from_pretrained(
    str(PROJECT_DIR / "processor")
)

tokenizer = processor.tokenizer

# ==========================================
# 2. LOAD BASE MODEL
# ==========================================

MODEL_NAME = "facebook/wav2vec2-xls-r-300m"

base_model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True
)

# ==========================================
# 3. FREEZE PRETRAINED WEIGHTS
# ==========================================

for param in base_model.parameters():
    param.requires_grad = False

base_model.config.layerdrop = 0.0

# ==========================================
# 4. LoRA CONFIGURATION
# ==========================================

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    modules_to_save=["lm_head"]
)

# ==========================================
# 5. CREATE LoRA MODEL
# ==========================================

full_model = get_peft_model(
    base_model,
    lora_config
)

# ==========================================
# 6. LOAD TRAINED WEIGHTS
# ==========================================

checkpoint_weights = checkpoint["model_weights"]

model_parameters = dict(
    full_model.named_parameters()
)

missing_names = (
    set(checkpoint_weights)
    - set(model_parameters)
)

assert not missing_names, (
    f"Checkpoint parameters not found: {missing_names}"
)

with torch.no_grad():

    for name, saved_tensor in checkpoint_weights.items():

        parameter = model_parameters[name]

        assert parameter.shape == saved_tensor.shape, (
            f"Shape mismatch: {name}"
        )

        parameter.copy_(
            saved_tensor.to(
                device=parameter.device,
                dtype=parameter.dtype
            )
        )

# ==========================================
# 7. MOVE TO GPU
# ==========================================

full_model = full_model.to("cuda")
full_model.eval()

# ==========================================
# 8. VERIFICATION
# ==========================================

print("=" * 60)
print("XLS-R-300M + LoRA MODEL RESTORED")
print("=" * 60)

print("Checkpoint step:", checkpoint["global_step"])
print("Vocabulary size:", len(tokenizer))
print("Loaded tensors:", len(checkpoint_weights))

trainable_count = sum(
    p.numel()
    for p in full_model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", f"{trainable_count:,}")

print(
    "Model device:",
    next(full_model.parameters()).device
)

assert len(checkpoint_weights) == 98
assert trainable_count == 823332

print("\n✅ Model restored successfully!")

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


XLS-R-300M + LoRA MODEL RESTORED
Checkpoint step: 13023
Vocabulary size: 36
Loaded tensors: 98
Trainable parameters: 823,332
Model device: cuda:0

✅ Model restored successfully!
